# Decoding odor and sequence position in the hippocampus

**Decoding Hippocampal Activity in Rat(atouille)s · Poster workbook**  
Project team: Hongbin Miao, Jessica Wu, Julia Eng, Jichuan Wu, Wonjae Lee, and Anthony Ko.

> **Main result:** Direct wavelet amplitude led on position; odor scores were nearly tied on the previously viewed test set.

**Start here.** Import this notebook and run it top to bottom. It needs NumPy, pandas, SciPy, and Plotly 6. The reviewed aggregate evidence is embedded, so only the optional recording figures need uploaded files. No model checkpoints or training runs are needed.

**Deepnote upload:** upload this file as **poster.ipynb** and the **poster/** dataset folder beside it. Keep the session folders intact; for example, the notebook expects `poster/090212_superchris/090212_superchris_lfp.npz` and the matching `090212_superchris_bvr.npz`. Upload only recordings you have permission to share in the project.

**Poster assets.** Every figure is interactive and saved as HTML. To also export SVG, PDF, and high-resolution PNG, install `kaleido>=1.1,<2` and make Chrome available (for example, run `plotly_get_chrome -y`). The final cell bundles the figures and tables in `poster_assets.zip`. The methods and captions are editable Markdown.

## The idea in plain language

Imagine a rat expects five smells in order: **A → B → C → D → E**. If C appears fifth, **what smell is it?** is a different question from **where did it appear in the order?** We ask whether a short recording of hippocampal activity can answer either question. Here, “position” always means **place in the sequence**, not where the rat was standing.

We compare a simple summary of **which signal frequencies were strong** (direct wavelet amplitude) with features learned by a model called **CEBRA-Time**. Both use the same kind of decoder to turn features into a prediction.

**Read sections 01–03 for the study and conclusion.** Read 04–06 to see what each model used and what the analysis can’t tell us. Read 07 only if you want extra recording figures. The **Poster copy** and **Caption** blocks are ready to edit or paste into the layout.

**Three words you’ll see often:** **training** fits the model; **validation** guides choices or checks sensitivity; **test** evaluates the frozen comparison on later trials. This test set was already viewed in earlier work, so its results are descriptive. **Balanced accuracy** gives each class equal weight rather than letting the most common class dominate.

In [ ]:
from pathlib import Path
from io import StringIO
from importlib.util import find_spec
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import HTML, display

RAW_ROOT = Path('poster')  # dataset folder beside poster.ipynb
RAW_SESSION = '090212_superchris'
OUT = Path('poster_assets')  # Notebook owns this output directory.
OUT.mkdir(exist_ok=True)
EXPORT_PRINT = find_spec('kaleido') is not None  # Set False for an HTML-only run.
FIGURES = {}

# Dark teal and mint for the paired models, sampled from the presentation.
MINT, WHITE, INK = '#e6f1ef', '#ffffff', '#183830'
TEAL, PALE_TEAL, LOSS = '#087c69', '#b3d7c9', '#a33e35'
GRID, MUTED = '#dce8e4', '#49645d'
CATS = ['#286491', '#a16a24', '#087c69', '#795382', '#a44d35']
MODEL_COLORS = [TEAL, PALE_TEAL]
SIGN_SCALE = [(0, '#d58e86'), (.5, WHITE), (1, '#75ab9a')]
WEIGHT_SCALE = [(0, WHITE), (.5, PALE_TEAL), (1, '#185b4d')]
RATS = ['Mitt', 'Barat', 'Stella', 'Superchris', 'Buchanan']
TARGETS = {'odor':'Odor', 'position':'Sequence position'}
MODELS = {'historical_anchor':'Direct wavelet amplitude', 'cebra_combined_mean':'Wavelet CEBRA-Time'}
BANDS = {'freq_0_4':'4–11 Hz', 'freq_4_8':'15–40 Hz', 'freq_8_12':'56–150 Hz',
         'time_0_5':'0–250 ms', 'time_5_10':'250–500 ms'}
pio.templates['presentation'] = go.layout.Template(layout={
    'paper_bgcolor': WHITE, 'plot_bgcolor': WHITE,
    'font': {'family':'Arial, sans-serif','size':15,'color':INK},
    'title': {'font':{'size':25,'color':INK},'x':.03},
    'xaxis': {'gridcolor':GRID,'linecolor':GRID,'zerolinecolor':MUTED,'automargin':True},
    'yaxis': {'gridcolor':GRID,'linecolor':GRID,'zerolinecolor':MUTED,'automargin':True},
    'hoverlabel': {'bgcolor':WHITE,'font':{'color':INK}},
})
pio.templates.default = 'presentation'

def publish(fig, name, title, *, width=1000, height=380, margin=None):
    fig.update_layout(title_text=f'<b>{title}</b>', autosize=True, height=height,
                      margin=margin or dict(l=60,r=45,t=84,b=52))
    fig.update_annotations(font_size=15, font_color=INK)
    # One local JavaScript file serves all charts in the downloadable bundle.
    config = {'responsive':True,'displaylogo':False}
    fig.write_html(OUT / f'{name}.html', include_plotlyjs='directory', config=config)
    FIGURES[name] = (fig, width)
    fig.show(config=config)

display(HTML(f"""
<div style="background:{MINT};color:{INK};padding:20px 24px;border-top:5px solid {PALE_TEAL};max-width:960px">
<div style="font:700 13px Arial;color:{TEAL}">TEAM 8 · POSTER WORKBOOK</div>
<div style="font:700 30px Georgia,serif;margin:7px 0">Decoding Hippocampal Activity in Rat(atouille)s</div>
<div style="background:{WHITE};padding:10px 12px;font:15px Arial">
01 Study design &nbsp;→&nbsp; 02–03 Main results &nbsp;→&nbsp; 04–06 What the models used &nbsp;→&nbsp; 07–08 Extras and sources</div>
</div>
"""))

def csv_frame(name):
    return pd.read_csv(StringIO(SNAPSHOT[name]))

## Reviewed evidence snapshot (data, not model outputs at the trial level)

This cell contains the saved per-rat scores, class and diagnostic counts, signal-quality summaries, and all-rat interpretation. It does not include recordings, fitted model files, or individual predictions. The reading guide at the end points to the relevant sections and exported files.

In [ ]:
SNAPSHOT = json.loads("{\"session_scores.csv\":\"target,model,split,session,ba,observed_macro_recall,n,n_rat,n_rat_ba\\nodor,cebra_combined_mean,test,Barat,0.5,0.5,18,1,1\\nodor,cebra_combined_mean,test,Buchanan,,0.5416666666666666,11,1,0\\nodor,cebra_combined_mean,test,Mitt,0.3083333333333333,0.3083333333333333,31,1,1\\nodor,cebra_combined_mean,test,Stella,0.5,0.5,10,1,1\\nodor,cebra_combined_mean,test,Superchris,0.6266666666666667,0.6266666666666667,16,1,1\\nodor,cebra_combined_mean,val,Barat,0.34523809523809523,0.34523809523809523,23,1,1\\nodor,cebra_combined_mean,val,Buchanan,0.55,0.55,17,1,1\\nodor,cebra_combined_mean,val,Mitt,0.3757142857142857,0.3757142857142857,42,1,1\\nodor,cebra_combined_mean,val,Stella,0.4961904761904762,0.4961904761904762,27,1,1\\nodor,cebra_combined_mean,val,Superchris,0.4666666666666667,0.4666666666666667,28,1,1\\nodor,historical_anchor,test,Barat,0.30476190476190473,0.30476190476190473,18,1,1\\nodor,historical_anchor,test,Buchanan,,0.875,11,1,0\\nodor,historical_anchor,test,Mitt,0.5505555555555556,0.5505555555555556,31,1,1\\nodor,historical_anchor,test,Stella,0.7,0.7,10,1,1\\nodor,historical_anchor,test,Superchris,0.4,0.4,16,1,1\\nodor,historical_anchor,val,Barat,0.45714285714285713,0.45714285714285713,23,1,1\\nodor,historical_anchor,val,Buchanan,0.32999999999999996,0.32999999999999996,17,1,1\\nodor,historical_anchor,val,Mitt,0.5599999999999999,0.5599999999999999,42,1,1\\nodor,historical_anchor,val,Stella,0.6247619047619047,0.6247619047619047,27,1,1\\nodor,historical_anchor,val,Superchris,0.38333333333333336,0.38333333333333336,28,1,1\\nposition,cebra_combined_mean,test,Barat,0.3833333333333333,0.3833333333333333,20,1,1\\nposition,cebra_combined_mean,test,Buchanan,,0.7291666666666666,11,1,0\\nposition,cebra_combined_mean,test,Mitt,0.27944444444444444,0.27944444444444444,33,1,1\\nposition,cebra_combined_mean,test,Stella,0.1,0.1,11,1,1\\nposition,cebra_combined_mean,test,Superchris,0.45,0.45,19,1,1\\nposition,cebra_combined_mean,val,Barat,0.4738095238095238,0.4738095238095238,24,1,1\\nposition,cebra_combined_mean,val,Buchanan,0.4133333333333333,0.4133333333333333,21,1,1\\nposition,cebra_combined_mean,val,Mitt,0.42051948051948046,0.42051948051948046,45,1,1\\nposition,cebra_combined_mean,val,Stella,0.42095238095238097,0.42095238095238097,30,1,1\\nposition,cebra_combined_mean,val,Superchris,0.48571428571428565,0.48571428571428565,29,1,1\\nposition,historical_anchor,test,Barat,0.4714285714285714,0.4714285714285714,20,1,1\\nposition,historical_anchor,test,Buchanan,,0.6666666666666666,11,1,0\\nposition,historical_anchor,test,Mitt,0.25666666666666665,0.25666666666666665,33,1,1\\nposition,historical_anchor,test,Stella,0.5,0.5,11,1,1\\nposition,historical_anchor,test,Superchris,0.37333333333333335,0.37333333333333335,19,1,1\\nposition,historical_anchor,val,Barat,0.6190476190476191,0.6190476190476191,24,1,1\\nposition,historical_anchor,val,Buchanan,0.45,0.45,21,1,1\\nposition,historical_anchor,val,Mitt,0.4545454545454545,0.4545454545454545,45,1,1\\nposition,historical_anchor,val,Stella,0.5847619047619047,0.5847619047619047,30,1,1\\nposition,historical_anchor,val,Superchris,0.4309523809523809,0.4309523809523809,29,1,1\\n\",\"headline.csv\":\"target,model,split,ba,observed_macro_recall,n,n_rat,n_rat_ba\\nodor,cebra_combined_mean,test,,0.4953333333333333,86,5,4\\nodor,cebra_combined_mean,val,0.44676190476190475,0.44676190476190475,137,5,5\\nodor,historical_anchor,test,,0.566063492063492,86,5,4\\nodor,historical_anchor,val,0.47104761904761905,0.47104761904761905,137,5,5\\nposition,cebra_combined_mean,test,,0.3883888888888889,94,5,4\\nposition,cebra_combined_mean,val,0.4428658008658008,0.4428658008658008,149,5,5\\nposition,historical_anchor,test,,0.4536190476190476,94,5,4\\nposition,historical_anchor,val,0.5078614718614718,0.5078614718614718,149,5,5\\n\",\"outseq.csv\":\"model,split,scope,session,n,actual_support,usual_support,follows_actual,follows_usual,neither\\ncebra_combined_mean,test,pooled_trials,pooled,8,\\\"[0,2,1,4,1]\\\",\\\"[0,0,3,1,4]\\\",0,2,6\\ncebra_combined_mean,test,session,Barat,2,\\\"[0,2,0,0,0]\\\",\\\"[0,0,0,1,1]\\\",0,0,2\\ncebra_combined_mean,test,session,Buchanan,0,\\\"[0,0,0,0,0]\\\",\\\"[0,0,0,0,0]\\\",0,0,0\\ncebra_combined_mean,test,session,Mitt,2,\\\"[0,0,0,2,0]\\\",\\\"[0,0,0,0,2]\\\",0,0,2\\ncebra_combined_mean,test,session,Stella,1,\\\"[0,0,0,0,1]\\\",\\\"[0,0,1,0,0]\\\",0,0,1\\ncebra_combined_mean,test,session,Superchris,3,\\\"[0,0,1,2,0]\\\",\\\"[0,0,2,0,1]\\\",0,2,1\\ncebra_combined_mean,val,pooled_trials,pooled,12,\\\"[0,4,2,0,6]\\\",\\\"[2,4,3,1,2]\\\",6,0,6\\ncebra_combined_mean,val,session,Barat,1,\\\"[0,0,0,0,1]\\\",\\\"[1,0,0,0,0]\\\",1,0,0\\ncebra_combined_mean,val,session,Buchanan,4,\\\"[0,1,1,0,2]\\\",\\\"[0,2,1,0,1]\\\",1,0,3\\ncebra_combined_mean,val,session,Mitt,3,\\\"[0,1,1,0,1]\\\",\\\"[0,1,2,0,0]\\\",2,0,1\\ncebra_combined_mean,val,session,Stella,3,\\\"[0,1,0,0,2]\\\",\\\"[1,1,0,0,1]\\\",1,0,2\\ncebra_combined_mean,val,session,Superchris,1,\\\"[0,1,0,0,0]\\\",\\\"[0,0,0,1,0]\\\",1,0,0\\nhistorical_anchor,test,pooled_trials,pooled,8,\\\"[0,2,1,4,1]\\\",\\\"[0,0,3,1,4]\\\",1,4,3\\nhistorical_anchor,test,session,Barat,2,\\\"[0,2,0,0,0]\\\",\\\"[0,0,0,1,1]\\\",1,0,1\\nhistorical_anchor,test,session,Buchanan,0,\\\"[0,0,0,0,0]\\\",\\\"[0,0,0,0,0]\\\",0,0,0\\nhistorical_anchor,test,session,Mitt,2,\\\"[0,0,0,2,0]\\\",\\\"[0,0,0,0,2]\\\",0,2,0\\nhistorical_anchor,test,session,Stella,1,\\\"[0,0,0,0,1]\\\",\\\"[0,0,1,0,0]\\\",0,1,0\\nhistorical_anchor,test,session,Superchris,3,\\\"[0,0,1,2,0]\\\",\\\"[0,0,2,0,1]\\\",0,1,2\\nhistorical_anchor,val,pooled_trials,pooled,12,\\\"[0,4,2,0,6]\\\",\\\"[2,4,3,1,2]\\\",6,0,6\\nhistorical_anchor,val,session,Barat,1,\\\"[0,0,0,0,1]\\\",\\\"[1,0,0,0,0]\\\",1,0,0\\nhistorical_anchor,val,session,Buchanan,4,\\\"[0,1,1,0,2]\\\",\\\"[0,2,1,0,1]\\\",2,0,2\\nhistorical_anchor,val,session,Mitt,3,\\\"[0,1,1,0,1]\\\",\\\"[0,1,2,0,0]\\\",1,0,2\\nhistorical_anchor,val,session,Stella,3,\\\"[0,1,0,0,2]\\\",\\\"[1,1,0,0,1]\\\",1,0,2\\nhistorical_anchor,val,session,Superchris,1,\\\"[0,1,0,0,0]\\\",\\\"[0,0,0,1,0]\\\",1,0,0\\n\",\"signal_qa.csv\":\"session,status,legacy_status,legacy_windows,legacy_peak_hz,legacy_attenuation_db,high_resolution_status,windows,peak_hz,attenuation_db,evidence_status\\nBarat,done,done,1274,60.0,30.629953975932438,done,156,,54.3598359218642,runner_reported\\nBuchanan,done,done,965,60.0,32.25756854364426,done,110,,58.74368769268109,runner_reported\\nMitt,done,done,1795,60.0,43.88550391039102,done,220,60.0,64.01723601378507,independently_reviewed_mitt\\nStella,done,done,1057,60.0,34.10086673371229,done,128,60.0,60.25626789488132,runner_reported\\nSuperchris,done,done,962,60.0,36.147381488475745,done,110,60.0,60.149831635511205,runner_reported\\n\",\"class_support.csv\":\"target,model,split,session,class,support,predicted,tp,recall\\nodor,cebra_combined_mean,test,Barat,1,7,9,7,1.0\\nodor,cebra_combined_mean,test,Barat,2,2,4,1,0.5\\nodor,cebra_combined_mean,test,Barat,3,3,0,0,0.0\\nodor,cebra_combined_mean,test,Barat,4,3,3,2,0.6666666666666666\\nodor,cebra_combined_mean,test,Barat,5,3,2,1,0.3333333333333333\\nodor,cebra_combined_mean,test,Buchanan,1,4,4,4,1.0\\nodor,cebra_combined_mean,test,Buchanan,2,3,3,2,0.6666666666666666\\nodor,cebra_combined_mean,test,Buchanan,3,2,1,1,0.5\\nodor,cebra_combined_mean,test,Buchanan,4,2,0,0,0.0\\nodor,cebra_combined_mean,test,Buchanan,5,0,3,0,\\nodor,cebra_combined_mean,test,Mitt,1,10,10,5,0.5\\nodor,cebra_combined_mean,test,Mitt,2,9,6,3,0.3333333333333333\\nodor,cebra_combined_mean,test,Mitt,3,8,8,3,0.375\\nodor,cebra_combined_mean,test,Mitt,4,3,5,1,0.3333333333333333\\nodor,cebra_combined_mean,test,Mitt,5,1,2,0,0.0\\nodor,cebra_combined_mean,test,Stella,1,4,3,2,0.5\\nodor,cebra_combined_mean,test,Stella,2,2,5,1,0.5\\nodor,cebra_combined_mean,test,Stella,3,1,0,0,0.0\\nodor,cebra_combined_mean,test,Stella,4,2,1,1,0.5\\nodor,cebra_combined_mean,test,Stella,5,1,1,1,1.0\\nodor,cebra_combined_mean,test,Superchris,1,5,4,4,0.8\\nodor,cebra_combined_mean,test,Superchris,2,3,1,1,0.3333333333333333\\nodor,cebra_combined_mean,test,Superchris,3,4,5,2,0.5\\nodor,cebra_combined_mean,test,Superchris,4,2,4,2,1.0\\nodor,cebra_combined_mean,test,Superchris,5,2,2,1,0.5\\nodor,cebra_combined_mean,val,Barat,1,6,11,6,1.0\\nodor,cebra_combined_mean,val,Barat,2,7,2,1,0.14285714285714285\\nodor,cebra_combined_mean,val,Barat,3,3,0,0,0.0\\nodor,cebra_combined_mean,val,Barat,4,4,4,1,0.25\\nodor,cebra_combined_mean,val,Barat,5,3,6,1,0.3333333333333333\\nodor,cebra_combined_mean,val,Buchanan,1,4,5,4,1.0\\nodor,cebra_combined_mean,val,Buchanan,2,4,7,3,0.75\\nodor,cebra_combined_mean,val,Buchanan,3,2,3,2,1.0\\nodor,cebra_combined_mean,val,Buchanan,4,5,1,0,0.0\\nodor,cebra_combined_mean,val,Buchanan,5,2,1,0,0.0\\nodor,cebra_combined_mean,val,Mitt,1,14,18,12,0.8571428571428571\\nodor,cebra_combined_mean,val,Mitt,2,10,7,2,0.2\\nodor,cebra_combined_mean,val,Mitt,3,7,6,3,0.42857142857142855\\nodor,cebra_combined_mean,val,Mitt,4,7,6,1,0.14285714285714285\\nodor,cebra_combined_mean,val,Mitt,5,4,5,1,0.25\\nodor,cebra_combined_mean,val,Stella,1,7,6,5,0.7142857142857143\\nodor,cebra_combined_mean,val,Stella,2,5,6,3,0.6\\nodor,cebra_combined_mean,val,Stella,3,6,3,3,0.5\\nodor,cebra_combined_mean,val,Stella,4,6,7,2,0.3333333333333333\\nodor,cebra_combined_mean,val,Stella,5,3,5,1,0.3333333333333333\\nodor,cebra_combined_mean,val,Superchris,1,8,10,8,1.0\\nodor,cebra_combined_mean,val,Superchris,2,6,5,2,0.3333333333333333\\nodor,cebra_combined_mean,val,Superchris,3,6,6,3,0.5\\nodor,cebra_combined_mean,val,Superchris,4,4,6,1,0.25\\nodor,cebra_combined_mean,val,Superchris,5,4,1,1,0.25\\nodor,historical_anchor,test,Barat,1,7,7,6,0.8571428571428571\\nodor,historical_anchor,test,Barat,2,2,3,0,0.0\\nodor,historical_anchor,test,Barat,3,3,1,0,0.0\\nodor,historical_anchor,test,Barat,4,3,3,0,0.0\\nodor,historical_anchor,test,Barat,5,3,4,2,0.6666666666666666\\nodor,historical_anchor,test,Buchanan,1,4,4,4,1.0\\nodor,historical_anchor,test,Buchanan,2,3,3,3,1.0\\nodor,historical_anchor,test,Buchanan,3,2,1,1,0.5\\nodor,historical_anchor,test,Buchanan,4,2,3,2,1.0\\nodor,historical_anchor,test,Buchanan,5,0,0,0,\\nodor,historical_anchor,test,Mitt,1,10,9,6,0.6\\nodor,historical_anchor,test,Mitt,2,9,7,1,0.1111111111111111\\nodor,historical_anchor,test,Mitt,3,8,6,3,0.375\\nodor,historical_anchor,test,Mitt,4,3,5,2,0.6666666666666666\\nodor,historical_anchor,test,Mitt,5,1,4,1,1.0\\nodor,historical_anchor,test,Stella,1,4,4,4,1.0\\nodor,historical_anchor,test,Stella,2,2,2,2,1.0\\nodor,historical_anchor,test,Stella,3,1,2,1,1.0\\nodor,historical_anchor,test,Stella,4,2,2,1,0.5\\nodor,historical_anchor,test,Stella,5,1,0,0,0.0\\nodor,historical_anchor,test,Superchris,1,5,5,5,1.0\\nodor,historical_anchor,test,Superchris,2,3,6,3,1.0\\nodor,historical_anchor,test,Superchris,3,4,1,0,0.0\\nodor,historical_anchor,test,Superchris,4,2,3,0,0.0\\nodor,historical_anchor,test,Superchris,5,2,1,0,0.0\\nodor,historical_anchor,val,Barat,1,6,8,6,1.0\\nodor,historical_anchor,val,Barat,2,7,5,2,0.2857142857142857\\nodor,historical_anchor,val,Barat,3,3,5,2,0.6666666666666666\\nodor,historical_anchor,val,Barat,4,4,1,0,0.0\\nodor,historical_anchor,val,Barat,5,3,4,1,0.3333333333333333\\nodor,historical_anchor,val,Buchanan,1,4,6,4,1.0\\nodor,historical_anchor,val,Buchanan,2,4,1,1,0.25\\nodor,historical_anchor,val,Buchanan,3,2,2,0,0.0\\nodor,historical_anchor,val,Buchanan,4,5,6,2,0.4\\nodor,historical_anchor,val,Buchanan,5,2,2,0,0.0\\nodor,historical_anchor,val,Mitt,1,14,8,8,0.5714285714285714\\nodor,historical_anchor,val,Mitt,2,10,10,3,0.3\\nodor,historical_anchor,val,Mitt,3,7,9,4,0.5714285714285714\\nodor,historical_anchor,val,Mitt,4,7,13,6,0.8571428571428571\\nodor,historical_anchor,val,Mitt,5,4,2,2,0.5\\nodor,historical_anchor,val,Stella,1,7,7,6,0.8571428571428571\\nodor,historical_anchor,val,Stella,2,5,7,3,0.6\\nodor,historical_anchor,val,Stella,3,6,5,3,0.5\\nodor,historical_anchor,val,Stella,4,6,4,3,0.5\\nodor,historical_anchor,val,Stella,5,3,4,2,0.6666666666666666\\nodor,historical_anchor,val,Superchris,1,8,11,8,1.0\\nodor,historical_anchor,val,Superchris,2,6,7,3,0.5\\nodor,historical_anchor,val,Superchris,3,6,6,1,0.16666666666666666\\nodor,historical_anchor,val,Superchris,4,4,2,1,0.25\\nodor,historical_anchor,val,Superchris,5,4,2,0,0.0\\nposition,cebra_combined_mean,test,Barat,1,7,12,7,1.0\\nposition,cebra_combined_mean,test,Barat,2,4,4,1,0.25\\nposition,cebra_combined_mean,test,Barat,3,3,0,0,0.0\\nposition,cebra_combined_mean,test,Barat,4,3,2,1,0.3333333333333333\\nposition,cebra_combined_mean,test,Barat,5,3,2,1,0.3333333333333333\\nposition,cebra_combined_mean,test,Buchanan,1,4,3,3,0.75\\nposition,cebra_combined_mean,test,Buchanan,2,3,3,2,0.6666666666666666\\nposition,cebra_combined_mean,test,Buchanan,3,2,2,1,0.5\\nposition,cebra_combined_mean,test,Buchanan,4,2,3,2,1.0\\nposition,cebra_combined_mean,test,Buchanan,5,0,0,0,\\nposition,cebra_combined_mean,test,Mitt,1,10,11,6,0.6\\nposition,cebra_combined_mean,test,Mitt,2,9,4,2,0.2222222222222222\\nposition,cebra_combined_mean,test,Mitt,3,8,10,3,0.375\\nposition,cebra_combined_mean,test,Mitt,4,5,5,1,0.2\\nposition,cebra_combined_mean,test,Mitt,5,1,3,0,0.0\\nposition,cebra_combined_mean,test,Stella,1,4,3,2,0.5\\nposition,cebra_combined_mean,test,Stella,2,2,2,0,0.0\\nposition,cebra_combined_mean,test,Stella,3,1,4,0,0.0\\nposition,cebra_combined_mean,test,Stella,4,2,0,0,0.0\\nposition,cebra_combined_mean,test,Stella,5,2,2,0,0.0\\nposition,cebra_combined_mean,test,Superchris,1,5,5,4,0.8\\nposition,cebra_combined_mean,test,Superchris,2,3,0,0,0.0\\nposition,cebra_combined_mean,test,Superchris,3,5,2,1,0.2\\nposition,cebra_combined_mean,test,Superchris,4,4,4,1,0.25\\nposition,cebra_combined_mean,test,Superchris,5,2,8,2,1.0\\nposition,cebra_combined_mean,val,Barat,1,6,11,6,1.0\\nposition,cebra_combined_mean,val,Barat,2,7,2,2,0.2857142857142857\\nposition,cebra_combined_mean,val,Barat,3,3,1,1,0.3333333333333333\\nposition,cebra_combined_mean,val,Barat,4,4,3,1,0.25\\nposition,cebra_combined_mean,val,Barat,5,4,7,2,0.5\\nposition,cebra_combined_mean,val,Buchanan,1,4,6,3,0.75\\nposition,cebra_combined_mean,val,Buchanan,2,5,4,1,0.2\\nposition,cebra_combined_mean,val,Buchanan,3,3,5,2,0.6666666666666666\\nposition,cebra_combined_mean,val,Buchanan,4,5,4,1,0.2\\nposition,cebra_combined_mean,val,Buchanan,5,4,2,1,0.25\\nposition,cebra_combined_mean,val,Mitt,1,14,15,12,0.8571428571428571\\nposition,cebra_combined_mean,val,Mitt,2,11,9,6,0.5454545454545454\\nposition,cebra_combined_mean,val,Mitt,3,8,10,4,0.5\\nposition,cebra_combined_mean,val,Mitt,4,7,6,0,0.0\\nposition,cebra_combined_mean,val,Mitt,5,5,5,1,0.2\\nposition,cebra_combined_mean,val,Stella,1,7,5,4,0.5714285714285714\\nposition,cebra_combined_mean,val,Stella,2,6,5,3,0.5\\nposition,cebra_combined_mean,val,Stella,3,6,9,3,0.5\\nposition,cebra_combined_mean,val,Stella,4,6,7,2,0.3333333333333333\\nposition,cebra_combined_mean,val,Stella,5,5,4,1,0.2\\nposition,cebra_combined_mean,val,Superchris,1,8,10,8,1.0\\nposition,cebra_combined_mean,val,Superchris,2,7,5,3,0.42857142857142855\\nposition,cebra_combined_mean,val,Superchris,3,6,5,3,0.5\\nposition,cebra_combined_mean,val,Superchris,4,4,1,0,0.0\\nposition,cebra_combined_mean,val,Superchris,5,4,8,2,0.5\\nposition,historical_anchor,test,Barat,1,7,7,6,0.8571428571428571\\nposition,historical_anchor,test,Barat,2,4,4,2,0.5\\nposition,historical_anchor,test,Barat,3,3,1,0,0.0\\nposition,historical_anchor,test,Barat,4,3,1,0,0.0\\nposition,historical_anchor,test,Barat,5,3,7,3,1.0\\nposition,historical_anchor,test,Buchanan,1,4,4,4,1.0\\nposition,historical_anchor,test,Buchanan,2,3,2,2,0.6666666666666666\\nposition,historical_anchor,test,Buchanan,3,2,3,1,0.5\\nposition,historical_anchor,test,Buchanan,4,2,2,1,0.5\\nposition,historical_anchor,test,Buchanan,5,0,0,0,\\nposition,historical_anchor,test,Mitt,1,10,9,5,0.5\\nposition,historical_anchor,test,Mitt,2,9,9,3,0.3333333333333333\\nposition,historical_anchor,test,Mitt,3,8,6,2,0.25\\nposition,historical_anchor,test,Mitt,4,5,4,1,0.2\\nposition,historical_anchor,test,Mitt,5,1,5,0,0.0\\nposition,historical_anchor,test,Stella,1,4,5,4,1.0\\nposition,historical_anchor,test,Stella,2,2,1,1,0.5\\nposition,historical_anchor,test,Stella,3,1,3,1,1.0\\nposition,historical_anchor,test,Stella,4,2,1,0,0.0\\nposition,historical_anchor,test,Stella,5,2,1,0,0.0\\nposition,historical_anchor,test,Superchris,1,5,7,5,1.0\\nposition,historical_anchor,test,Superchris,2,3,4,2,0.6666666666666666\\nposition,historical_anchor,test,Superchris,3,5,3,1,0.2\\nposition,historical_anchor,test,Superchris,4,4,3,0,0.0\\nposition,historical_anchor,test,Superchris,5,2,2,0,0.0\\nposition,historical_anchor,val,Barat,1,6,9,6,1.0\\nposition,historical_anchor,val,Barat,2,7,3,3,0.42857142857142855\\nposition,historical_anchor,val,Barat,3,3,4,2,0.6666666666666666\\nposition,historical_anchor,val,Barat,4,4,2,1,0.25\\nposition,historical_anchor,val,Barat,5,4,6,3,0.75\\nposition,historical_anchor,val,Buchanan,1,4,5,4,1.0\\nposition,historical_anchor,val,Buchanan,2,5,3,2,0.4\\nposition,historical_anchor,val,Buchanan,3,3,2,0,0.0\\nposition,historical_anchor,val,Buchanan,4,5,9,3,0.6\\nposition,historical_anchor,val,Buchanan,5,4,2,1,0.25\\nposition,historical_anchor,val,Mitt,1,14,14,11,0.7857142857142857\\nposition,historical_anchor,val,Mitt,2,11,8,3,0.2727272727272727\\nposition,historical_anchor,val,Mitt,3,8,11,4,0.5\\nposition,historical_anchor,val,Mitt,4,7,11,5,0.7142857142857143\\nposition,historical_anchor,val,Mitt,5,5,1,0,0.0\\nposition,historical_anchor,val,Stella,1,7,7,6,0.8571428571428571\\nposition,historical_anchor,val,Stella,2,6,6,3,0.5\\nposition,historical_anchor,val,Stella,3,6,7,3,0.5\\nposition,historical_anchor,val,Stella,4,6,7,4,0.6666666666666666\\nposition,historical_anchor,val,Stella,5,5,3,2,0.4\\nposition,historical_anchor,val,Superchris,1,8,11,8,1.0\\nposition,historical_anchor,val,Superchris,2,7,8,4,0.5714285714285714\\nposition,historical_anchor,val,Superchris,3,6,4,2,0.3333333333333333\\nposition,historical_anchor,val,Superchris,4,4,5,1,0.25\\nposition,historical_anchor,val,Superchris,5,4,1,0,0.0\\n\",\"interpretation\":{\"spec_sha256\":\"4e81424fafb56e8f36d9aab22a08961fbe8535994b7995fcef95fc5eb198f8d2\",\"validation_receipt_sha256\":\"c345416b45b3dbb4ddda00f48d286237f411cf1e7cf3064f93f4e7d9dea85d19\",\"weights\":[{\"rat\":\"Mitt\",\"target\":\"odor\",\"n_channels\":22,\"signed\":[[[0.0012394756,-0.0069394061,-0.0022210226,-0.0062309507,-0.0004001632,0.0062487153,0.0013604433,0.0024537579,0.0064732122,0.0157319718,0.0143376432,0.0087025934],[-0.0004006674,-0.0024301335,-0.0010712732,-0.001166691,-0.0082438203,-0.0008839535,-0.0029927039,0.0036927911,0.003397069,-0.0006073333,-0.0042924363,-0.0060667565],[-0.0012700545,0.0001321734,-0.0036157357,0.0084292759,-0.0010916203,0.0030650089,-0.0020386195,0.0019789066,0.007256495,0.0031400879,-0.0018741303,-0.0023974333],[0.0007417067,0.0024534688,-0.004315805,0.0002604444,0.0029920059,-0.0034758873,-0.0014688572,0.0045768671,0.0035654852,-0.0008750434,-0.0014365087,-0.0015274778],[0.0027040043,0.0039500739,-0.006162832,0.004927697,0.0013953862,-0.0023201824,-0.0073699396,0.0044388665,0.0036338483,-0.0007836586,0.0169393268,0.0103094077],[0.0054024229,0.0041507473,-0.0091901339,0.0043523608,-0.0000658369,0.0014718165,-0.0007226801,-0.0032303776,-0.0032940226,0.0087885193,0.0098701747,0.0136870472],[0.0059278576,0.0011426416,-0.010186405,-0.0121503265,-0.003114204,-0.0121796599,-0.0010239293,0.0017292308,0.0093782736,0.0028169783,-0.0054184108,-0.0126992918],[0.0046955191,-0.0079339593,-0.0097351126,-0.0158166644,-0.004958388,0.0020970858,-0.0003740248,-0.0084343952,-0.0053560845,-0.0010264143,-0.0032373119,-0.0056174711],[0.0028190138,-0.0057607998,-0.0056183586,0.0019680839,-0.0014810844,0.0006989186,-0.0018239319,-0.0041800243,0.0006160652,-0.0026263291,-0.0027405784,-0.0011886882],[0.000348247,-0.0033558014,-0.0055179958,-0.0022020115,-0.0006422259,-0.0018491845,-0.0067447217,-0.0015766303,-0.0067932697,-0.0083636637,0.0025404661,0.002445417]],[[0.0005756934,0.0031341516,-0.0021141071,-0.0038590823,-0.006665789,-0.021654196,-0.0026258274,-0.0012404505,0.0005727924,-0.0014038118,-0.0045957502,-0.003768079],[-0.0004927909,0.0061057752,-0.0041077253,-0.0087813457,0.0027744202,-0.0098962191,0.0033667173,0.0068150223,0.0174386614,0.0100762723,0.0074052834,0.0024765526],[-0.0006406849,0.0059078563,-0.0017594989,-0.0066247801,0.0005745101,-0.004832407,0.0049653812,-0.0000734791,0.0056400761,0.0032232735,0.0030034628,-0.009033041],[-0.0009531359,0.002890802,-0.0046024484,-0.005907802,-0.0043119179,-0.0008041982,0.0148550797,0.005599957,0.0072758373,-0.0041094444,-0.004767791,-0.004255321],[-0.0022121741,-0.0009126869,-0.0025947539,-0.007838271,-0.0020245809,-0.0027493671,-0.0006951301,-0.0014717253,0.0028650398,0.0070605867,-0.0083530574,-0.0059388109],[-0.0042703432,-0.0106115121,0.0018033265,-0.000602535,0.0014317611,-0.0109645918,-0.0032747297,-0.0000149075,0.0017885758,-0.0123744117,-0.0175443947,-0.0062281678],[-0.0066413197,-0.0055638793,0.0020839788,0.0138132876,-0.0011909974,0.0054414093,0.0059715461,-0.0013037883,-0.0032613391,0.0122776458,0.0030982209,0.0104436267],[-0.010525374,0.0048733029,0.0022847736,0.015278085,-0.0082035396,-0.0074997187,0.0006673164,0.0045616045,0.0082482071,0.0015037443,0.0050375544,-0.0042790601],[-0.0078296718,0.0058277508,0.0009379824,0.0004651563,-0.0145797283,0.004852203,0.0048015374,0.0024893899,0.0020937008,0.0085456033,0.0011422646,0.0085887068],[-0.0028704595,0.0013950921,0.0005990968,-0.0002086613,0.0008095687,0.0095342517,0.0048266186,0.0012372684,0.0071486557,0.0025735102,-0.008215047,-0.0074344096]],[[0.00245907,0.0018686818,0.0041201563,0.0004190467,-0.0028320178,0.0000261365,0.0093762055,0.0071984075,0.0038780781,-0.0012897711,-0.0064890346,-0.0032168278],[0.0034523157,0.0015617748,0.0028216161,0.0087726294,0.0027981712,0.0058769419,0.0115777316,0.0073160624,-0.0060387563,-0.0001110632,0.0010529617,0.0095070817],[0.0048093136,-0.0006981686,0.0023555466,0.0072439783,0.0074548388,-0.0001335407,-0.000545203,-0.0097087038,-0.018514411,0.0071543575,0.0067946372,0.0023028171],[0.0028116165,-0.0012255663,0.0039429284,0.0013313738,-0.0060196486,-0.0090240917,-0.0044434532,-0.0043933063,-0.005584589,0.0092062018,0.0078618082,0.0047806264],[0.0005005443,0.0025634998,0.0040916117,-0.0021110134,-0.0045681792,0.0090492911,0.0077752074,0.0048381896,0.0035846453,-0.0072461079,-0.0124153741,-0.0098412658],[-0.00008944,-0.0010551186,0.0035787842,0.0004735927,-0.0047801646,0.0027495053,-0.0075436504,-0.0023628778,-0.0008974011,-0.000342787,0.0059141732,-0.0100869864],[-0.0009919491,-0.0023075559,0.0016180045,0.0009251092,-0.0121670418,-0.009031897,-0.0002754042,0.0050836044,0.0027736477,-0.0069397991,-0.0005325111,-0.0083934066],[-0.0003375512,-0.0022983321,0.0006187235,0.005390265,0.0102989256,-0.0070214168,-0.0019208315,0.0026297888,-0.0100436627,-0.0009831868,-0.0062459398,0.0041718568],[-0.0002528679,-0.0011209459,0.0008099983,0.0062862613,0.0139928039,-0.0133930444,-0.0084589762,0.0113223287,-0.0011267001,-0.0085963174,-0.0080635655,0.0034571113],[-0.0012898686,0.0026547526,0.0006962827,-0.0004718281,-0.0049624391,-0.0054201506,-0.0070725713,0.0024855371,0.0033507643,-0.0019499133,0.0024025236,0.0037464284]],[[0.0001520525,0.0036311057,0.0031426933,0.0053612043,0.0092518795,0.0054554928,-0.0069905564,-0.0101908004,-0.0060226469,-0.0059618056,-0.0030095155,-0.004080997],[-0.0011752877,-0.0004404382,0.0031243841,-0.0059393963,0.0048509772,0.0041254982,-0.0039763841,-0.0121525767,-0.0102510357,-0.0046933737,-0.0066663824,-0.0083473157],[-0.0042837184,-0.001994575,0.002606992,-0.0121587066,-0.0014239648,0.0014817402,-0.0033829141,-0.003346135,-0.0058698248,-0.0081430332,-0.0033847866,0.0056029348],[-0.0031391871,-0.0069158667,0.004390056,0.0019693043,0.0053866207,0.0075666296,-0.001661778,-0.0012303997,-0.0025372792,-0.0041882882,-0.0058057993,-0.0042477269],[0.0000943174,-0.0133398241,0.0037691049,0.0004357387,0.00421363,0.000882516,0.0052293773,-0.0093632513,-0.0028885456,0.0072526071,0.0016041718,0.0068725428],[0.0002867564,-0.0016079283,0.001408228,-0.0034141209,0.0016917154,0.0010967242,0.011946261,0.0024967814,-0.0035075236,0.0043288043,0.0103701845,0.0092289732],[0.0013599997,0.0031350892,0.0024930485,0.0005937751,0.0187945596,0.0066480411,-0.0051026064,-0.0082893277,-0.0054765332,-0.0026081623,0.003570176,0.008131084],[0.004772946,0.0035954037,0.0039621938,-0.0027304951,0.0033997896,0.0006540064,0.002352936,-0.0030639294,0.0044801749,0.0051806736,0.007779236,0.0050546006],[0.0052568168,-0.0001773045,0.0018885463,-0.0037035829,-0.0026478435,0.0062478875,0.0091703895,-0.0059561001,-0.0012353195,0.0013614628,0.0075273748,0.0014794412],[0.0061363124,-0.0063087625,0.0023618626,0.0032127882,-0.0044833294,-0.0048309075,0.0090755787,-0.0011957616,-0.0014574503,0.0057221656,0.0002615663,0.0106277417]],[[-0.0044262916,-0.0016945333,-0.0029277196,0.0043097818,0.0006460903,0.0099238514,-0.0011202653,0.0017790853,-0.0049014361,-0.0070765839,-0.0002433429,0.0023633106],[-0.0013835698,-0.004796978,-0.0007670022,0.0071148036,-0.0021797484,0.0007777323,-0.007975361,-0.0056712991,-0.004545939,-0.0046645017,0.0025005741,0.0024304379],[0.0013851441,-0.0033472861,0.0004126961,0.0031102325,-0.0055137639,0.0004191986,0.0010013557,0.0111494108,0.0114876641,-0.005374686,-0.0045391831,0.0035247225],[0.0005389998,0.0027971619,0.0005852688,0.0023466796,0.00195294,0.0057375478,-0.0072809913,-0.0045531181,-0.0027194541,-0.0000334259,0.004148291,0.0052498989],[-0.0010866918,0.0077389374,0.0008968696,0.0045858486,0.0009837437,-0.0048622577,-0.0049395151,0.0015579204,-0.0071949875,-0.0062834269,0.0022249328,-0.0014018741],[-0.0013293959,0.0091238119,0.0023997951,-0.0008092976,0.0017225253,0.0056465456,-0.0004052007,0.003111382,0.0059103713,-0.0004001248,-0.0086101377,-0.0066008663],[0.0003454117,0.0035937044,0.003991373,-0.0031818457,-0.0023223166,0.0091221059,0.0004303936,0.0027802808,-0.0034140489,-0.0055466631,-0.000717475,0.0025179875],[0.00139446,0.0017635847,0.0028694218,-0.0021211905,-0.0005367871,0.0117700434,-0.0007253963,0.0043069313,0.002671365,-0.0046748167,-0.0033335386,0.0006700737],[0.0000067092,0.0012312996,0.0019818315,-0.0050159183,0.0047158523,0.0015940351,-0.0036890189,-0.0036755943,-0.0003477461,0.00131558,0.002134505,-0.012336571],[-0.0023242312,0.0056147192,0.0018607537,-0.0003302871,0.0092784254,0.0025659908,-0.000084904,-0.0009504136,-0.0022487,0.0020179011,0.0030104911,-0.0093851774]]],\"magnitude\":[[[0.0053318052,0.0076667976,0.0153047234,0.0069986344,0.008617025,0.0065150218,0.0053591993,0.0033168556,0.0080926766,0.0163011589,0.0160068706,0.0096456292],[0.0032655432,0.0045583722,0.0153624728,0.0059578524,0.0101083082,0.0059907041,0.0043313985,0.0042945432,0.0058296959,0.0075519001,0.0085553919,0.0073437775],[0.004188458,0.0037721932,0.0113417194,0.0103626426,0.0081584161,0.0044552938,0.0036407582,0.0074934329,0.0090089955,0.0071031541,0.0098216335,0.0051567418],[0.0047324454,0.0063687264,0.0087533749,0.003799795,0.0076405239,0.005656368,0.0031182146,0.0064902133,0.0126290372,0.0073258719,0.0073553352,0.0131889954],[0.0064478233,0.0051538738,0.0079632905,0.0087474655,0.0075093391,0.0045686203,0.0100844017,0.0054504805,0.0083001591,0.0057522496,0.0172124503,0.0116783928],[0.0057366798,0.00770408,0.0103137884,0.0045784818,0.0058998996,0.0054592444,0.0045699017,0.0055294431,0.0054937244,0.0090219392,0.0119137954,0.0187772498],[0.0062186837,0.0042488488,0.0130785438,0.0124453901,0.0098474704,0.0137471673,0.0069206608,0.0042315027,0.0099224137,0.0092844934,0.0060353714,0.0140617416],[0.0058070662,0.0101911732,0.0126288249,0.0160369348,0.0060262571,0.005592381,0.0045978256,0.0119609196,0.0125172039,0.0047759235,0.0059064963,0.0082779945],[0.0068654429,0.0076469386,0.0106990512,0.0113116097,0.0047325262,0.0043952966,0.0048741474,0.0082179913,0.0038000164,0.006823496,0.0067876267,0.0096149017],[0.0069644916,0.0072753824,0.0105469606,0.0090827205,0.0082869926,0.0035927842,0.0087483786,0.004150018,0.0080529079,0.0089677329,0.0049888222,0.0108255666]],[[0.0062215077,0.0043735914,0.008397724,0.0074113428,0.0072792349,0.0218126166,0.0040217002,0.0037559116,0.0076783826,0.0045532195,0.0081811534,0.0093522646],[0.0045608035,0.0117728786,0.0076950771,0.0134657533,0.0057346691,0.0140860246,0.0082715806,0.0080595391,0.017857224,0.018450176,0.010533857,0.0071684484],[0.0033563204,0.0105394599,0.0052665584,0.0099096642,0.002962819,0.0057161933,0.0069100456,0.0063813741,0.0153975383,0.0102147323,0.0071265282,0.011970257],[0.0041792517,0.0055103837,0.0062435661,0.0080399071,0.0067161808,0.0063363313,0.0186407798,0.0097800119,0.0105360134,0.0093003973,0.0142209685,0.0089913602],[0.0027856169,0.0087926319,0.0075175629,0.0107985885,0.003406107,0.0049826381,0.0059618723,0.0033526319,0.0066181872,0.0115992755,0.0154431837,0.0137696235],[0.0046635063,0.0154428931,0.0078036334,0.0036174375,0.0065439388,0.0113453921,0.0056516719,0.0086629538,0.007596936,0.0130469289,0.018534661,0.0118730662],[0.0077388168,0.0100675947,0.0093836066,0.0140071667,0.0031943369,0.0094221614,0.0061028899,0.0039149139,0.013616083,0.0126489864,0.0052577875,0.0121873722],[0.0121945578,0.0091400077,0.0082362162,0.0164093543,0.0089147642,0.008722097,0.0047769761,0.0057573533,0.0085799862,0.0064721027,0.0078054901,0.0112845506],[0.0098067313,0.0074685418,0.0055395953,0.0067720478,0.0145797283,0.0075800511,0.0075029305,0.0066736334,0.0070105411,0.0090304832,0.0035450042,0.0133289176],[0.0061881732,0.0060739465,0.0063927547,0.0045881057,0.0065393357,0.0125230992,0.0073916392,0.0042888685,0.007897799,0.0056241559,0.011473038,0.0193675308]],[[0.005620378,0.0062563389,0.0088503211,0.0041795449,0.003673958,0.0056827233,0.013191607,0.0085027522,0.0075824136,0.0083756102,0.0110308886,0.0060201681],[0.005586817,0.0099125216,0.0076330432,0.0090559596,0.0039043875,0.0060198036,0.012779655,0.0094507008,0.0086206618,0.0085297533,0.0097792476,0.0104298169],[0.0094037918,0.009772055,0.0071962062,0.0097841109,0.0079165007,0.0076485243,0.0055198873,0.0103370932,0.0217715398,0.0111293421,0.0089912974,0.0072768532],[0.0094070093,0.0073746191,0.0074334376,0.0040566568,0.0093383599,0.0106621808,0.0067479666,0.0111829958,0.0073119093,0.0114311326,0.0089130249,0.0077835208],[0.0057215482,0.0053158705,0.0090058893,0.0052820925,0.0071345374,0.0095944941,0.0129835511,0.0076203738,0.0068333779,0.0101540571,0.0167735159,0.0129082407],[0.003100693,0.00393729,0.0055544941,0.0031823011,0.0068742101,0.0075916103,0.0086856063,0.0066404404,0.0080661876,0.0092155617,0.0109386966,0.0128419123],[0.0028211521,0.0053111352,0.0025407033,0.001820737,0.0145453381,0.0123191739,0.0063774056,0.0056184636,0.0060985634,0.0092260006,0.0064685677,0.0103511912],[0.0036888258,0.0054846994,0.005713689,0.0088993319,0.0103049569,0.0103418567,0.0051158542,0.0035904469,0.0136767162,0.0049241906,0.0104414818,0.0096462647],[0.0024291936,0.0048238525,0.0068245578,0.0093272977,0.0161889061,0.0135664217,0.0117722851,0.0113932539,0.0093561024,0.0105722517,0.0123748892,0.0072255672],[0.003201845,0.0102548261,0.0032260919,0.0064474174,0.0090920335,0.007616941,0.0123641697,0.0049439074,0.0044358027,0.0102590626,0.008926632,0.0082899556]],[[0.0054213928,0.0044419735,0.0034250643,0.0062866673,0.0131125106,0.0087846754,0.0084966067,0.0130334904,0.0124111784,0.0098629027,0.0083710675,0.0105233093],[0.0039602652,0.0041021015,0.0044097635,0.0080775184,0.0059266059,0.0042879935,0.0064221131,0.0141133742,0.0118330322,0.0129225322,0.0100651282,0.0100333439],[0.0054947594,0.0024260109,0.0030526607,0.0127490924,0.0048165467,0.0032178254,0.0087926478,0.0042788973,0.0125653379,0.0112797311,0.0074410343,0.006518154],[0.0059113736,0.0073445095,0.004989414,0.0068983523,0.0119041119,0.0098381864,0.0052576523,0.005629288,0.0089702354,0.0065197736,0.008378223,0.0109881423],[0.0035014384,0.0135392114,0.0044001492,0.0056102085,0.0057219732,0.0078081756,0.0072177904,0.0095924813,0.0094170907,0.0104208232,0.0099150296,0.007988179],[0.003531308,0.0047382486,0.004205312,0.0042181644,0.0060545177,0.0060533218,0.012399921,0.0078015192,0.0135342259,0.0066257846,0.0128943092,0.0123518024],[0.0044206316,0.0042491915,0.0044460214,0.0028189257,0.0225855602,0.0105997085,0.0088120551,0.0092068356,0.0083646733,0.0081186924,0.0109429582,0.0123382244],[0.0064926683,0.004349922,0.0042071217,0.0056460512,0.0069246315,0.0026703729,0.0055838668,0.0062584809,0.0105374698,0.0107627858,0.0097861428,0.0121790823],[0.0068690527,0.0035168885,0.0038227459,0.0068361123,0.0048559624,0.0066545402,0.0116967565,0.0069553249,0.0052088227,0.0071125875,0.0094509704,0.0094792774],[0.0067247918,0.0091983884,0.005671331,0.0081320474,0.0076591102,0.0056389556,0.0105199646,0.0063771927,0.0073735533,0.0116851218,0.0112699785,0.0156339045]],[[0.0074237254,0.003900244,0.0079448087,0.0059372839,0.0072390159,0.0100589082,0.0076368973,0.0032395878,0.0059950774,0.010562989,0.0081557799,0.0053739026],[0.004176761,0.0091341817,0.0052559314,0.0089034233,0.005086191,0.0025896403,0.0083543731,0.0075023913,0.0073017581,0.0099910038,0.0089632881,0.0074651676],[0.0043959975,0.0058540361,0.0032523875,0.004372803,0.00892394,0.0028972744,0.0036427716,0.013413156,0.0121655975,0.0072615704,0.0066566389,0.0055351846],[0.0049680698,0.0058665872,0.003415723,0.0056022481,0.0049657624,0.0102017955,0.0075228519,0.0051726149,0.0057551701,0.0087308508,0.0079679168,0.0096154946],[0.0049830974,0.0083840331,0.0036465726,0.0072119435,0.001847691,0.0066648511,0.0060653159,0.003134028,0.0100952649,0.0067430801,0.0078406607,0.007494053],[0.004401917,0.0093414495,0.0028170668,0.0015783211,0.0049450299,0.0067884458,0.0068245725,0.0040431659,0.0091251371,0.0034000091,0.0094281198,0.0085820665],[0.0037887076,0.0040700787,0.0043079393,0.006045062,0.0083929239,0.0160689729,0.0061542051,0.0048741128,0.0047813492,0.0092085916,0.0115116834,0.0098981026],[0.0026863938,0.0025184776,0.005131427,0.0042559883,0.0032273751,0.0117700434,0.0065278636,0.0072884392,0.0087583623,0.0095557964,0.0074406029,0.0059949124],[0.0031401094,0.0026381459,0.0052546244,0.0076879542,0.0108699652,0.0040525254,0.0061184145,0.0054549898,0.0066492598,0.0075906376,0.00646994,0.0148546602],[0.0059506544,0.0071559801,0.0039250928,0.0033369972,0.0119180978,0.0063554199,0.0028257418,0.002207654,0.0033971154,0.0097032006,0.007522345,0.0108496935]]],\"nonzero\":[[[7,14,13,11,11,8,10,3,11,13,15,13],[6,6,17,7,12,11,7,7,11,11,11,9],[6,8,16,10,11,10,7,9,11,10,14,10],[9,15,14,8,10,10,4,11,13,13,8,17],[11,8,12,13,12,7,16,9,12,10,17,14],[10,13,15,10,12,8,6,8,9,11,14,16],[9,5,14,16,11,12,10,8,11,14,10,17],[11,11,13,15,11,7,12,17,14,7,12,13],[12,10,15,17,5,6,9,9,8,9,13,12],[11,11,17,13,12,7,14,10,10,15,7,13]],[[10,9,9,11,11,18,6,5,10,8,11,12],[12,15,9,15,13,19,11,14,17,16,14,12],[7,13,10,11,4,12,10,10,16,14,11,12],[9,11,7,12,10,12,18,13,15,13,13,13],[8,12,10,9,8,9,13,8,12,12,13,17],[9,17,9,7,12,14,8,13,10,14,15,17],[11,12,10,15,3,12,10,8,17,10,8,14],[17,11,10,14,12,15,11,10,9,12,14,14],[13,13,7,12,14,10,9,12,9,11,5,14],[11,7,9,11,13,17,14,11,12,8,16,18]],[[8,12,16,7,3,8,14,11,10,10,16,15],[9,13,13,14,4,9,13,12,17,12,13,14],[12,13,13,11,14,16,10,16,18,10,12,10],[13,14,14,7,18,17,10,19,10,12,13,10],[8,9,15,7,11,11,18,10,12,11,15,14],[7,4,12,8,13,11,8,12,14,15,15,13],[4,8,5,1,16,13,9,10,12,13,11,10],[3,12,12,13,15,13,9,5,15,6,16,11],[3,8,11,13,16,14,11,14,10,11,16,9],[7,15,6,12,13,12,15,8,8,15,11,10]],[[5,7,9,9,12,11,10,13,12,12,11,12],[5,10,9,9,9,7,11,14,13,12,11,14],[9,8,5,18,8,4,15,6,11,13,12,9],[9,12,7,12,19,12,10,9,15,11,9,11],[9,15,7,7,9,10,12,11,9,12,12,14],[6,9,6,8,9,11,14,12,15,12,14,10],[5,6,8,6,19,10,15,11,16,10,13,12],[9,8,6,14,8,5,7,10,12,11,12,16],[9,7,9,9,8,10,12,11,7,12,13,9],[9,11,9,14,11,10,17,13,11,13,11,14]],[[13,8,15,9,11,13,10,7,9,15,12,9],[8,8,8,10,7,6,14,11,10,13,15,11],[6,13,5,9,13,6,9,17,11,11,9,10],[8,13,7,8,10,14,13,9,7,12,11,12],[9,10,10,12,4,9,11,5,14,10,11,10],[11,11,5,3,11,11,11,5,9,6,13,14],[9,9,9,13,14,15,9,7,8,15,15,10],[5,8,8,10,9,19,12,12,12,11,10,7],[8,3,11,9,10,9,8,8,11,8,9,13],[8,9,10,6,12,9,5,4,9,13,9,12]]]},{\"rat\":\"Mitt\",\"target\":\"position\",\"n_channels\":22,\"signed\":[[[0.0009791691,0.0008816768,-0.0072728228,-0.0009693001,0.0001730642,-0.000014871,0.0004638368,0.0002941888,0.0004277955,0.0000778151,0.0035307563,0],[0,0,-0.0051743917,-0.0018090917,-0.0004235932,0,-0.0003577801,0.0016708423,0.0002247602,0.0018694773,0,-0.0001266294],[0,0,-0.0020298572,0.0018054873,0.0000077757,0,0.0004178406,0.0028439537,0,0.0017439837,0,-0.0003636909],[0.0002919925,-0.0000298957,-0.0012270467,0.0006314532,0,-0.0006667797,-0.001565374,0,0.0013099987,-0.0001003857,0.0005297031,-0.0017478872],[0,0.0004593268,0,0.0000490574,0,0,-0.000098416,0,0,0,0.0017101325,0.0000033345],[0,0.0009190574,0,0,-0.0007311846,0,0,0,0,0,0.0002646923,0.0070283079],[0,0.0000897586,-0.0064437035,-0.0000412417,-0.0022322787,-0.0040827447,0,0,0,-0.0014645339,0,0],[0.0001427291,0,-0.007128814,-0.0035065766,0,0,0,0,0,0,0,-0.0024145896],[0,-0.0098689456,-0.0031699775,-0.0006056938,-0.0003495917,0.0002384801,0.0021247907,0,-0.0007695116,0.0008665782,-0.0023664369,0.0008983223],[0,-0.0004525842,-0.0026663772,0,0,0.001002953,-0.0004905536,0,0,-0.0007943928,0.0003087823,-0.0018037147]],[[0.0009791691,0.0008816768,0.0018182057,0.0001362346,-0.0003801036,-0.0016495616,0.0004638368,0.0002941888,0.0004277955,-0.0000194538,-0.0006734903,0],[0,0,0.0012935979,-0.0021980582,0.0001058983,0,-0.0003577801,0.0016708423,0.000233523,0.0049574256,0,-0.0001266294],[0,0,0.0005074643,-0.0072219494,0.0000077757,0,0.0004178406,-0.0007109884,0,0.0017439837,0,-0.0003636909],[0.0002919925,-0.0000298957,0.0003067617,-0.0025258129,0,-0.0006667797,0.006261496,0,-0.0003274997,-0.0001003857,-0.0021188124,-0.0005204632],[0,0.0004593268,0,-0.0001962298,0,0,-0.000098416,0,0,0,-0.0012349133,-0.0000133382],[0,-0.0077739321,0,0,-0.0007311846,0,0,0,0,0,-0.0010587693,-0.0028376688],[0,-0.0003590343,0.0016109259,0.0001649668,-0.0022322787,-0.0032414593,0,0,0,0.0058581355,0,0],[-0.0005709162,0,0.0022192784,0.0008766442,0,0,0,0,0,0,0,0.0006036474],[0,0.0024672364,0.0007924944,-0.0006056938,-0.0003495917,0.0002384801,0.0021247907,0,0.0030878088,0.0008665782,-0.0023664369,0.0008983223],[0,-0.0004525842,0.0006665943,0,0,0.001002953,-0.0004905536,0,0,-0.0007943928,0.0003087823,-0.0039762655]],[[0.0009791691,0.0008816768,0.0018182057,0.0001362346,-0.0003801036,-0.000014871,0.0004638368,0.0002941888,0.0004277955,-0.0000194538,-0.0015102853,0],[0,0,0.0012935979,0.0013357166,0.0001058983,0,0.0014311204,0.0016708423,0.0002247602,0.0014872321,0,0.0005065176],[0,0,0.0005074643,0.0018054873,0.0000077757,0,0.0004178406,-0.0007109884,0,0.0017439837,0,-0.0003636909],[-0.0011679699,-0.0000298957,0.0003067617,0.0006314532,0,-0.0022091258,-0.001565374,0,-0.0003274997,-0.0001003857,0.0005297031,0.003309277],[0,0.0004593268,0,0.0000490574,0,0,0.0003936642,0,0,0,-0.0038954841,0.0000033345],[0,0.0009190574,0,0,-0.0007311846,0,0,0,0,0,0.0002646923,-0.0028376688],[0,0.0000897586,0.0016109259,-0.0000412417,-0.005616474,-0.0032414593,0,0,0,-0.0014645339,0,0],[0.0001427291,0,0.0016365121,0.0008766442,0,0,0,0,0,0,0,0.0006036474],[0,0.0024672364,0.0007924944,0.0022585133,-0.0003495917,-0.0009539204,-0.0086789325,0,-0.0007792741,-0.0043136174,-0.0023664369,0.0008983223],[0,-0.0052989667,0.0006665943,0,0,-0.0040118119,-0.0008297176,0,0,-0.0007943928,0.0003087823,-0.0018037147]],[[0.0009791691,0.0008816768,0.0018182057,0.0001362346,0.0009672466,-0.000014871,0.0004638368,-0.0011767554,-0.001711182,-0.0000194538,-0.0006734903,0],[0,0,0.0012935979,0.0013357166,0.0001058983,0,-0.0003577801,-0.0066833693,-0.0009078035,-0.0098560611,0,-0.0001266294],[0,0,0.0005074643,0.0018054873,0.0000077757,0,-0.0016713623,-0.0007109884,0,0.0017439837,0,0.0014547635],[0.0002919925,-0.0001893246,0.0003067617,0.0006314532,0,-0.0006667797,-0.001565374,0,-0.0003274997,-0.0001003857,0.0005297031,-0.0005204632],[0,-0.0018373072,0,0.0000490574,0,0,-0.000098416,0,0,0,0.0017101325,0.0000033345],[0,0.0009190574,0,0,-0.0007311846,0,0,0,0,0,0.0002646923,0.0014846984],[0,0.0000897586,0.0016109259,-0.0000412417,0.0123133099,-0.0032414593,0,0,0,-0.0014645339,0,0],[0.0001427291,0,0.0016365121,0.0008766442,0,0,0,0,0,0,0,0.0006036474],[0,0.0024672364,0.0007924944,-0.0006056938,-0.0003495917,0.0002384801,0.0023045605,0,-0.0007695116,0.0008665782,0.0037253733,0.0019782118],[0,-0.0004525842,0.0006665943,0,0,0.001002953,0.0023013784,0,0,-0.0007943928,-0.0040162657,0.0093874096]],[[-0.0039166764,-0.0035267073,0.0018182057,0.0005605963,-0.0003801036,0.0016941745,-0.0018553473,0.0002941888,0.0004277955,-0.0000194538,-0.0006734903,0],[0,0,0.0012935979,0.0013357166,0.0001058983,0,-0.0003577801,0.0016708423,0.0002247602,0.0015419261,0,-0.0001266294],[0,0,0.0005074643,0.0018054873,-0.0000311028,0,0.0004178406,-0.0007109884,0,-0.0069759349,0,-0.0003636909],[0.0002919925,0.0002790116,0.0003067617,0.0006314532,0,0.0042094649,-0.001565374,0,-0.0003274997,0.0004015428,0.0005297031,-0.0005204632],[0,0.0004593268,0,0.0000490574,0,0,-0.000098416,0,0,0,0.0017101325,0.0000033345],[0,0.0050167599,0,0,0.0029247383,0,0,0,0,0,0.0002646923,-0.0028376688],[0,0.0000897586,0.0016109259,-0.0000412417,-0.0022322787,0.0138071226,0,0,0,-0.0014645339,0,0],[0.0001427291,0,0.0016365121,0.0008766442,0,0,0,0,0,0,0,0.0006036474],[0,0.0024672364,0.0007924944,-0.000441432,0.001398367,0.0002384801,0.0021247907,0,-0.0007695116,0.0017138829,0.0033739374,-0.0046731787],[0,0.0066567192,0.0006665943,0,0,0.001002953,-0.0004905536,0,0,0.0031775712,0.0030899188,-0.0018037147]]],\"magnitude\":[[[0.0009791691,0.0008816768,0.0072728228,0.0009693001,0.0007120043,0.0006687472,0.0004638368,0.0002941888,0.0004277955,0.0000778151,0.0035307563,0],[0,0,0.0051743917,0.0032226017,0.0004235932,0,0.0003577801,0.0016708423,0.0002282653,0.0031046566,0,0.0001266294],[0,0,0.0020298572,0.0018054873,0.0000077757,0,0.0004178406,0.0028439537,0,0.0017439837,0,0.0003636909],[0.0002919925,0.0000936672,0.0012270467,0.0006314532,0,0.0012837181,0.001565374,0,0.0013099987,0.0001003857,0.0005297031,0.0017478872],[0,0.0004593268,0,0.0000490574,0,0,0.000098416,0,0,0,0.0017101325,0.0000033345],[0,0.0025581384,0,0,0.0007311846,0,0,0,0,0,0.0002646923,0.0087572548],[0,0.0000897586,0.0064437035,0.0000412417,0.0035859568,0.0040827447,0,0,0,0.0014645339,0,0],[0.0001427291,0,0.007128814,0.0035065766,0,0,0,0,0,0,0,0.0024145896],[0,0.0098689456,0.0031699775,0.0006056938,0.0003495917,0.0002384801,0.0021966986,0,0.0007695116,0.0008665782,0.0023664369,0.0013302781],[0,0.0004525842,0.0026663772,0,0,0.001002953,0.0006262192,0,0,0.0007943928,0.0014212369,0.002672735]],[[0.0009791691,0.0008816768,0.0018182057,0.0003059793,0.0003801036,0.0016495616,0.0004638368,0.0002941888,0.0004277955,0.0000194538,0.0010082083,0],[0,0,0.0012935979,0.0034559815,0.0001058983,0,0.0003577801,0.0016708423,0.000233523,0.0049574256,0,0.0001266294],[0,0,0.0005074643,0.0072219494,0.0000077757,0,0.0004178406,0.0007109884,0,0.0017439837,0,0.0003636909],[0.0002919925,0.0000936672,0.0003067617,0.0025258129,0,0.0012837181,0.006261496,0,0.0003274997,0.0001003857,0.0021188124,0.0010114328],[0,0.0004593268,0,0.0001962298,0,0,0.000098416,0,0,0,0.00347716,0.0000133382],[0,0.0077739321,0,0,0.0007311846,0,0,0,0,0,0.0010587693,0.0028376688],[0,0.0003590343,0.0016109259,0.0001649668,0.0035859568,0.0035779735,0,0,0,0.0058581355,0,0],[0.0005709162,0,0.0022192784,0.0008766442,0,0,0,0,0,0,0,0.0006036474],[0,0.0024672364,0.0007924944,0.0006056938,0.0003495917,0.0002384801,0.0021966986,0,0.0030878088,0.0008665782,0.0023664369,0.0013302781],[0,0.0004525842,0.0006665943,0,0,0.001002953,0.0006262192,0,0,0.0007943928,0.0014212369,0.0039762655]],[[0.0009791691,0.0008816768,0.0018182057,0.0003059793,0.0003801036,0.0006687472,0.0004638368,0.0002941888,0.0004277955,0.0000194538,0.0015102853,0],[0,0,0.0012935979,0.0013357166,0.0001058983,0,0.0014311204,0.0016708423,0.0002282653,0.0033340037,0,0.0005065176],[0,0,0.0005074643,0.0018054873,0.0000077757,0,0.0004178406,0.0007109884,0,0.0017439837,0,0.0003636909],[0.0011679699,0.0000936672,0.0003067617,0.0006314532,0,0.0022091258,0.001565374,0,0.0003274997,0.0001003857,0.0005297031,0.003309277],[0,0.0004593268,0,0.0000490574,0,0,0.0003936642,0,0,0,0.0050735024,0.0000033345],[0,0.0025581384,0,0,0.0007311846,0,0,0,0,0,0.0002646923,0.0028376688],[0,0.0000897586,0.0016109259,0.0000412417,0.005616474,0.0035779735,0,0,0,0.0014645339,0,0],[0.0001427291,0,0.0016365121,0.0008766442,0,0,0,0,0,0,0,0.0006036474],[0,0.0024672364,0.0007924944,0.002324218,0.0003495917,0.0009539204,0.0086789325,0,0.0007792741,0.0043136174,0.0023664369,0.0013302781],[0,0.0052989667,0.0006665943,0,0,0.0040118119,0.0008297176,0,0,0.0007943928,0.0014212369,0.002672735]],[[0.0009791691,0.0008816768,0.0018182057,0.0003059793,0.0011885137,0.0006687472,0.0004638368,0.0011767554,0.001711182,0.0000194538,0.0010082083,0],[0,0,0.0012935979,0.0013357166,0.0001058983,0,0.0003577801,0.0066833693,0.0009078035,0.0101399796,0,0.0001266294],[0,0,0.0005074643,0.0018054873,0.0000077757,0,0.0016713623,0.0007109884,0,0.0017439837,0,0.0014547635],[0.0002919925,0.0001893246,0.0003067617,0.0006314532,0,0.0012837181,0.001565374,0,0.0003274997,0.0001003857,0.0005297031,0.0010114328],[0,0.0018373072,0,0.0000490574,0,0,0.000098416,0,0,0,0.0017101325,0.0000033345],[0,0.0025581384,0,0,0.0007311846,0,0,0,0,0,0.0002646923,0.005431089],[0,0.0000897586,0.0016109259,0.0000412417,0.0123133099,0.0035779735,0,0,0,0.0014645339,0,0],[0.0001427291,0,0.0016365121,0.0008766442,0,0,0,0,0,0,0,0.0006036474],[0,0.0024672364,0.0007924944,0.0006056938,0.0003495917,0.0002384801,0.0023045605,0,0.0007695116,0.0008665782,0.006021523,0.0019782118],[0,0.0004525842,0.0006665943,0,0,0.001002953,0.0023013784,0,0,0.0007943928,0.0040162657,0.0093874096]],[[0.0039166764,0.0035267073,0.0018182057,0.0005605963,0.0003801036,0.0016941745,0.0018553473,0.0002941888,0.0004277955,0.0000194538,0.0010082083,0],[0,0,0.0012935979,0.0013357166,0.0001058983,0,0.0003577801,0.0016708423,0.0002282653,0.0033011873,0,0.0001266294],[0,0,0.0005074643,0.0018054873,0.0000311028,0,0.0004178406,0.0007109884,0,0.0069759349,0,0.0003636909],[0.0002919925,0.0002790116,0.0003067617,0.0006314532,0,0.0042094649,0.001565374,0,0.0003274997,0.0004015428,0.0005297031,0.0010114328],[0,0.0004593268,0,0.0000490574,0,0,0.000098416,0,0,0,0.0017101325,0.0000033345],[0,0.0050167599,0,0,0.0029247383,0,0,0,0,0,0.0002646923,0.0028376688],[0,0.0000897586,0.0016109259,0.0000412417,0.0035859568,0.0138071226,0,0,0,0.0014645339,0,0],[0.0001427291,0,0.0016365121,0.0008766442,0,0,0,0,0,0,0,0.0006036474],[0,0.0024672364,0.0007924944,0.0007042508,0.001398367,0.0002384801,0.0021966986,0,0.0007695116,0.0017138829,0.0058106615,0.0046731787],[0,0.0066567192,0.0006665943,0,0,0.001002953,0.0006262192,0,0,0.0031775712,0.0030899188,0.002672735]]],\"nonzero\":[[[0,0,2,1,1,0,0,0,0,1,1,0],[0,0,2,1,1,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,1,0,0,0,0],[0,0,1,0,0,0,0,0,2,0,0,1],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,1],[0,0,2,0,0,1,0,0,0,0,0,0],[0,0,3,3,0,0,0,0,0,0,0,1],[0,1,1,0,0,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,0,0,0,0,0]],[[0,0,0,0,0,1,0,0,0,0,0,0],[0,0,0,1,0,0,0,0,1,1,0,0],[0,0,0,1,0,0,0,0,0,0,0,0],[0,0,0,1,0,0,3,0,0,0,1,0],[0,0,0,1,0,0,0,0,0,0,1,1],[0,1,0,0,0,0,0,0,0,0,1,0],[0,1,0,1,0,0,0,0,0,1,0,0],[1,0,1,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,1,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,1]],[[0,0,0,0,0,0,0,0,0,0,1,0],[0,0,0,0,0,0,1,0,0,1,0,1],[0,0,0,0,0,0,0,0,0,0,0,0],[1,0,0,0,0,2,0,0,0,0,0,1],[0,0,0,0,0,0,1,0,0,0,1,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,1,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,2,0,1,2,0,1,2,0,0],[0,1,0,0,0,1,1,0,0,0,0,0]],[[0,0,0,0,1,0,0,1,1,0,0,0],[0,0,0,0,0,0,0,2,1,2,0,0],[0,0,0,0,0,0,1,0,0,0,0,1],[0,1,0,0,0,0,0,0,0,0,0,0],[0,3,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,1],[0,0,0,0,2,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,1,0,0,0,1,1],[0,0,0,0,0,0,1,0,0,0,1,1]],[[1,1,0,1,0,1,1,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,1,0,0],[0,0,0,0,1,0,0,0,0,2,0,0],[0,1,0,0,0,1,0,0,0,1,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,1,0,0,1,0,0,0,0,0,0,0],[0,0,0,0,0,3,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,1,2,0,0,0,0,1,1,3],[0,1,0,0,0,0,0,0,0,2,1,0]]]},{\"rat\":\"Barat\",\"target\":\"odor\",\"n_channels\":22,\"signed\":[[[-0.000424207,-0.0002378838,-0.0045735013,0,0.0002322277,0,0,0,0,0.0026291834,0.0070799748,0.0009498486],[0.0001602298,0,-0.0018277808,0,0,0,0.0014852313,-0.0006332328,0.0000922974,0.0231294483,0.0019486886,0],[-0.0001980644,0,-0.000261374,0.0000938698,0,0,0.0004734423,0,0,0.0021712756,0.0020042268,0.0008165101],[0,0,0,0,0,0,0,0,0,0.004349252,0.0018060836,0],[0,0,-0.0006227042,-0.0005217401,-0.0004051987,0,0.0001767463,0,0,0,0,0],[0,0,-0.0025478031,0.0009023502,0.0019179749,0,0.0016800551,0,0,0,0,0.000687204],[0,0,-0.0055985321,0,-0.0014610898,0.0001199383,0.0007926159,0.0007292905,0.0004922093,0,-0.000014971,0],[0,0,-0.0031895867,-0.0013398567,0,0.0008982198,0,0,0,0,0,0],[0,0,-0.0011018394,0,-0.0000464508,0,0.000616215,-0.0021231771,0,0.0016436807,0,0],[0.0000636056,0,-0.0030255064,0,0,0,0.0006764535,-0.0003079311,0,0,0,0]],[[-0.000424207,-0.0002378838,0.0011433753,0,0.0002322277,0,0,0,0,-0.0006572959,-0.0017699937,-0.0002374621],[-0.003855733,0,0.0009392696,0,0,0,0.0014852313,-0.0006332328,0.0000922974,-0.0057823621,-0.0004498184,0],[-0.00417164,0,0.0000653435,0.0000938698,0,0,0.0004734423,0,0,-0.0005428189,0.0020042268,0.0008165101],[0,0,0,0,0,0,0,0,0,-0.001087313,-0.0004515209,0],[0,0,-0.0006227042,-0.0014776018,0.0073749067,0,-0.0007069853,0,0,0,0,0],[0,0,-0.0002314052,-0.0036094009,-0.0076718994,0,0.0016800551,0,0,0,0,0.000687204],[0,0,0.001075268,0,-0.0014610898,0.0001199383,-0.0031704635,-0.0029171621,-0.0019688373,0,-0.000014971,0],[0,0,-0.0018926634,0.0003349642,0,0.0008982198,0,0,0,0,0,0],[0,0,0.0002754598,0,-0.0000464508,0,-0.0026478766,-0.0021231771,0,0.0016436807,0,0],[0.0000636056,0,0.0007563766,0,0,0,0.0006764535,-0.0003079311,0,0,0,0]],[[-0.000424207,-0.0002378838,0.0011433753,0,0.0002322277,0,0,0,0,-0.0006572959,-0.0017699937,-0.0002374621],[0.0001602298,0,-0.0009900278,0,0,0,0.0014852313,-0.0006332328,0.0000922974,-0.0057823621,-0.0004498184,0],[-0.0001980644,0,0.0000653435,-0.0003754793,0,0,0.0004734423,0,0,-0.0005428189,0.0020042268,0.0008165101],[0,0,0,0,0,0,0,0,0,-0.001087313,-0.0004515209,0],[0,0,-0.0006227042,-0.0005217401,-0.0004051987,0,0.0001767463,0,0,0,0,0],[0,0,-0.0002314052,0.0009023502,0.0019179749,0,-0.0067202205,0,0,0,0,0.000687204],[0,0,0.0015077547,0,0.0058443594,-0.000479753,0.0007926159,0.0007292905,0.0004922093,0,-0.000014971,0],[0,0,0.0016940834,0.0003349642,0,0.0008982198,0,0,0,0,0,0],[0,0,0.0002754598,0,0.000185803,0,0.000616215,-0.0021231771,0,-0.0065747229,0,0],[0.0000636056,0,0.0007563766,0,0,0,-0.0027058138,-0.0003079311,0,0,0,0]],[[-0.000424207,-0.0002378838,0.0011433753,0,-0.0009289108,0,0,0,0,-0.0006572959,-0.0017699937,-0.0002374621],[0.0001602298,0,0.0009392696,0,0,0,0.0014852313,0.0027826241,0.0003126772,-0.0057823621,-0.0005992333,0],[-0.0001980644,0,0.0000653435,0.0000938698,0,0,-0.0060560662,0,0,-0.0005428189,-0.0080169074,-0.0032660402],[0,0,0,0,0,0,0,0,0,-0.001087313,-0.0004515209,0],[0,0,-0.0006227042,-0.0005217401,-0.0061593105,0,0.0001767463,0,0,0,0,0],[0,0,-0.0002314052,0.0009023502,0.0019179749,0,0.0016800551,0,0,0,0,0.000687204],[0,0,0.0015077547,0,-0.0014610898,0.0001199383,0.0007926159,0.0007292905,0.0004922093,0,-0.000014971,0],[0,0,0.0016940834,0.0003349642,0,0.0008982198,0,0,0,0,0,0],[0,0,0.0002754598,0,-0.0000464508,0,0.000616215,-0.0021231771,0,0.0016436807,0,0],[0.0000636056,0,0.0007563766,0,0,0,0.0006764535,-0.0003079311,0,0,0,0]],[[0.0016968279,0.0009515353,0.0011433753,0,0.0002322277,0,0,0,0,-0.0006572959,-0.0017699937,-0.0002374621],[0.0033750436,0,0.0009392696,0,0,0,-0.0059409252,-0.0008829259,-0.0005895693,-0.0057823621,-0.0004498184,0],[0.004765833,0,0.0000653435,0.0000938698,0,0,0.0046357391,0,0,-0.0005428189,0.0020042268,0.0008165101],[0,0,0,0,0,0,0,0,0,-0.001087313,-0.0004515209,0],[0,0,0.0024908166,0.0030428221,-0.0004051987,0,0.0001767463,0,0,0,0,0],[0,0,0.0032420187,0.0009023502,0.0019179749,0,0.0016800551,0,0,0,0,-0.0027488159],[0,0,0.0015077547,0,-0.0014610898,0.0001199383,0.0007926159,0.0007292905,0.0004922093,0,0.000059884,0],[0,0,0.0016940834,0.0003349642,0,-0.0035928793,0,0,0,0,0,0],[0,0,0.0002754598,0,-0.0000464508,0,0.0007992315,0.0084927085,0,0.0016436807,0,0],[-0.0002544223,0,0.0007563766,0,0,0,0.0006764535,0.0012317244,0,0,0,0]]],\"magnitude\":[[[0.000424207,0.0002378838,0.0045735013,0,0.0002322277,0,0,0,0,0.0026291834,0.0070799748,0.0009498486],[0.0001602298,0,0.0025994998,0,0,0,0.0014852313,0.00073311,0.0000922974,0.0231294483,0.0019486886,0],[0.0001980644,0,0.000261374,0.0000938698,0,0,0.0021383611,0,0,0.0021712756,0.0020042268,0.0008165101],[0,0,0,0,0,0,0,0,0,0.004349252,0.0018060836,0],[0,0,0.0006227042,0.0009040848,0.0027068434,0,0.0001767463,0,0,0,0,0],[0,0,0.0025478031,0.0009023502,0.0019179749,0,0.0016800551,0,0,0,0,0.000687204],[0,0,0.0057715267,0,0.0014610898,0.0001199383,0.0007926159,0.0007292905,0.0004922093,0,0.000014971,0],[0,0,0.0046242854,0.0013398567,0,0.0008982198,0,0,0,0,0,0],[0,0,0.0011018394,0,0.0000464508,0,0.000616215,0.0021231771,0,0.0016436807,0,0],[0.0000636056,0,0.0030255064,0,0,0,0.0006764535,0.0003079311,0,0,0,0]],[[0.000424207,0.0002378838,0.0011433753,0,0.0002322277,0,0,0,0,0.0006572959,0.0017699937,0.0002374621],[0.003855733,0,0.0009392696,0,0,0,0.0014852313,0.00073311,0.0000922974,0.0057823621,0.0005095844,0],[0.00417164,0,0.0000653435,0.0000938698,0,0,0.0021383611,0,0,0.0005428189,0.0020042268,0.0008165101],[0,0,0,0,0,0,0,0,0,0.001087313,0.0004515209,0],[0,0,0.0006227042,0.0014776018,0.0073749067,0,0.0007069853,0,0,0,0,0],[0,0,0.0011579644,0.0036094009,0.0076718994,0,0.0016800551,0,0,0,0,0.000687204],[0,0,0.0017672467,0,0.0014610898,0.0001199383,0.0031704635,0.0029171621,0.0019688373,0,0.000014971,0],[0,0,0.0038461314,0.0003349642,0,0.0008982198,0,0,0,0,0,0],[0,0,0.0002754598,0,0.0000464508,0,0.0026478766,0.0021231771,0,0.0016436807,0,0],[0.0000636056,0,0.0007563766,0,0,0,0.0006764535,0.0003079311,0,0,0,0]],[[0.000424207,0.0002378838,0.0011433753,0,0.0002322277,0,0,0,0,0.0006572959,0.0017699937,0.0002374621],[0.0001602298,0,0.002096848,0,0,0,0.0014852313,0.00073311,0.0000922974,0.0057823621,0.0005095844,0],[0.0001980644,0,0.0000653435,0.0003754793,0,0,0.0021383611,0,0,0.0005428189,0.0020042268,0.0008165101],[0,0,0,0,0,0,0,0,0,0.001087313,0.0004515209,0],[0,0,0.0006227042,0.0009040848,0.0027068434,0,0.0001767463,0,0,0,0,0],[0,0,0.0011579644,0.0009023502,0.0019179749,0,0.0067202205,0,0,0,0,0.000687204],[0,0,0.0015077547,0,0.0058443594,0.000479753,0.0007926159,0.0007292905,0.0004922093,0,0.000014971,0],[0,0,0.0016940834,0.0003349642,0,0.0008982198,0,0,0,0,0,0],[0,0,0.0002754598,0,0.000185803,0,0.000616215,0.0021231771,0,0.0065747229,0,0],[0.0000636056,0,0.0007563766,0,0,0,0.0027058138,0.0003079311,0,0,0,0]],[[0.000424207,0.0002378838,0.0011433753,0,0.0009289108,0,0,0,0,0.0006572959,0.0017699937,0.0002374621],[0.0001602298,0,0.0009392696,0,0,0,0.0014852313,0.0027826241,0.0003126772,0.0057823621,0.0005992333,0],[0.0001980644,0,0.0000653435,0.0000938698,0,0,0.0060560662,0,0,0.0005428189,0.0080169074,0.0032660402],[0,0,0,0,0,0,0,0,0,0.001087313,0.0004515209,0],[0,0,0.0006227042,0.0009040848,0.0061593105,0,0.0001767463,0,0,0,0,0],[0,0,0.0011579644,0.0009023502,0.0019179749,0,0.0016800551,0,0,0,0,0.000687204],[0,0,0.0015077547,0,0.0014610898,0.0001199383,0.0007926159,0.0007292905,0.0004922093,0,0.000014971,0],[0,0,0.0016940834,0.0003349642,0,0.0008982198,0,0,0,0,0,0],[0,0,0.0002754598,0,0.0000464508,0,0.000616215,0.0021231771,0,0.0016436807,0,0],[0.0000636056,0,0.0007563766,0,0,0,0.0006764535,0.0003079311,0,0,0,0]],[[0.0016968279,0.0009515353,0.0011433753,0,0.0002322277,0,0,0,0,0.0006572959,0.0017699937,0.0002374621],[0.0033750436,0,0.0009392696,0,0,0,0.0059409252,0.0008829259,0.0005895693,0.0057823621,0.0005095844,0],[0.004765833,0,0.0000653435,0.0000938698,0,0,0.0046357391,0,0,0.0005428189,0.0020042268,0.0008165101],[0,0,0,0,0,0,0,0,0,0.001087313,0.0004515209,0],[0,0,0.0024908166,0.0030428221,0.0027068434,0,0.0001767463,0,0,0,0,0],[0,0,0.0032420187,0.0009023502,0.0019179749,0,0.0016800551,0,0,0,0,0.0027488159],[0,0,0.0015077547,0,0.0014610898,0.0001199383,0.0007926159,0.0007292905,0.0004922093,0,0.000059884,0],[0,0,0.0016940834,0.0003349642,0,0.0035928793,0,0,0,0,0,0],[0,0,0.0002754598,0,0.0000464508,0,0.0007992315,0.0084927085,0,0.0016436807,0,0],[0.0002544223,0,0.0007563766,0,0,0,0.0006764535,0.0012317244,0,0,0,0]]],\"nonzero\":[[[0,0,1,0,0,0,0,0,0,1,2,2],[0,0,1,0,0,0,0,0,0,4,1,0],[0,0,1,0,0,0,0,0,0,1,0,0],[0,0,0,0,0,0,0,0,0,1,1,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,2,0,0,0,0,0,0,0,0,0],[0,0,2,0,0,0,0,0,0,0,0,0],[0,0,2,1,0,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,0,0,0,0,0],[0,0,2,0,0,0,0,0,0,0,0,0]],[[0,0,0,0,0,0,0,0,0,0,0,0],[1,0,0,0,0,0,0,0,0,0,0,0],[1,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,1,1,0,1,0,0,0,0,0],[0,0,0,2,1,0,0,0,0,0,0,0],[0,0,1,0,0,0,1,1,1,0,0,0],[0,0,1,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,1,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0]],[[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,0,0,0,0,0],[0,0,0,1,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,1,0,0,0,0,0],[0,0,0,0,1,1,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,1,0,0,0,0,1,0,0],[0,0,0,0,0,0,1,0,0,0,0,0]],[[0,0,0,0,1,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,1,1,0,1,0],[0,0,0,0,0,0,1,0,0,0,1,1],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,1,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0]],[[2,1,0,0,0,0,0,0,0,0,0,0],[1,0,0,0,0,0,2,1,1,0,0,0],[1,0,0,0,0,0,2,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,1,1,0,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,0,0,0,0,1],[0,0,0,0,0,0,0,0,0,0,1,0],[0,0,0,0,0,1,0,0,0,0,0,0],[0,0,0,0,0,0,1,2,0,0,0,0],[1,0,0,0,0,0,0,1,0,0,0,0]]]},{\"rat\":\"Barat\",\"target\":\"position\",\"n_channels\":22,\"signed\":[[[-0.0011496009,-0.0073841896,-0.0047101902,0.0014481821,-0.0011613893,-0.0005612011,0.0011363036,-0.0009703441,0.0032115056,0.0152142108,0.0185199185,0.0132063889],[-0.0014825787,-0.0053518662,-0.0057243077,-0.0004675222,0.0005802583,0.0001629104,-0.0024295438,0.0012509144,0.0058142034,0.0225444814,0.0173442111,0.0124272928],[-0.0015001003,-0.000746348,-0.0041458216,-0.0022941596,0.0002720948,0.0007634118,0.0024347611,0.0009967635,0.0009052663,0.0115033403,0.0095718142,0.0071525087],[-0.0014249488,0.0004151927,-0.0028792552,-0.0004556917,0.0001296059,-0.0048286073,-0.000261398,0.001903484,0.0016761737,0.0078290265,0.0068508339,0.0046458041],[-0.000893628,0.0003497284,-0.0036565859,0.000364831,-0.0006250325,0.0001568323,0.0007657105,0.0009855394,0.0009009248,0.0040034551,0.0023607098,0.0025801835],[-0.0003615312,-0.0020415997,-0.0055582897,-0.0044520368,-0.0001213911,-0.0014690557,-0.0003081476,0.0000616249,0.0038505601,0.0039391147,0.0000157073,-0.0027928359],[-0.0005442948,-0.0025314022,-0.0055494044,-0.0110966617,0.0020048231,0.0016985485,0.0046449597,0.0080135879,0.0016914838,0.0031261736,0.0016824009,0.002379891],[-0.000145507,-0.0003496722,-0.0049002178,-0.0109004586,0.002482857,-0.0001034361,-0.0002579386,0.000088032,0.0008510337,0.0054916658,0.005036638,0.0015874623],[0.0003299129,0.0005104527,-0.005893482,-0.0093144359,0.0012929928,0.0032953859,0.0012896775,-0.001438049,-0.0021815528,0.0024021397,0.0012141653,-0.001584429],[-0.0005807795,0.0009810244,-0.0095590706,-0.0064176464,-0.001381654,0.0014516843,0.0003403707,0.0017219423,0.0018041543,0.0012990764,0.0009402081,0.0004507909]],[[-0.0027658701,-0.0011731834,-0.0020466941,-0.0030784515,0.0023275728,0.0040555545,-0.0001846212,0.00118252,0.0053059221,-0.0021170268,-0.002375838,-0.0007255144],[-0.0058373119,0.0004078163,0.0015064347,0.001504515,-0.0016453753,-0.0003663751,0.0054850992,-0.0009489593,-0.0036619925,-0.0049675698,0.0014326698,-0.0034309438],[-0.0047266302,0.0002298308,0.0014892366,0.0098583644,0.0108850515,-0.0015655364,0.0041048166,-0.000750083,0.0009733112,-0.0017778331,0.0029395377,0.0059801788],[-0.0022319565,-0.0039024187,-0.0027091103,0.0022767217,0.0047919033,0.0122836572,0.0010363764,-0.0016664926,0.0021924349,0.0003752499,-0.0022584908,-0.0025418536],[-0.0000525715,-0.0113779756,-0.0067223954,-0.0084756055,0.0027231231,0.0023955651,-0.0035650639,0.0030599756,0.0019950294,-0.0000924468,-0.0003559486,-0.0007202605],[0.0008220958,-0.0062361528,-0.0077527641,-0.0068461756,-0.0072809829,-0.0015173111,0.0009075477,-0.0000881897,-0.0083780701,-0.0024451335,-0.0003483582,0.008066814],[0.00018668,-0.000807782,-0.0089232076,-0.0020284606,-0.0110406194,-0.0040775027,-0.0133795677,-0.0182739184,-0.01089683,0.0014146628,-0.0038606459,-0.0098860392],[0.0006496489,0.000182304,-0.0050549195,0.0009718308,0.0042563964,0.0030248393,-0.0032474994,-0.0044104985,-0.0063551676,-0.0003517081,-0.0011411999,-0.0023083997],[0.002655099,0.0007081658,-0.0015036837,-0.0005959022,-0.0016551905,0.0061043906,-0.0043498361,-0.00382865,0.0004407344,-0.0000472934,-0.0004715018,-0.0036435074],[0.0045431083,0.0017481418,-0.0042648437,0.0011449493,-0.0014690861,-0.0029312356,-0.0028390898,-0.0101323897,-0.0137715777,-0.0004420148,-0.0006519989,-0.0044807433]],[[-0.0034206872,-0.001139779,0.0008645847,0.0054319491,0.0041531795,-0.003084496,-0.0006645673,-0.00064329,-0.0045534876,-0.0023031338,-0.0033187566,-0.0119152495],[-0.0010599768,0.0016790049,-0.0024269493,-0.0017644811,-0.0002615699,-0.0002451272,-0.0047420493,-0.0076772208,-0.0035695003,-0.0111475935,-0.0080903749,-0.005250669],[-0.0017874368,0.0007366219,-0.0035267484,-0.0094535407,-0.0043891765,-0.0014180792,-0.0088053951,-0.004328366,-0.0004043242,-0.0054257185,-0.0014818201,0.0031595098],[-0.0030505246,0.0010340605,-0.0016835685,-0.0016413175,0.0016329944,0.0001996218,0.0064371349,0.0020072868,-0.0061827947,-0.0037718796,-0.0018674314,-0.0020980049],[-0.0025389079,0.0054955312,0.001397446,-0.0003678144,0.001522707,-0.0003449619,0.0017336821,-0.0032143598,-0.0047028685,-0.0015960259,-0.00276723,0.0018175098],[-0.0026239914,0.0042369663,0.0031346808,0.006070143,0.0118854951,-0.003692437,-0.0048266547,-0.00187113,0.0013068428,0.0029321741,0.0009351129,0.002858329],[-0.0006460091,0.0002285304,0.003331308,0.0050585816,0.0094035405,-0.0029380151,0.0022789764,0.002990226,0.0017012952,-0.0004930809,-0.0081817982,0.0050485915],[0.002193075,-0.0002048297,-0.000329902,0.0015244553,-0.0034291161,-0.0043094697,-0.0001457582,0.0026834956,0.0023056201,-0.008041471,-0.0056323112,-0.0027695049],[0.0031616325,0.0009013374,-0.0022389501,0.0059232011,0.0032387396,-0.0076817326,-0.0124987416,-0.0022063304,0.0014417809,-0.0014035025,0.0053496828,0.0085397371],[0.0025915353,0.0033104173,0.0034140415,0.0006594741,0.0034378297,-0.0015846403,-0.0033753991,0.000607238,-0.0012564453,0.0010163488,0.000877306,0.0036354835]],[[-0.0026278883,0.0033228784,0.0046558643,-0.004816715,-0.007433745,0.0015446124,0.0005918669,-0.0003350169,0.003354065,-0.0013940053,-0.0046404949,0.0030628055],[-0.0000072263,-0.002071819,0.0051068353,-0.0029090365,-0.0024924054,-0.0027929903,0.01148147,0.0119193466,0.0050381685,-0.0060820058,-0.0091377068,-0.0029584081],[0.0023638488,-0.0028333001,0.0044327145,0.0025094669,-0.0044375505,0.0023321489,-0.0051607886,0.0041442748,-0.000722725,-0.006640641,-0.012295534,-0.0169409586],[0.0034561738,-0.0027012278,0.0026641486,0.0021680647,-0.0051004948,-0.0058387577,-0.0013574752,0.002254838,0.0013606609,-0.0110497761,-0.0053684747,0.0003737106],[0.0027725603,-0.0007503806,0.000172683,0.0014649986,-0.0030991526,-0.0019892339,0.0034681655,-0.0013283067,-0.0000189588,-0.001014685,0.000308505,-0.0033975304],[0.0034134695,0.0019143658,0.0001453316,0.0019817107,-0.0037867947,0.0033695523,0.0049296667,0.0013663781,0.0023892945,-0.0039945398,-0.0003464222,-0.001008752],[0.0025177598,0.0016326929,0.0015653755,0.0102991666,-0.0014351781,0.0113651434,0.0065170008,0.0007276997,0.0039776275,-0.0049956343,0.0086587464,0.0057445015],[-0.000168969,-0.0009593914,0.0022946272,0.0094175074,-0.002813714,0.0032554184,-0.0000258166,-0.0001759266,0.0007482331,-0.0014140618,0.0025631196,0.0047625247],[-0.0007548684,-0.00546444,0.0023319647,0.0044273066,-0.0027748937,-0.0053949634,0.0049305177,-0.0042758043,-0.0028935103,-0.0071248172,-0.0049321581,0.0030376443],[-0.0007738975,-0.008782411,0.0035375531,0.0042463266,-0.0024900004,-0.0008134797,0.0053751809,0.0014116414,0.0033601748,-0.0026239073,-0.0005662505,0.0026867639]],[[0.0099640466,0.0063742732,0.0012364355,0.0010150353,0.0021143819,-0.0019544699,-0.000878982,0.0007661311,-0.0073180053,-0.0094000446,-0.0081848291,-0.0036284302],[0.0083870936,0.0053368642,0.0015379869,0.003636525,0.0038190923,0.0032415821,-0.0097949759,-0.004544081,-0.0036208795,-0.0003473119,-0.0015487993,-0.0007872718],[0.0056503186,0.0026131954,0.001750619,-0.0006201309,-0.0023304191,-0.000111945,0.0074266058,-0.0000625893,-0.0007515283,0.0023408525,0.0012660023,0.0006487614],[0.003251256,0.0051543934,0.0046077854,-0.002347777,-0.001454009,-0.001815914,-0.0058546381,-0.0044991159,0.0009535252,0.0066173791,0.0026435629,-0.0003796562],[0.0007125472,0.0062830965,0.0088088523,0.0070135904,-0.0005216448,-0.0002182015,-0.0024024941,0.0004971515,0.0018258732,-0.0013002971,0.0004539638,-0.0002799026],[-0.0012500427,0.0021264206,0.0100310414,0.003246359,-0.000696326,0.0033092514,-0.0007024118,0.0005313168,0.0008313724,-0.0004316156,-0.0002560398,-0.0071235549],[-0.0015141359,0.0014779608,0.0095759283,-0.0022326259,0.0010674339,-0.0060481742,-0.0000613692,0.0065424047,0.0035264239,0.0009478789,0.0017012968,-0.0032869444],[-0.0025282478,0.0013315895,0.0079904122,-0.0010133345,-0.0004964233,-0.001867352,0.0036770128,0.0018148975,0.0024502809,0.0043155751,-0.0008262463,-0.0012720825],[-0.0053917761,0.0033444842,0.0073041511,-0.00044017,-0.0001016482,0.0036769193,0.0106283823,0.0117488338,0.0031925478,0.0061734736,-0.0011601881,-0.0063494452],[-0.0057799665,0.0027428273,0.00687232,0.0003668964,0.0019029107,0.0038776713,0.0004989372,0.0063915681,0.0098636938,0.0007504967,-0.0005992648,-0.0022922951]]],\"magnitude\":[[[0.0031544943,0.0077773667,0.0086708656,0.0035584814,0.0037187485,0.0028989776,0.0018220599,0.0022460573,0.0034882988,0.0154274424,0.0188998419,0.0135942553],[0.0031715216,0.0055215039,0.0068043536,0.0023697572,0.0022242192,0.0033503978,0.0041486434,0.0023443897,0.0066495301,0.0227784733,0.0174489194,0.0124272928],[0.0030046558,0.00143413,0.0056151989,0.0039848589,0.0017328702,0.0019785402,0.0032457293,0.0028328474,0.0018151655,0.0116186473,0.0095718142,0.007693359],[0.0018604983,0.0018258854,0.0052486709,0.0014187076,0.0011896583,0.0048802956,0.002361872,0.00300982,0.0036963768,0.0083424981,0.0075141025,0.0054366459],[0.0013900272,0.0023253818,0.0057728713,0.0019444036,0.0014343568,0.0012006534,0.0018889703,0.002369535,0.0023484501,0.004696782,0.0030447543,0.0034502105],[0.000923326,0.0034980896,0.0084777991,0.0056502371,0.0026577717,0.0021952133,0.0015698452,0.0011813439,0.0041452367,0.0043458268,0.0039745586,0.005445028],[0.0006854168,0.0042937373,0.0112838726,0.0129139076,0.0034656822,0.0023362074,0.0047290951,0.0083940074,0.0026995701,0.0036086948,0.003074599,0.0047924168],[0.0006822508,0.0021219549,0.013129468,0.0119921009,0.0033728327,0.0020481479,0.0016409788,0.001121498,0.0018922088,0.0059371452,0.0059482801,0.0027778509],[0.0011909795,0.0019280042,0.0135068485,0.0093934861,0.0025120107,0.0036801129,0.0021062734,0.0022619676,0.0026528447,0.0027361836,0.0019625236,0.0028766204],[0.0016529841,0.0019416366,0.0139885739,0.0065981764,0.0019758503,0.0022428963,0.001225394,0.0042166209,0.0025009567,0.0022256897,0.0021598319,0.0024583703]],[[0.0061144678,0.0055887874,0.0079843748,0.0036525889,0.0058412079,0.005439295,0.0034190735,0.0057544084,0.0058970595,0.003751219,0.0044179609,0.0037401901],[0.0089042795,0.0048403061,0.0029653804,0.0051819784,0.0057134584,0.0073962993,0.0068387867,0.0053113392,0.0078254493,0.0095045031,0.0071209625,0.0062847261],[0.0083627167,0.0050926207,0.0029651475,0.0103586571,0.0119244159,0.0026064212,0.0062933266,0.0057013279,0.0018525574,0.0052068171,0.0050020599,0.0066451752],[0.0060119262,0.0072291937,0.0041741846,0.0042245844,0.0054168879,0.0130656716,0.0038294069,0.004592463,0.0035226412,0.0057314761,0.0055956565,0.0045141224],[0.0040812458,0.0122963486,0.0072892922,0.0123831482,0.0039187314,0.0031721695,0.0044643671,0.004839966,0.0056469809,0.0047613763,0.0044382354,0.0067515631],[0.0035945816,0.0067205991,0.0089593752,0.0102051327,0.011967656,0.0047902128,0.0049439606,0.0044786931,0.008689072,0.003465418,0.0023051561,0.0112901551],[0.0030046492,0.0027189746,0.0112168264,0.0050791912,0.011507065,0.0069561395,0.0135388192,0.0189644128,0.011265324,0.0053202171,0.0081217252,0.0117457914],[0.002352973,0.002219415,0.0077855816,0.0055160953,0.0062097291,0.0067272027,0.0043347281,0.0048527321,0.0069470261,0.0025153391,0.0035027056,0.0032887402],[0.0035773674,0.0022181759,0.0051943429,0.0033129597,0.0062749272,0.0086319873,0.007107373,0.0052638312,0.0023977563,0.0031289056,0.0055635409,0.0061476955],[0.0050812509,0.0028676986,0.0076587408,0.0056199834,0.0037158256,0.0070332288,0.0035336888,0.0105856148,0.0144045318,0.0032756165,0.0053021322,0.0070070103]],[[0.0052548339,0.0029786169,0.008362458,0.0085342987,0.0072855329,0.0072443581,0.0043613717,0.0065678653,0.0086427349,0.0061267231,0.0077829824,0.0119508126],[0.0029824708,0.0054071358,0.0078302714,0.0063680316,0.0086727414,0.010230558,0.0059755437,0.008038737,0.0110723071,0.0117169899,0.0111089862,0.0053907698],[0.0037803188,0.0032779415,0.0061798809,0.0097439479,0.011610724,0.0027262297,0.0102339212,0.0062638612,0.0025019387,0.0064064913,0.0063593119,0.0052819691],[0.0049014337,0.0053707675,0.0037142821,0.0022651219,0.005281581,0.0074220688,0.0072098218,0.0078205974,0.0062038712,0.005669881,0.0047253608,0.0053018088],[0.003964666,0.0086135771,0.0030580568,0.0025994979,0.0047880386,0.0030452106,0.0049833026,0.0069450702,0.0063961357,0.0051019468,0.0034544175,0.0062271831],[0.0030096201,0.0094870731,0.0043292331,0.0073601642,0.0120017616,0.0047603221,0.0075818874,0.003854836,0.0030176017,0.0048504059,0.0059517616,0.0094205218],[0.0020120645,0.0087779923,0.0047295528,0.0088861761,0.0101348951,0.0095582513,0.0102329437,0.0079992197,0.0037520855,0.0043029538,0.0094420208,0.0061020015],[0.0025074315,0.0046126554,0.0055058945,0.0101141105,0.0067901587,0.007684452,0.0033872663,0.0045617176,0.0059121349,0.0088896793,0.0072950534,0.0090602187],[0.0037377934,0.0051272626,0.0069233082,0.0077719897,0.007995333,0.0090013321,0.0128576145,0.008347808,0.0042600565,0.0069021457,0.0069057301,0.0099385885],[0.0048787899,0.0038007544,0.007390896,0.0064001114,0.0065619949,0.0047570072,0.0041229422,0.0051581789,0.0048108467,0.0033692559,0.006867855,0.006526902]],[[0.0049669292,0.0077820887,0.0076418315,0.0055357835,0.0092954843,0.0049721557,0.0031232966,0.0021422378,0.0056830921,0.004528276,0.007638434,0.0074732349],[0.0065485998,0.0046647957,0.0072731625,0.005827871,0.0058468537,0.0083370149,0.01148147,0.0120584395,0.0083626897,0.0088605566,0.0106034408,0.0063226489],[0.0071144267,0.0037783085,0.004757255,0.0050866299,0.0061785557,0.0029492936,0.0073747453,0.0046292844,0.0038390342,0.0078434269,0.0145643856,0.0171370351],[0.0066985025,0.0041047912,0.0038846021,0.0037272309,0.0067234259,0.0058885009,0.0027627372,0.0041772388,0.0038287615,0.0112048222,0.0062390438,0.0046058924],[0.0057905117,0.0041192025,0.0047822564,0.006634934,0.0072088846,0.0054529541,0.004220382,0.003435855,0.0026797827,0.0035742856,0.0054367485,0.0051173161],[0.0039438588,0.0059920044,0.0044434294,0.0045227373,0.0057278842,0.0081377725,0.0069815324,0.006564272,0.0044029575,0.0062761713,0.0041096114,0.0035837822],[0.0036121846,0.0052597577,0.0040869341,0.011432629,0.0055234899,0.0114575481,0.008502821,0.0054745979,0.0058795894,0.0069116752,0.0103045543,0.0077538873],[0.0036194887,0.0039581905,0.0040527242,0.0112262061,0.0079623026,0.0046714694,0.0066107164,0.0046059307,0.0029095158,0.0053794721,0.0040950562,0.008311047],[0.0021687425,0.0081711828,0.0050583849,0.0050102887,0.0067349911,0.0067024692,0.0060638851,0.0062296469,0.0036647882,0.0096279331,0.0078797086,0.0065777749],[0.0020836899,0.0098449669,0.0049238742,0.0073376191,0.0046066609,0.0046760761,0.0054531011,0.0036909318,0.0040915554,0.0040776494,0.0029534491,0.0048793692]],[[0.0100924883,0.0073255837,0.0042523203,0.0039415736,0.0035138657,0.0030348138,0.0023403352,0.0028258785,0.0082547622,0.0096941159,0.0082546164,0.0056327567],[0.0087672453,0.0069813834,0.0029758595,0.0053453712,0.005227322,0.0035932209,0.0104573266,0.0090022146,0.006494302,0.0074010627,0.0064266294,0.0028731894],[0.0062036025,0.0040539338,0.0021500659,0.0028620686,0.0062029351,0.0033255022,0.0075586939,0.0032610371,0.0014839862,0.0052631623,0.0042909769,0.0048357997],[0.0041938566,0.0056415141,0.0048174155,0.0030732471,0.0036962028,0.0028210136,0.0065790716,0.0049211522,0.0031432381,0.0069148795,0.0032345304,0.0038501192],[0.0029587088,0.0071225582,0.0088359865,0.0096000769,0.0014086994,0.0031547307,0.0034852718,0.0023557344,0.0039809224,0.0052078065,0.0066794069,0.0035862591],[0.0020807486,0.0054579442,0.0100519017,0.0065821431,0.003322166,0.0071222525,0.0034611697,0.0028633643,0.0055562751,0.0033567132,0.0041336782,0.0080804885],[0.0016056436,0.002767636,0.0097702238,0.0061938338,0.0042472902,0.0073454772,0.0038859632,0.0069942233,0.0046841928,0.0038662267,0.0057628322,0.0053519445],[0.0025579566,0.002254104,0.0080882785,0.0032757125,0.0032770284,0.0050152975,0.0049496904,0.0031349756,0.0031203719,0.0049232746,0.0033167617,0.0057811608],[0.0056369347,0.0037000688,0.0074042198,0.002470894,0.0039617715,0.0058183411,0.0106283823,0.0121821668,0.0039139295,0.0072471789,0.0038261325,0.0063649789],[0.0061319003,0.0029436474,0.0073357448,0.0037332099,0.0032422509,0.0048425741,0.0017830641,0.0071471048,0.010090861,0.0016500342,0.002596875,0.0036617914]]],\"nonzero\":[[[5,14,11,6,7,6,3,8,8,15,20,17],[5,12,11,4,5,7,5,3,7,17,15,16],[5,4,10,7,1,4,5,3,2,14,11,12],[4,4,12,3,0,8,5,5,6,12,10,6],[1,2,10,1,1,3,3,2,5,10,8,7],[0,5,11,9,4,2,0,1,9,7,9,13],[0,7,14,13,4,2,8,11,1,6,5,8],[0,2,14,13,7,2,5,0,2,11,9,3],[0,1,14,12,5,8,4,5,6,3,1,5],[1,2,15,14,5,7,2,7,3,4,6,4]],[[10,11,10,7,10,11,6,10,11,4,4,5],[10,11,7,10,13,13,11,6,11,15,9,10],[9,8,6,12,13,6,11,10,6,8,4,10],[9,12,8,8,9,16,8,8,5,10,8,5],[6,12,10,16,4,6,6,10,11,7,10,12],[6,8,13,16,12,9,9,10,11,4,5,15],[4,4,10,8,13,11,14,16,11,10,12,12],[5,4,12,7,12,12,5,11,11,5,6,5],[10,6,11,9,9,13,7,8,6,5,8,10],[9,5,9,8,7,15,7,17,15,12,9,11]],[[9,5,11,10,11,10,9,11,12,8,9,11],[8,10,12,14,13,14,7,9,14,9,12,9],[6,7,8,11,13,6,10,10,5,9,9,6],[8,7,8,5,12,12,9,12,10,9,10,9],[5,12,6,2,10,4,11,9,11,7,7,11],[8,14,8,13,16,7,10,8,8,8,10,10],[5,14,8,11,11,14,12,10,5,7,12,9],[6,10,8,11,10,9,6,7,12,14,11,10],[7,7,7,12,7,14,15,13,9,9,9,14],[7,5,12,11,12,8,5,7,10,10,9,8]],[[8,13,11,6,12,6,5,4,11,5,8,9],[10,8,12,9,10,11,18,15,9,10,11,10],[11,6,8,8,8,8,10,7,7,13,15,18],[11,9,9,8,9,11,7,7,8,18,11,10],[12,7,8,11,8,11,7,6,6,6,11,11],[10,12,7,9,6,11,13,10,6,9,6,7],[10,6,4,14,6,15,10,5,6,11,13,12],[7,7,8,16,14,9,9,6,4,7,8,13],[3,15,8,8,8,13,8,12,8,11,12,10],[2,17,6,9,9,10,10,6,5,9,3,10]],[[13,11,8,8,5,5,3,6,12,13,11,6],[12,12,3,8,8,6,15,13,7,9,8,5],[7,10,3,4,11,5,12,6,2,8,8,7],[6,10,11,7,7,3,11,9,8,11,4,8],[5,9,14,15,2,8,4,4,8,7,10,6],[4,7,19,14,3,13,6,6,10,7,5,11],[5,5,18,7,8,12,4,6,7,7,9,7],[7,5,15,5,8,9,9,7,7,10,6,7],[13,6,12,3,7,6,16,17,10,10,4,10],[10,6,12,4,7,9,3,8,11,4,3,9]]]},{\"rat\":\"Stella\",\"target\":\"odor\",\"n_channels\":21,\"signed\":[[[-0.0080600231,-0.0006391472,-0.0122427072,-0.0029460898,-0.0042641221,-0.0004846786,0.0030739111,0.0001436123,0.0032472597,0.0116925234,0.0055268322,0.0012847858],[-0.0052782452,0.0035419531,-0.009267423,-0.0034824908,-0.0100026453,-0.0068484425,0.0030785745,0.0008295219,0.0040972933,0.0057755637,0.0038060385,0.0011254678],[-0.0032141131,0.0007897367,-0.0089543859,0.0015739277,-0.0074775735,0.0006666293,0.0037199219,0.0104580159,0.0054497936,0.0118598235,0.0044920529,0.0027706179],[-0.0022759179,-0.005847382,-0.0093319159,0.0014775521,-0.0105283512,-0.0015961822,0.0037614312,0.0098704194,0.0041111314,0.0038905033,0.0013254435,0.002204401],[-0.0026605965,-0.0070860056,-0.006922331,-0.0026355955,-0.008411421,-0.0022526247,0.0014197655,0.0015064868,0.0023569803,0.0111840062,0.0103455413,0.0029207909],[-0.0002348546,-0.0035505059,-0.0046684627,0.0012584667,-0.0047944933,-0.0030053133,0.0006591914,0.00309771,0.0021399736,0.0079045654,0.0026224635,0.0047002228],[-0.0003183255,-0.0017789951,-0.0058551649,0.0045393589,-0.0004003492,0.0029757221,0.0020895177,-0.0031753192,-0.0040872041,-0.001304259,-0.0000731457,0.0025862745],[-0.0000630021,-0.0008849402,-0.0102966345,-0.0003154312,0.0000501578,0.0022107438,0.0055140638,-0.0001266994,0.0011184178,0.0025722828,0.0013419672,0.0027011211],[-0.0013308196,-0.0038553691,-0.0136714573,-0.0008630845,-0.0015586806,0.0018309322,-0.000546724,-0.0064666323,0.0025839811,0.0025591534,0.0006877483,-0.0007467714],[-0.005339898,-0.0112110787,-0.0138695767,-0.0047481714,0.0000952004,0.0064464476,0.0011510691,-0.0072313083,0.0013348599,0.0087945786,0.0029759294,0.0003827353]],[[-0.0021413767,-0.0003100437,0.0131873235,0.0092716157,-0.0016411998,0.0020720209,-0.01542592,-0.0109940702,-0.0044001979,-0.0252281923,-0.0087200501,-0.00207247],[-0.0049701203,-0.0006488296,0.0054295282,-0.0005276806,-0.0036080962,-0.0012208254,-0.0048795029,-0.0093140569,-0.0063051694,-0.0001393907,0.0006838059,0.000289337],[-0.0035075549,-0.0046793759,0.0025319843,-0.006600202,-0.0005783631,0.003620761,0.0068451571,-0.0073190763,-0.0088462874,-0.0022577128,-0.0103758744,-0.003146664],[-0.001216414,-0.0051610134,0.0030735719,0.0032934128,0.0107551734,0.0011870623,-0.0064900815,-0.0134889073,-0.0077099382,0.0082433868,0.0045279087,0.0006622625],[-0.0030355037,-0.0023243001,0.0016599093,0.000496643,-0.0058043997,-0.0004357114,-0.0085301139,0.0050931967,0.0112511603,0.0003987235,-0.005924803,0.0009343278],[-0.0028288605,0.0012573449,-0.001006266,0.0010789521,-0.0098966288,-0.0081392229,-0.0080840329,-0.0001193606,-0.005931557,0.0004378336,0.0089027198,-0.0032633675],[-0.0012285119,0.0014922351,-0.0004102795,-0.0024404999,0.0000932567,0.0008485151,-0.0081804297,-0.0050980205,0.0009829863,-0.0011825827,-0.0058427332,-0.0101852973],[-0.0004183535,0.0004564552,0.0013102699,-0.0076708659,-0.0078886731,-0.003730584,-0.0041114914,0.0043855308,0.0061856559,0.0031713965,-0.0028541457,-0.0046058117],[0.0018209718,-0.0013351158,0.0041793427,-0.0038406502,-0.0050088328,-0.0011604144,-0.0000721488,-0.0064595011,-0.007510088,-0.005692972,0.0103380885,-0.0053029616],[0.0058968023,-0.0015605818,0.0063030898,-0.0050770855,-0.0038591849,-0.0077967784,-0.0101987366,-0.0005314,0.0036195495,-0.00673554,-0.0004569415,0.0049163133]],[[0.0015915811,-0.0068298347,-0.0026006067,0.0010453096,0.0068907311,0.0080514156,0.0092682696,0.0020660573,0.0036325763,-0.0003695908,0.0035427439,0.0067811244],[0.0021855505,-0.0080715246,0.0013189679,0.0035654912,0.0078101953,0.012358379,0.0084402989,0.0057491513,0.0014028083,-0.0027282297,0.0068345499,0.0056092384],[-0.0000243076,-0.0008080412,0.0040689307,0.0043287741,0.0003612162,0.0044545534,-0.0030144997,0.0015721255,-0.0019310364,-0.0129888592,0.0041736555,0.0031102956],[-0.0009513427,0.00672336,0.0026846909,0.0031984582,-0.0081717893,0.0001053375,0.0015469635,0.0070869494,-0.0068316865,-0.0017560061,-0.0024067571,0.0018767276],[-0.0010998972,0.0023009803,0.0005725777,-0.0003543184,-0.0040418601,-0.0070032783,-0.002649112,-0.0075386086,-0.0066981925,-0.0042517485,0.0038031533,-0.0080206857],[-0.0002955876,-0.0001326414,0.0000199855,-0.0072270961,0.006662004,0.0044681466,0.0016401886,-0.0002719321,-0.0021978543,-0.0069036551,-0.0132483452,-0.0064525733],[0.0004455783,-0.0006778046,0.0003596387,-0.004600342,0.0001271824,-0.002856948,-0.0005011526,0.0017105529,0.0007310249,0.001070649,-0.0001262201,0.0035120947],[0.0017216848,0.001558094,0.0017049509,0.003318211,0.0067304588,0.0095100139,-0.0144457713,-0.0189585991,-0.0140347914,-0.0073491117,0.0038269479,0.0097187731],[0.0020998169,0.0033533652,0.0009238288,-0.0016471637,0.0023430291,0.0052030721,0.0013065114,0.001369945,-0.0094715883,-0.011210108,-0.0104690092,-0.0025183322],[0.0026409548,0.0045070977,-0.002378114,-0.0000429267,0.0070420693,-0.0015701725,0.0034705777,0.0152916901,0.0047925475,0.0017790476,-0.0035820446,-0.0024299078]],[[0.0052340804,0.0070790857,0.0044344255,-0.0022833452,-0.0029999106,-0.0099233538,0.0000921787,0.0164435892,0.0017456762,-0.0020857003,-0.0134894334,-0.0063144248],[0.004169935,0.0065104658,0.0055396502,0.001856471,0.006163557,0.0023889962,-0.0027166492,0.0013739101,-0.0063726011,-0.0104696677,-0.0111393319,-0.0049890625],[0.0036917192,0.0013414993,0.0031570407,0.0007698777,0.0101308546,-0.0006610752,-0.0038071014,0.0000425049,0.0034312837,-0.0036255769,0.0015966213,0.0030588521],[0.0019794714,-0.0000139791,0.0018768015,-0.007592519,0.0059559481,0.0037977968,0.0031048445,-0.0010788938,0.0063163832,-0.0007458577,-0.0038868428,0.0015862039],[0.0056478499,0.002131509,0.0028801531,-0.0005596264,0.0116952492,-0.0012823006,0.0003416875,0.0014164294,-0.0054503971,-0.0031948189,-0.0005709958,0.0039206087],[0.0055861611,-0.0006947457,0.0035075067,-0.0001390421,0.0024054499,-0.0083338893,0.0068083601,-0.0005810492,0.0026254807,0.0015951413,0.0030367868,0.0068167884],[0.0031802479,-0.0010161573,0.0037517366,0.0001584308,-0.0057577819,-0.0116030232,0.0055836351,0.0093570145,0.0030454838,-0.0021447919,-0.0067932394,-0.0035204598],[-0.0000423651,-0.0002433511,0.0051598013,0.0042915462,-0.0038619273,-0.0064642682,0.0069448564,0.0017885183,0.001152512,-0.0002306629,0.0022786582,-0.0017141925],[-0.0012326703,-0.0009785293,0.006056286,0.0024255836,-0.0020830795,-0.0045197252,0.0003295198,0.0145690936,0.0069146932,-0.0003570583,-0.0002972114,0.0023377041],[-0.0031813957,-0.0058629165,0.0049334038,0.0063484487,-0.0073238547,-0.0008729626,0.0021485461,-0.0057042102,-0.0099976675,-0.00201296,-0.0028818454,-0.0055811547]],[[0.0033757387,0.0006999404,-0.0027784351,-0.0050874907,0.0020145014,0.0002845963,0.0029915608,-0.0076591888,-0.0042253142,0.0159909601,0.0131399071,0.0003209846],[0.0038928799,-0.0013320642,-0.0030207233,-0.0014117909,-0.000363011,-0.0066781069,-0.0039227216,0.0013614738,0.0071776688,0.0075617246,-0.0001850626,-0.0020349808],[0.0030542562,0.0033561811,-0.0008035698,-0.0000723775,-0.002436134,-0.0080808686,-0.0037434778,-0.00475357,0.0018962462,0.0070123255,0.0001135448,-0.0057931014],[0.0024642032,0.0042990148,0.0016968518,-0.0003769041,0.0019890191,-0.0034940144,-0.0019231577,-0.0023895674,0.0041141099,-0.0096320264,0.0004402476,-0.0063295949],[0.0011481476,0.0049778162,0.0018096909,0.0030528972,0.0065624322,0.0109739147,0.0094177728,-0.0004775042,-0.0014595509,-0.004136162,-0.0076528957,0.0002449583],[-0.0022268583,0.0031205479,0.0021472366,0.0050287193,0.0056236682,0.0150102786,-0.001023707,-0.0021253681,0.003363957,-0.0030338851,-0.0013136245,-0.0018010706],[-0.0020789888,0.0019807219,0.0021540689,0.0023430523,0.0059376919,0.0106357341,0.0010084296,-0.0027942279,-0.0006722912,0.0035609843,0.0128353381,0.0076073879],[-0.0011979641,-0.0008862579,0.0021216125,0.0003765398,0.004969984,-0.0015259055,0.0060983429,0.0129112496,0.0055782059,0.0018360954,-0.0045934276,-0.0060998903],[-0.0013572989,0.0028156492,0.002512,0.003925315,0.0063075634,-0.0013538651,-0.0010171584,-0.0030129053,0.0074830023,0.0147009848,-0.0002596161,0.0062303611],[-0.0000164634,0.0141274794,0.0050111971,0.0035197348,0.0040457702,0.003793466,0.0034285437,-0.0018247718,0.0002507106,-0.0018251262,0.0039449022,0.002712014]]],\"magnitude\":[[[0.0094995735,0.0058997008,0.0129767545,0.0050609497,0.009114242,0.0056718024,0.0037768921,0.003787971,0.0042062118,0.0130899568,0.0064279965,0.0039506523],[0.0057047673,0.0066906965,0.0096978178,0.0049589089,0.0146335809,0.0106664579,0.0036877894,0.0028902903,0.0051740816,0.0072229299,0.0048148975,0.0029878283],[0.0039637925,0.0061030923,0.0095442314,0.0033685639,0.0094643853,0.0034722646,0.0046987855,0.0108520196,0.0084898181,0.0138038734,0.0071462772,0.0065445798],[0.0037784748,0.0062876862,0.0093827936,0.0049452635,0.0109277562,0.0041004369,0.0044254273,0.0107542523,0.0065307495,0.0054271614,0.0029186427,0.0029615578],[0.0037446389,0.0076711991,0.0069553838,0.0038297236,0.0087737232,0.0050392189,0.0026915119,0.0035997198,0.0031710944,0.012784765,0.0125558384,0.00585336],[0.0014526478,0.0063275083,0.0050891749,0.0028619843,0.0060672972,0.0050233019,0.0036631168,0.0059573336,0.0026901605,0.007931493,0.0043053205,0.0061715746],[0.0011265197,0.0047525491,0.0073941661,0.0047737671,0.0026855329,0.0040347664,0.0023497936,0.0048263468,0.0041757092,0.0026947628,0.0017977888,0.0042532205],[0.0006555403,0.0023585771,0.0112567421,0.001584812,0.0018500086,0.0038814562,0.0057003099,0.0033920689,0.0035124024,0.0033729819,0.0043727994,0.0032650364],[0.0019284649,0.0043163135,0.0145236531,0.0013727717,0.0025500161,0.0043030052,0.001310326,0.0076219799,0.002972923,0.0038912109,0.0028581814,0.0028181562],[0.0055778857,0.0113093644,0.0151200423,0.0061721232,0.0025947395,0.0070877935,0.0021433397,0.007880673,0.0037801658,0.0094200818,0.0041615807,0.0045783913]],[[0.0039325453,0.0060456822,0.0137305751,0.0098943482,0.0061114504,0.0056877751,0.0165741737,0.0125330406,0.0066308104,0.0254658895,0.0105833335,0.0104180539],[0.0061446577,0.0044389323,0.0076046689,0.0064761439,0.0120650582,0.0065532661,0.0156153932,0.0100271705,0.008282537,0.0041435269,0.009161866,0.0054865158],[0.0039718527,0.0062807224,0.0078914273,0.0084148213,0.0114535717,0.0091856843,0.0125878359,0.0093063199,0.0101375098,0.0109243486,0.0105165624,0.0066265349],[0.0037134572,0.0066539318,0.0071559374,0.0081856117,0.0111826298,0.0067960788,0.0075423414,0.013589001,0.0151831996,0.0103239673,0.0114084261,0.0056944679],[0.0051428782,0.0049969591,0.0036748747,0.0030420678,0.0076298906,0.0098265618,0.0097966543,0.0068877809,0.0118975591,0.0077681536,0.006015264,0.0081653868],[0.004459064,0.0021381431,0.0027695013,0.0028579445,0.0103819089,0.0110204897,0.0085985635,0.0082209946,0.0078281127,0.0053653521,0.0117567762,0.0073797902],[0.0025772806,0.0027887117,0.002503449,0.005149856,0.0030667536,0.0098887836,0.0102350559,0.0087209718,0.0071352858,0.0068386808,0.0090939247,0.0112504212],[0.0017519628,0.0023774463,0.0031939249,0.0077878363,0.0087034671,0.0069116265,0.0082615432,0.0089732329,0.0109092899,0.006987249,0.009007021,0.0133365886],[0.0032476897,0.0037432889,0.0050191269,0.0060491983,0.0093745608,0.0051076235,0.0015383958,0.007082273,0.0107565033,0.0087482572,0.0127762684,0.0077739896],[0.0073929837,0.0050432585,0.0071573558,0.0075824724,0.0106007502,0.0081487631,0.0105641398,0.0065286854,0.00938158,0.0073012233,0.0094241714,0.0070703375]],[[0.0083151927,0.0075093376,0.0035226383,0.0031032281,0.0071410345,0.0082934616,0.0098645931,0.0076600454,0.0063763857,0.0069541866,0.0078727863,0.0120569954],[0.0078281629,0.0102139036,0.0030171377,0.004409765,0.0087663034,0.0136068726,0.0125708199,0.0096416322,0.0063748346,0.007836627,0.0090619057,0.0069696776],[0.0063974768,0.0077861999,0.0045990413,0.0059932714,0.0104410067,0.0066490216,0.0085511601,0.0049970206,0.0081204087,0.0141762005,0.0073301141,0.0051128752],[0.0042414179,0.0076683671,0.0032421463,0.0051045273,0.0114060413,0.0019117147,0.0048640889,0.0184389313,0.0178229924,0.0039543131,0.0054451496,0.0048360445],[0.0048502426,0.00445205,0.0018604502,0.0043529547,0.01220711,0.0087324603,0.0066613943,0.0077239763,0.0075804913,0.006082489,0.00673915,0.0094309965],[0.0037105374,0.0029815544,0.0012398198,0.0078010628,0.0079647448,0.0093134856,0.004529365,0.0025556004,0.005617971,0.0073979186,0.0137623077,0.0069886624],[0.0025529579,0.0028610045,0.0023794864,0.0073573813,0.0036286516,0.0080653005,0.0043715263,0.0108726006,0.0047600191,0.0053835629,0.0032297883,0.006752152],[0.0042160491,0.0030572659,0.002808938,0.003786649,0.0076164917,0.0141183051,0.0168624502,0.0207091666,0.0153792183,0.0075816036,0.0056448422,0.0101306306],[0.0053214521,0.004991943,0.0035645436,0.0022441235,0.0040937498,0.0088145509,0.0044836923,0.0043910914,0.0125541679,0.0127041303,0.0105955163,0.0044128274],[0.0067625519,0.0062146538,0.0049791187,0.0032022229,0.0072835817,0.0029267987,0.006194881,0.0152943429,0.0062570089,0.0062471406,0.0045937905,0.005134757]],[[0.0093598695,0.0072357883,0.0065473594,0.0053461933,0.0086634627,0.0116366822,0.0067209855,0.0186607378,0.0048712577,0.0056159509,0.0158995705,0.0108319667],[0.0084030953,0.0067397116,0.0064525106,0.0040765681,0.0103087952,0.0041843916,0.0107040216,0.0074394876,0.0090620222,0.0123096962,0.0139964469,0.0097203481],[0.0079098149,0.0024597029,0.0036870115,0.0023430041,0.0108713296,0.0020835232,0.0067392609,0.0063353136,0.0073742084,0.0076961395,0.0093912384,0.0055651958],[0.0088593699,0.0026237165,0.0024342569,0.0082358693,0.0074144414,0.0063430585,0.0061664227,0.0043021191,0.007209193,0.0042406745,0.0082697104,0.0040026587],[0.0070265628,0.0025471656,0.0033885909,0.003300977,0.0119941746,0.0067983429,0.0033718655,0.0058090818,0.0090584428,0.0040768883,0.0064420783,0.0067208323],[0.0062379783,0.0034115133,0.0037779528,0.0037930034,0.0076346512,0.0105595337,0.00901122,0.004698621,0.0030893873,0.003400899,0.0068818948,0.0083925788],[0.0034614726,0.0035322595,0.0052206149,0.0045791407,0.0064639721,0.0130767092,0.0064686885,0.0126725802,0.0043431193,0.0045252821,0.0080235517,0.0072993874],[0.0019161442,0.0021958936,0.0054511946,0.0055688948,0.007106067,0.0091799353,0.0092719037,0.002981643,0.0046436285,0.0042968774,0.0045707822,0.0065668409],[0.002028248,0.0022259818,0.0062466904,0.0035227519,0.0067680068,0.0071741675,0.0059874835,0.0157813963,0.0072193589,0.0061775546,0.0062185619,0.0047709158],[0.0038194302,0.0071050264,0.0051419055,0.0067575753,0.0099386734,0.0040288508,0.0037494899,0.0099674639,0.0110238353,0.0074407473,0.0132561104,0.0079083832]],[[0.0062838633,0.0050559236,0.0043830862,0.0064946386,0.0034628979,0.0036862118,0.0086705428,0.0088887878,0.0059018752,0.0177829201,0.0146509862,0.0078257865],[0.0059537523,0.0051102153,0.0049192511,0.0027523501,0.0046350315,0.0076264136,0.0081462781,0.0033136758,0.0149350231,0.0105789635,0.0091238692,0.0062200814],[0.0054471476,0.0043862576,0.0031570601,0.0023541849,0.005408223,0.0081500662,0.0072743265,0.0071724047,0.0062615796,0.0082670982,0.0044682237,0.0064338277],[0.0069006846,0.0049166934,0.0021967357,0.0029156558,0.0037684672,0.0064048791,0.0065746804,0.0069723053,0.0061461299,0.0127831072,0.0078756333,0.0092879911],[0.007844719,0.0050213721,0.0021063173,0.0052693716,0.0106016269,0.011852169,0.0096278382,0.0036990775,0.003111972,0.0048719661,0.008091552,0.0051979223],[0.0059240452,0.0033715276,0.0023676748,0.0051598736,0.0075890611,0.0152246829,0.0085542581,0.0057837076,0.0065139972,0.0064108294,0.0065118809,0.0030252845],[0.0033264388,0.0029116165,0.002650532,0.0029461181,0.0061614946,0.0107252558,0.0038930553,0.0041114643,0.0079748262,0.0057860001,0.0133693497,0.0078907696],[0.0022643345,0.0026768075,0.002750587,0.0014370526,0.0070760938,0.0058441872,0.0062543002,0.0131491893,0.0066222891,0.0030758341,0.0085247161,0.010127143],[0.0030890242,0.0056225399,0.0039456594,0.0066382627,0.0090695113,0.0133051735,0.0024148415,0.0041530164,0.0074990669,0.0147009848,0.0047323929,0.0072827689],[0.0027767054,0.0152813649,0.0054751953,0.0035197348,0.0053276116,0.0046881904,0.0041334616,0.0037390009,0.0032298144,0.0030375887,0.0079171006,0.0058517369]]],\"nonzero\":[[[10,9,15,9,9,8,4,2,5,11,8,3],[8,8,15,10,10,12,7,3,7,10,7,4],[7,7,13,8,10,9,11,15,12,11,9,13],[4,8,13,6,14,8,6,10,7,8,4,5],[3,6,11,8,10,3,4,8,7,12,12,8],[2,6,9,7,7,5,4,10,4,10,5,8],[1,6,12,6,3,4,5,6,8,3,2,8],[1,5,15,3,1,5,10,3,3,5,7,6],[4,9,17,3,3,6,3,11,3,6,3,3],[12,12,15,10,4,12,2,13,7,10,5,6]],[[6,8,11,11,9,10,12,12,8,16,13,11],[8,9,8,12,15,7,16,15,11,5,8,9],[5,9,9,9,12,12,11,13,12,11,14,8],[4,11,7,7,11,10,8,11,11,9,12,7],[7,6,4,1,10,10,13,8,14,11,6,11],[8,3,5,6,14,15,8,10,10,6,12,11],[5,6,2,6,4,9,13,7,9,9,7,12],[3,6,2,9,11,9,6,11,11,7,12,10],[8,6,8,8,12,7,4,7,11,8,13,12],[10,5,10,10,12,14,11,7,12,7,10,11]],[[14,12,3,4,6,10,10,7,10,5,8,12],[10,13,3,4,6,13,14,12,9,9,9,9],[6,8,5,10,10,8,11,6,8,12,8,6],[8,9,2,7,13,3,8,15,17,4,8,7],[9,5,2,6,11,11,9,8,9,7,6,10],[8,3,2,10,12,9,7,4,7,13,14,10],[4,4,2,13,6,8,5,8,7,8,5,10],[8,5,2,9,14,13,14,16,12,11,9,9],[8,11,5,3,4,10,8,7,11,10,9,6],[6,10,10,7,11,4,11,16,8,8,8,9]],[[12,9,8,6,12,12,9,18,7,7,12,8],[12,10,11,7,12,7,12,9,11,9,11,12],[14,2,7,2,12,3,9,9,10,7,11,10],[13,2,5,10,8,11,10,7,5,3,9,6],[10,4,7,4,14,7,6,8,9,5,8,12],[8,8,6,5,11,13,8,5,5,6,7,8],[8,6,8,10,11,9,10,14,6,7,13,11],[5,1,12,10,8,10,9,4,3,8,6,8],[5,3,8,5,9,6,7,13,7,5,9,9],[4,8,8,8,10,5,6,12,11,12,12,7]],[[7,7,7,9,5,3,11,8,8,14,15,7],[9,6,4,4,5,8,9,3,12,12,12,6],[11,8,4,2,7,10,10,10,7,10,6,8],[12,10,2,3,6,10,10,7,7,13,10,9],[11,8,4,9,13,14,11,10,4,6,8,6],[12,8,6,10,8,15,6,7,8,9,9,5],[4,5,3,3,10,13,5,5,13,7,12,7],[4,5,3,1,7,8,7,16,11,7,11,9],[7,11,5,8,11,14,6,5,10,18,9,8],[6,13,8,6,8,6,7,2,3,4,10,10]]]},{\"rat\":\"Stella\",\"target\":\"position\",\"n_channels\":21,\"signed\":[[[-0.0065675732,-0.000150014,-0.0118319035,-0.0020934127,-0.0037186538,-0.0007896452,0.003301979,0.0002449493,0.0023609647,0.0132747834,0.0061377692,0.001442043],[-0.0054154951,0.0044346186,-0.0101300607,-0.0031229886,-0.0091240734,-0.0081744485,0.0029681778,0.0002835516,0.0047121286,0.0070957993,0.0039096779,0.0006672228],[-0.0036045068,0.0010892812,-0.0099314485,0.0017383777,-0.0084640695,0.0006519818,0.0041935589,0.0085976438,0.0058361809,0.0120584689,0.0050706231,0.0038668293],[-0.0022783241,-0.005621482,-0.0101571641,0.0008473171,-0.0115194316,-0.001612543,0.0047297197,0.0094803852,0.0036451315,0.0039540954,0.0021809034,0.0015487829],[-0.0024743832,-0.0072417437,-0.007693882,-0.002164619,-0.0076337841,-0.0024483163,0.0012552942,-0.000239695,0.0022031261,0.0110214068,0.0102113434,0.0030597067],[-0.0006824007,-0.0038844592,-0.0052823219,0.0019073918,-0.0050690842,-0.0037228093,0.0009026654,0.002897611,0.0019610296,0.0072567453,0.0020203658,0.0046574008],[-0.0009036443,-0.0018508005,-0.0051761025,0.0042242848,-0.0000904939,0.0022831176,0.0024546696,-0.0029094305,-0.004845563,-0.0017244142,-0.0003493742,0.002328095],[-0.0001760557,-0.000949233,-0.0093791381,-0.0006065521,-0.0002715338,0.0018226528,0.0044682219,-0.0000127215,0.0004348152,0.0028650493,0.0005271669,0.001483467],[-0.0015868597,-0.003415305,-0.013062703,-0.0008528476,-0.0007896973,0.0017365868,-0.0009857451,-0.0055824496,0.0022285843,0.0030339733,0.0006771674,-0.0011335471],[-0.0054277382,-0.0109272574,-0.0133207802,-0.0050083214,0.0006409496,0.0077870143,0.0007196249,-0.0078107886,0.0000995927,0.0072857815,0.0032046593,0.0015132231]],[[-0.0010893811,-0.0001537643,0.0144918973,0.0104662334,-0.001523523,0.0024768648,-0.0155856138,-0.0138517525,-0.0059446795,-0.0235933733,-0.0087213259,-0.0023149806],[-0.0039499352,0.0005492524,0.0052028557,-0.0004632213,-0.0049112815,-0.0052985783,-0.0079979671,-0.0084860138,-0.0059128803,-0.0012043659,-0.0018026422,-0.0011399107],[-0.0035660814,-0.0052607314,0.0017754495,-0.0084503714,-0.0027653193,0.0005882643,0.0047913553,-0.0043574436,-0.0076092921,-0.0014452643,-0.0093251301,-0.002560832],[-0.0011240155,-0.0061124605,0.0023813474,0.0017430925,0.0107387519,0.0028387712,-0.0056067842,-0.0097635496,-0.0059393702,0.0072667309,0.004651117,-0.000012079],[-0.0019821761,-0.0022796361,0.0014477829,0.0009517278,-0.0057424916,0.0013834082,-0.0065167783,0.0062863422,0.0126827408,0.0002205607,-0.0048011014,0.0002824229],[-0.0026267623,0.0008374881,-0.0007589067,0.0017681798,-0.0108857997,-0.0089178029,-0.0105489822,0.0024998901,-0.0053699799,0.0001023781,0.0053155312,-0.0037242451],[-0.0018477252,0.0007243487,-0.0007555405,-0.0003195208,0.0015568465,-0.0004545665,-0.0101935942,-0.0046761656,0.0023480936,-0.0012599839,-0.008621402,-0.0137290426],[-0.0006680515,0.0004680344,0.0006654823,-0.0056090196,-0.0079446133,-0.0030391082,-0.0035830006,0.0030772642,0.0075730071,0.004175583,-0.0010228004,-0.0029172102],[0.001203656,-0.0016891192,0.0036437983,-0.004774853,-0.0063964679,0.0002594285,-0.0003671278,-0.0064072126,-0.0064579129,-0.0073147997,0.0126479828,-0.0038853023],[0.0046243726,-0.003391558,0.005853706,-0.005694585,-0.005492191,-0.0097041637,-0.0126368057,-0.0001158867,0.0020967096,-0.0068801461,-0.0026005258,0.00362802]],[[-0.0001634527,-0.0056506366,-0.0045663199,-0.0006341214,0.0068671184,0.0101038451,0.0116346114,0.0026950884,0.0013645416,-0.001537046,0.0030265761,0.0071715125],[0.0011672889,-0.0095931522,0.0015685355,0.0043093898,0.0103265034,0.0189959953,0.0122774463,0.0090587141,0.0003753058,-0.0026617895,0.0081284173,0.0084214148],[0.0002986124,-0.0023596985,0.0041633567,0.0064638933,0.0027760188,0.0043338334,-0.0077254332,0.0026237049,-0.0020955807,-0.0141027567,0.0034213038,0.0030882853],[-0.0015718735,0.0045786488,0.00194167,0.0046572217,-0.006955347,0.0002561248,0.0015417584,0.0071364598,-0.0050408875,-0.0004734284,-0.0025441001,0.0049074358],[-0.0017697555,0.0008199525,-0.0002830375,-0.000668213,-0.0039228859,-0.0089150605,-0.003589323,-0.0048319389,-0.0071256954,-0.0051978696,0.0021687019,-0.0067260734],[-0.0000698988,-0.0010325739,-0.0003710514,-0.0101717622,0.006131216,0.0054916573,0.0030715032,-0.000158363,-0.0025063237,-0.0045400069,-0.0087289434,-0.0059468612],[0.0017438742,-0.0010641804,-0.0002348352,-0.0072993779,-0.0008354924,-0.0018712192,-0.0001750015,0.0015937704,0.0012469542,-0.0005481173,0.0009786123,0.0044909418],[0.0023149688,0.0016218024,0.0002641124,0.0015409334,0.0056211344,0.0076737614,-0.0128549813,-0.0180633093,-0.0153684517,-0.0103802832,0.0046558384,0.0102913126],[0.0028061686,0.0033704578,-0.0006023925,-0.0013249058,0.0018074903,0.0008677615,0.0031149556,0.0047969039,-0.0076617487,-0.0088288549,-0.0097325173,-0.0027975086],[0.0035654119,0.0056774695,-0.0038202164,0.0006570201,0.0060441544,-0.0013728473,0.0043408969,0.0124222887,0.0048307562,0.0033191407,-0.0021547304,-0.0025200686]],[[0.0038196249,0.0045190153,0.005537757,-0.0026205905,-0.0028590221,-0.0126242953,-0.0028904382,0.0186912283,0.0082789789,-0.0060505028,-0.0142646566,-0.0065771967],[0.0037859453,0.0052477871,0.0072015924,0.0008922907,0.0035481172,0.0013607399,-0.0031699293,-0.0023577531,-0.0070450154,-0.0119190967,-0.0094067158,-0.0068851294],[0.0035288862,0.0042285213,0.0053524842,0.0006568005,0.0108237671,0.0038049087,0.0037769374,-0.0020114091,0.0015783253,-0.0048525012,0.0014358939,0.0007752023],[0.0018374569,0.0038192806,0.00465371,-0.0061665539,0.0056239355,0.0004747605,0.0002249495,-0.0053919666,0.0032484496,-0.0016534185,-0.0063083241,-0.0010945769],[0.0040024817,0.0049418627,0.0053036608,-0.0022447032,0.0099421344,-0.0033514299,-0.0011589717,-0.0015344246,-0.0055544077,-0.0017901027,-0.0007237237,0.0028417354],[0.0052657978,0.0017353497,0.0045092301,0.001130379,0.0046877307,-0.0060938124,0.0074929843,-0.002683808,0.001430376,0.0003062797,0.0023103639,0.0062442668],[0.0033418654,0.0005452942,0.0044363861,0.0013365537,-0.0068618685,-0.0093956622,0.0062885373,0.0083535497,0.0022816303,0.0002682193,-0.0030700208,-0.0015454709],[0.0001119719,0.0000519577,0.0071653205,0.0050304657,-0.0014402833,-0.0045267855,0.0060203529,0.0021948979,0.0011055796,0.0005489851,0.0000199261,-0.0024308043],[-0.0004589717,-0.0009372194,0.0080974772,0.0024635781,-0.0012866359,0.0008561287,-0.0006954057,0.0096620726,0.0039704936,-0.0020935042,-0.0031388986,0.001093957],[-0.0026988497,-0.0045413344,0.0059257957,0.0062053731,-0.0050103561,-0.0006177983,0.0045573808,-0.0027595243,-0.0065129026,-0.0016401262,-0.0025342129,-0.0056699907]],[[0.0040007822,0.0014353999,-0.0036314314,-0.0051181088,0.0012340803,0.0008332308,0.0035394617,-0.0077795129,-0.0060598054,0.0179061397,0.0138216374,0.000278622],[0.0044121959,-0.000638506,-0.0038429229,-0.0016154704,0.0001607343,-0.0068837087,-0.0040777271,0.0015015017,0.0078704613,0.0086894533,-0.0008287371,-0.0010635977],[0.0033430893,0.0023026275,-0.0013598415,-0.0004087003,-0.0023703973,-0.0093789883,-0.0050364187,-0.0048524961,0.0022903666,0.0083420534,-0.0006026905,-0.0051694852],[0.0031367562,0.0033360132,0.0011804369,-0.0010810776,0.0021120909,-0.0019571137,-0.0008896434,-0.0014613282,0.0040866767,-0.009093979,0.002020404,-0.005349563],[0.0022238333,0.0037595643,0.0012254758,0.0041258073,0.0073570266,0.0133313985,0.0100097788,0.0003197162,-0.0022057635,-0.0042539953,-0.0068552202,0.0005422085],[-0.001886736,0.0023441953,0.0019030501,0.0053658113,0.0051359373,0.0132427669,-0.0009181707,-0.0025553304,0.004484898,-0.0031253962,-0.0009173178,-0.0012305615],[-0.0023343701,0.0016453381,0.001730092,0.0020580603,0.0062310085,0.0094383304,0.0016253888,-0.0023617242,-0.0010311151,0.0032642962,0.0110621846,0.0084554766],[-0.0015828335,-0.0011925615,0.0012842227,-0.0003558275,0.0040352959,-0.0019305203,0.0059494072,0.0128038692,0.0062550502,0.0027906656,-0.004180131,-0.0064267652],[-0.0019639933,0.0026711859,0.0019238198,0.0044890284,0.0066653105,-0.0037199053,-0.001066677,-0.0024693144,0.0079205836,0.0152031852,-0.0004537346,0.0067224006],[-0.0000631967,0.0131826804,0.0053614948,0.0038405131,0.0038174431,0.0039077951,0.003018903,-0.0017360889,-0.000514156,-0.00208465,0.00408481,0.0030488162]]],\"magnitude\":[[[0.0078839583,0.0058843265,0.0128345365,0.0045744725,0.0078851689,0.0061613907,0.004681924,0.003801999,0.0048063203,0.0140591546,0.0072453363,0.0038917003],[0.0058125896,0.0072874872,0.0105438779,0.0048938734,0.0143170657,0.0104858909,0.0035865988,0.0025933856,0.0057078894,0.0084048727,0.0052136482,0.003143705],[0.0042534608,0.0068899233,0.0104986266,0.0042928331,0.0099092915,0.0037598715,0.0048683969,0.0090971103,0.0090661271,0.0138117881,0.0079985123,0.0075810864],[0.0032647071,0.0066272171,0.0101773889,0.0047731379,0.0117245636,0.0039897395,0.0051385048,0.010121871,0.0063122001,0.0059235094,0.0035878186,0.0021620029],[0.0032687179,0.0080081387,0.0078624237,0.0039885891,0.0082666206,0.0045864928,0.0029925113,0.0029386639,0.0032060105,0.0125704784,0.0122324431,0.005424341],[0.0015166923,0.0059842443,0.0054597694,0.0039720261,0.0059740785,0.00582172,0.0038140003,0.0055634022,0.002326222,0.0073610101,0.00373533,0.005878284],[0.0014934668,0.0046087463,0.0077923253,0.0045228651,0.0029060712,0.003094689,0.0028609206,0.0049593612,0.0049520061,0.0029248972,0.0022023221,0.0037556687],[0.0005510324,0.0023032334,0.0113627998,0.0011363945,0.0021163386,0.0034534907,0.0046803249,0.0027945272,0.0033714194,0.0040189266,0.003771062,0.0034151948],[0.0020807006,0.0039882913,0.0144124512,0.0013568693,0.0021737277,0.0033166304,0.0015092133,0.0068022117,0.0032926431,0.0034761585,0.0031766278,0.0027579503],[0.0057555841,0.0110127435,0.0147446622,0.006413246,0.0023837986,0.008609082,0.0022291963,0.0087016149,0.0040376424,0.0084292479,0.0043449991,0.0047368507]],[[0.004283467,0.0060895511,0.0148704069,0.0111544096,0.0055631872,0.0061543906,0.0168964788,0.015572523,0.0072777718,0.0238225537,0.0109587663,0.0087760132],[0.0057492264,0.0041312841,0.0078502407,0.0063726648,0.0131243176,0.008405701,0.0185349883,0.0094811023,0.0082102582,0.0044882365,0.008340306,0.0062677458],[0.0044722318,0.0067521709,0.008066719,0.0098956598,0.0114893706,0.0093658995,0.0128939472,0.0071470867,0.0088093943,0.0104105009,0.0093251301,0.0078621602],[0.002697479,0.0076714995,0.0068080555,0.0079761471,0.0112918153,0.0078092016,0.0076264121,0.0105096574,0.0125630379,0.0100085405,0.0110077872,0.0062219398],[0.0036567972,0.0049649334,0.0036520225,0.0031292642,0.0089838092,0.0108006592,0.0076834971,0.0081180211,0.0136550096,0.0089053131,0.0053393814,0.0079866144],[0.0037973183,0.001707201,0.0018934002,0.0030658726,0.0114204528,0.011863082,0.0111057417,0.0093816946,0.0082036952,0.0068215489,0.0111463994,0.0084255452],[0.0026344041,0.0028021517,0.0019460791,0.0044394512,0.0033309853,0.0091578244,0.0130410455,0.0079856739,0.0085535928,0.0077785634,0.0112724505,0.0144758557],[0.0015102132,0.0032458679,0.0024122847,0.005612213,0.0087275793,0.0060183261,0.006610085,0.0072192531,0.0119964871,0.008228948,0.0096306724,0.0118290826],[0.0026573731,0.0043005756,0.0047233726,0.0066267001,0.0102104597,0.0049950956,0.0019056638,0.0068357729,0.0092634994,0.0094265273,0.0143838989,0.0083885428],[0.0060969474,0.0059450232,0.0076905676,0.0086801291,0.0111218161,0.010452447,0.0130197214,0.0054214697,0.0097514533,0.0080504306,0.0106775771,0.0056952069]],[[0.0077353524,0.0065812043,0.0051957367,0.0030017005,0.0075658616,0.0101755045,0.0120851805,0.0074077218,0.0048266519,0.0070062896,0.0082551771,0.0132745422],[0.0072698099,0.0128387793,0.0040827905,0.0056383388,0.010955974,0.0192147851,0.0137163571,0.0121145687,0.0074994441,0.0094891117,0.0106751998,0.0087039385],[0.0064563153,0.0088591177,0.0047565846,0.0073573613,0.0101949865,0.0063518318,0.0108093301,0.0064981884,0.0077638984,0.0150256988,0.008692027,0.0051904344],[0.0045518741,0.0073150349,0.002583858,0.0061113976,0.0106767687,0.0028488275,0.0041638437,0.016648107,0.014867307,0.0043904534,0.0055526298,0.0057675358],[0.0050366168,0.0049687156,0.0020012789,0.0044610491,0.0129770333,0.0122854888,0.0066136637,0.0050036826,0.0078861743,0.0076839695,0.0070448109,0.0081093232],[0.0046303176,0.0035446761,0.0010206213,0.0106767557,0.0079573162,0.0101764131,0.0062998974,0.0019054741,0.0062804571,0.0055159496,0.0103680036,0.0066978012],[0.0035259232,0.0034846888,0.0019400954,0.0082740119,0.0051435828,0.0055343833,0.0056102823,0.0095184518,0.0036783363,0.0060162823,0.0040027826,0.0064612534],[0.003640515,0.0031377354,0.002345486,0.002342403,0.0080304798,0.0109759203,0.0148406693,0.0187584809,0.0168717534,0.0104122333,0.0054281815,0.0109082104],[0.0051038871,0.0051702107,0.0044222436,0.0031229983,0.0046268817,0.0070099006,0.0052149236,0.0061686527,0.0097200139,0.0121434926,0.0111644068,0.0042756648],[0.0068005458,0.0069346398,0.0059194113,0.0036261702,0.0064437443,0.003960608,0.0096228399,0.0126011926,0.0063724057,0.0063049536,0.0039361562,0.0057361834]],[[0.0065915081,0.0047610178,0.0072984959,0.0053197847,0.0085457818,0.0133882539,0.00837226,0.0230680453,0.0108780041,0.0081571402,0.0179370374,0.0110296266],[0.0083670115,0.0057099466,0.0078717721,0.0059830125,0.0124740894,0.0058697766,0.0118024033,0.0076367383,0.0088790277,0.0128402358,0.0110151573,0.0094207203],[0.0087628613,0.0046014634,0.0055738051,0.0024990407,0.01186534,0.0043952321,0.0098976746,0.0085325025,0.0067438366,0.0098615697,0.0131078097,0.0086219339],[0.0074642146,0.0045893965,0.0049769057,0.0071553057,0.0079332857,0.0050427409,0.0059498849,0.0066868927,0.0087056385,0.0062755155,0.010426461,0.0042308573],[0.0062302156,0.0050890544,0.0053778389,0.0048074777,0.0125889854,0.0058526399,0.0054225347,0.0064577766,0.0088078701,0.0050539427,0.0072998812,0.0057067228],[0.0065892297,0.0024685578,0.0045466604,0.004475919,0.0104673165,0.0114309997,0.0106730082,0.0066341184,0.0028804191,0.0040214091,0.0064119928,0.0071129722],[0.004157723,0.0025987771,0.0061443544,0.0066303316,0.0075868542,0.0107565519,0.0068509388,0.0128777574,0.0043091309,0.0039873335,0.0071989891,0.0093892065],[0.0033129659,0.0018028319,0.0080197573,0.005441702,0.0070838911,0.0064680838,0.0087588976,0.0045481076,0.0047923425,0.0033666229,0.0040051941,0.0060003986],[0.0023544466,0.0025492049,0.0082817492,0.0032676484,0.006464199,0.0050300759,0.0051796772,0.0119565556,0.0054220482,0.0074488527,0.0085752871,0.0049678238],[0.0037735208,0.00652306,0.0060431263,0.0065790916,0.0079514428,0.0059803029,0.0057307495,0.0101899869,0.008300348,0.0077536487,0.0138672686,0.0075007399]],[[0.0070483726,0.0049861143,0.0050882387,0.0062308982,0.0032933135,0.0041105399,0.0081981012,0.0096394171,0.006629314,0.0195065303,0.0159255064,0.0083217379],[0.0065094938,0.0047166556,0.0054794915,0.0028238892,0.0048455957,0.0082140263,0.0081388854,0.0036650274,0.0158519687,0.0117050249,0.0075599818,0.0060791664],[0.0061486988,0.0040181856,0.0033878416,0.0029013088,0.0057379992,0.0097187574,0.008162387,0.0078468432,0.0070142735,0.0094120502,0.0041987735,0.0055585789],[0.0068527961,0.0044291554,0.0016707843,0.0029017474,0.0036457274,0.0067444083,0.0063900883,0.0064446228,0.0056135933,0.012365073,0.0069482154,0.0083547507],[0.0079567028,0.0043877677,0.0016358605,0.0064996113,0.0111414937,0.0141501428,0.0102048379,0.0032350807,0.0038317789,0.0060228991,0.0073477659,0.0053546356],[0.0063330945,0.0028367153,0.0022073026,0.0057191727,0.0072147833,0.0134807404,0.009334972,0.0061436273,0.0071066463,0.0065003722,0.0068493953,0.0029757829],[0.0034696543,0.0028634096,0.0023572988,0.0030352153,0.0063714434,0.0094988087,0.004453618,0.0044849522,0.0090759341,0.0060109986,0.0116714259,0.0085572762],[0.0023878088,0.0030461206,0.00220201,0.0011069123,0.0059883809,0.0062944379,0.0060935148,0.0128458765,0.0072305595,0.0041960273,0.0077149481,0.0103428103],[0.0033116267,0.005740217,0.0034988002,0.0067025041,0.0095455781,0.0127964461,0.0026071968,0.0037946755,0.0079718451,0.0152031852,0.0047311073,0.0077361137],[0.0029900648,0.0142229279,0.0058357353,0.0038405131,0.0049629878,0.004760316,0.004292768,0.0039041402,0.0026083174,0.0029729858,0.0086202877,0.0050936771]]],\"nonzero\":[[[10,10,15,9,10,9,6,4,7,13,8,5],[8,8,15,9,10,13,7,2,8,11,9,4],[8,8,15,9,9,8,12,13,14,11,10,11],[5,10,14,9,13,9,8,10,9,8,6,4],[5,7,13,10,10,3,4,5,7,12,12,8],[2,6,10,8,6,7,4,9,6,11,5,7],[1,6,13,5,3,5,6,6,9,3,3,6],[0,5,16,3,1,5,11,3,2,6,8,8],[4,9,17,2,3,4,4,12,3,7,3,4],[11,12,15,10,4,12,2,14,7,11,6,7]],[[5,8,13,12,9,12,13,14,8,16,12,11],[8,8,8,13,14,11,15,15,10,7,7,8],[6,12,9,9,12,11,12,11,12,13,14,11],[3,13,8,10,12,10,9,11,12,9,15,8],[7,6,5,3,12,9,10,11,15,14,7,11],[7,3,4,6,13,16,11,13,11,8,12,11],[5,7,1,9,6,11,15,9,11,12,11,14],[3,7,5,9,13,9,6,11,12,8,14,12],[7,7,9,10,12,6,7,6,11,9,16,12],[10,7,13,13,13,14,11,7,11,7,11,10]],[[11,11,6,7,7,11,10,9,7,9,11,13],[12,15,6,7,10,16,15,15,12,11,10,10],[7,10,6,10,10,8,12,7,7,14,11,8],[8,9,4,8,10,7,7,14,16,8,9,7],[10,5,4,6,12,14,9,6,11,7,9,12],[8,4,3,13,11,11,9,4,8,10,11,10],[5,7,2,11,7,9,9,8,7,6,6,10],[8,6,4,5,16,13,14,14,13,13,8,13],[10,10,8,6,6,8,11,11,11,11,9,4],[7,11,12,9,11,7,14,13,8,6,4,11]],[[12,10,11,7,13,12,12,18,14,8,13,8],[12,9,14,9,13,9,10,7,10,8,11,11],[14,9,11,5,11,7,14,11,11,9,11,12],[9,9,10,10,9,11,7,10,11,4,13,7],[7,8,9,9,15,6,12,11,9,7,10,10],[8,5,8,8,13,13,12,7,4,8,9,6],[7,4,11,12,10,11,9,14,10,7,11,12],[9,2,14,11,9,10,10,7,5,8,4,6],[6,5,13,4,8,7,8,12,7,9,13,7],[6,6,11,8,8,9,10,11,11,9,12,8]],[[9,9,9,8,5,5,8,10,8,17,16,9],[9,6,6,5,4,10,11,5,13,14,10,9],[13,8,5,5,8,12,12,11,10,14,6,8],[11,8,3,4,5,11,11,7,8,13,9,9],[11,7,4,9,14,16,11,6,6,10,9,8],[12,8,7,11,9,15,6,8,9,9,10,7],[5,5,4,6,11,13,6,4,15,6,13,5],[4,5,3,3,8,8,7,17,10,10,11,9],[7,11,5,8,11,13,6,6,12,17,9,8],[8,13,10,9,10,6,7,7,3,4,11,11]]]},{\"rat\":\"Superchris\",\"target\":\"odor\",\"n_channels\":21,\"signed\":[[[-0.0007174485,-0.0029797087,-0.0038602694,-0.0070169788,-0.0018846287,-0.0006535083,0.0015416723,0.0008520819,-0.0018703274,0.0031727539,0.0093269533,0.0109793103],[0.0007380774,-0.0032132188,-0.004402399,-0.0024403883,0.0008751618,0.0004942365,-0.0028845867,0.00152639,-0.0005345668,-0.0013598224,0.0034606369,0.0079962419],[0.0016479655,-0.0016379491,-0.0044693188,-0.0039089489,0.0017546345,-0.0015638948,0.0025079455,0.0017791429,0.0000652368,0.000645718,0.001263221,0.0050239281],[0.0011617326,-0.0057210412,-0.0093976381,-0.0268742241,0.0011575734,-0.0058430008,-0.0017503223,-0.0004211219,-0.0007972228,-0.002914198,0.0049718055,0.0112217579],[-0.0005740364,-0.0063582171,-0.0155160528,-0.0118877494,0.0047671514,-0.0002599526,0.0012232297,0.0032055586,-0.0006320424,0.0010616594,-0.0001482946,0.0013012028],[-0.0015019235,-0.0038696923,-0.0165039409,-0.0047233464,0.0002695766,-0.0009553448,0.0011367265,0.0026137908,-0.0003492003,-0.0061398744,-0.0015411013,-0.0023123141],[-0.0028430338,-0.0050013686,-0.0120762788,-0.0053415956,0.0073747177,0.0063144004,-0.0056191868,-0.0120854504,-0.006170826,-0.0031358457,-0.0027703443,-0.0011916773],[-0.0037084289,-0.0045664841,-0.0122220823,-0.0028689018,0.008038049,0.0084601062,-0.0006740322,-0.0030029412,-0.001855441,-0.0045259789,-0.016361673,-0.0053882482],[-0.007155182,-0.0046428818,-0.011867865,-0.0025331432,0.0008890089,0.0037496708,0.0008869053,-0.001296419,-0.0038751096,-0.0050140308,0.0000849091,0.0028982325],[-0.0065205601,-0.0108092636,-0.012069684,-0.0044059725,0.0028100702,0.0035397644,-0.0077430685,-0.0085359631,0.000450286,0.0008085228,0.0031737615,0.0112182236]],[[-0.0016359925,0.005936907,0.005580095,0.0013065293,-0.0028800005,0.006158413,-0.0128573868,-0.0005808952,0.0136066826,0.002000625,0.0006568345,0.0006229543],[-0.0014057245,0.0080378824,0.0049337231,0.0001352417,-0.0051249305,0.001094074,0.0064589342,0.0017074375,-0.0031653412,0.0015415644,-0.0035674919,-0.0113734988],[-0.0019730201,0.0045399089,0.0035621702,-0.0055051852,0.0145858865,0.0057145958,0.004147734,-0.0008369489,-0.003629327,-0.0051273494,-0.000772155,0.0017028439],[-0.0026070441,0.0007915966,0.004476613,-0.0015103356,-0.0070250786,-0.0018389768,-0.0044915185,-0.0040116173,0.0033707376,0.0071547103,0.0120502137,-0.0001102882],[-0.0022477391,-0.0006289755,0.0053177018,-0.0023751498,-0.0003894767,-0.0035663768,-0.0025894301,-0.0055212415,-0.0044961622,0.0000821609,-0.000339848,-0.0033661585],[-0.000167703,-0.0077940552,0.0057783484,0.0032142315,0.0005614843,-0.003863717,-0.0098090579,-0.0157771861,-0.008962979,0.000723793,-0.0027529714,-0.0043984317],[0.0032331705,-0.0052643773,0.0052812804,-0.0002899346,0.0013758922,-0.0004518789,-0.0119046662,-0.0046063867,-0.0029750984,-0.0009020834,0.0023193034,-0.0030097808],[0.0051235945,-0.0066990159,0.0036357229,-0.0191439204,0.0027021201,0.0066946842,-0.0011286465,0.0010520207,0.0022098051,-0.0043041223,-0.0026486541,-0.0092872497],[0.0089861579,-0.0049211732,0.0022057426,-0.0152089771,0.0031343897,0.0004537639,0.002190164,0.0057297308,0.0136752986,0.0085091388,-0.0011843856,0.0034066192],[0.0139566209,0.0015111509,0.0054625486,-0.0044178733,-0.0057598671,-0.0034475379,-0.000321517,-0.0026947723,-0.0026956097,-0.0025142411,-0.0003387336,0.0041951829]],[[0.0001857806,0.009256782,0.0031316668,-0.00022396,0.0062775464,0.0031981516,0.003251577,-0.0042444949,-0.012337254,-0.0043579206,0.0004625126,-0.0018610916],[0.0005850256,0.0023130369,-0.0011822594,-0.0006570142,0.0029871667,0.0023828754,0.0010331136,0.0009342978,-0.0041162693,-0.0107805314,-0.0177701419,-0.0083978545],[0.0010189834,0.0025394971,-0.001854589,-0.0009679109,-0.0135662184,-0.0074833236,-0.0101183755,0.0035737321,0.0004682753,0.0065416865,0.0001461329,0.0051159867],[0.0004506081,0.0031870924,0.0029830971,0.0080353836,-0.0060312367,0.0007542543,-0.0043576377,-0.0046703415,-0.0007919305,-0.0051237052,-0.0127384803,-0.005075312],[0.0014518553,-0.000672696,0.0066336696,0.0032072549,0.0001460659,0.0043857202,0.0028786411,0.0021736238,0.0104647904,0.0063699382,0.0084095426,0.0104998364],[-0.0010482002,0.0004929544,0.0039668264,0.0052172946,-0.0000276884,-0.0055096995,0.0021281748,0.0033552895,0.0062690182,0.0036658753,0.0059040944,-0.0136140061],[-0.0024372658,0.0013094101,0.0036265948,0.0079513464,-0.005821429,-0.014076895,0.0000905278,-0.0006918113,0.0126729754,0.0093219603,-0.0030661591,-0.0058727175],[-0.0012876222,0.0005358651,0.00406144,0.0134815563,-0.0005289981,-0.0098456925,0.0023995492,0.0034849966,0.0058837282,0.0130296071,0.0064381586,0.0032602668],[-0.0005818241,-0.0033893161,0.0031012148,0.0070374464,-0.0053355699,-0.0040667868,-0.0068511276,-0.0063811911,-0.0005984928,0.0001364812,-0.0020833417,-0.0110244803],[-0.0030276735,-0.0057643939,-0.0011944039,0.0107886438,-0.005222516,0.0008374728,0.0009248357,-0.002928321,0.0012407925,0.0074356165,0.0073989825,-0.0077144713]],[[-0.0037545934,-0.0138395887,-0.0067988039,0.0033149467,-0.0029276274,0.0024214089,0.0097743174,0.0014309806,-0.0008605147,-0.0022706218,-0.0066390912,-0.0006969999],[-0.0046479427,-0.008141322,-0.0016010489,-0.0011767675,0.0064078301,-0.0064190966,-0.0099850026,-0.0065219723,-0.0005487584,0.003886659,0.0090114485,0.0036648779],[-0.0067268851,-0.0007750469,0.0014202702,0.0007177808,0.0016437743,0.0009959085,-0.0029149105,-0.0001339388,0.0047451922,-0.0043304813,0.0006192359,-0.0029835556],[-0.0034939804,0.0064116989,0.0017455146,0.0079750493,0.0209583885,0.0107055573,0.0146301898,0.0032386369,-0.0039602268,0.0005772852,-0.0031481699,-0.0061873324],[-0.0012061659,0.0067501454,0.0016111369,0.0078875652,0.0028176416,0.0003267147,-0.0041601772,-0.0008551962,-0.0047680975,-0.0070698409,-0.0048395196,-0.001551559],[0.0012781494,0.0116798285,0.0037081614,0.0018733886,0.0042589825,0.0114513577,0.0038981688,0.0062168125,0.0009964416,0.0028732562,-0.0031251367,0.0188611999],[0.0042887385,0.008452418,0.0026253836,-0.0011905146,-0.0022559124,0.0009726011,0.0025873148,0.0112311481,0.0058749282,0.0053069131,0.0051409185,-0.0012581435],[0.0037544129,0.0070654705,0.0015888493,0.0014827455,-0.0130196923,-0.0037877572,-0.0005730808,0.0013541579,-0.005090476,-0.0071004668,-0.0012733042,0.0045375404],[0.0010789115,0.0098389241,0.0008070575,0.0037300984,0.0011763236,0.0015609471,0.0035668445,0.0041278824,0.001135005,-0.0020984621,-0.0015920209,0.0011498208],[0.0014663489,0.0124752805,0.0020810111,-0.0021575424,0.0092909588,0.0010142925,0.0074596905,0.0131723371,-0.0056917119,-0.0077181146,-0.0123144511,-0.0083249901]],[[0.0059222536,0.0016256082,0.0019473114,0.0026194629,0.0014147102,-0.0111244652,-0.0017101798,0.0025423277,0.0014614134,0.0014551635,-0.0038072093,-0.0090441732],[0.004730564,0.0010036214,0.0022519841,0.0041389281,-0.0051452283,0.0024479106,0.0053775415,0.0023538467,0.0083649356,0.0067121304,0.0088655478,0.0081102336],[0.0060329563,-0.0046664101,0.0013414674,0.0096642641,-0.0044180769,0.0023367141,0.0063776068,-0.0043819873,-0.0016493772,0.0022704262,-0.0012564345,-0.0088592028],[0.004488684,-0.0046693464,0.0001924136,0.0123741266,-0.0090596468,-0.0037778342,-0.0040307111,0.0058644438,0.0021786423,0.0003059076,-0.0011353688,0.0001511746],[0.002576086,0.0009097431,0.0019535446,0.0031680792,-0.0073413821,-0.0008861056,0.0026477364,0.0009972552,-0.0005684885,-0.0004439178,-0.0030818805,-0.006883322],[0.0014396774,-0.0005090355,0.0030506047,-0.0055815684,-0.0050623548,-0.0011225965,0.0026459873,0.0035912935,0.0020467195,-0.0011230501,0.0015151148,0.0014635525],[-0.0022416097,0.0005039178,0.0005430202,-0.0011293017,-0.0006732684,0.0072417725,0.0148460106,0.0061525003,-0.0094019795,-0.0105909443,-0.0016237185,0.0113323193],[-0.0038819561,0.0036641643,0.0029360703,0.0070485207,0.0028085218,-0.0015213413,-0.0000237896,-0.0028882341,-0.0011476165,0.0029009606,0.0138454726,0.0068776906],[-0.0023280632,0.003114447,0.00575385,0.0069745754,0.0001358478,-0.001697595,0.0002072139,-0.0021800032,-0.0103367012,-0.001533127,0.0047748392,0.003569808],[-0.0058747363,0.0025872262,0.0057205281,0.0001927445,-0.0011186456,-0.001943992,-0.0003199409,0.0009867194,0.006696243,0.0019882163,0.0020804407,0.000626055]]],\"magnitude\":[[[0.0017317323,0.0054899202,0.0117526421,0.007283792,0.0064862036,0.0037287018,0.0025097741,0.0015143682,0.0075380282,0.0035043812,0.0095038672,0.0111480137],[0.0017455097,0.0036150662,0.0119025055,0.0025567626,0.0050162313,0.0010701328,0.004733362,0.0021641244,0.0024151534,0.0029969207,0.005361501,0.0085751878],[0.0026089685,0.003597977,0.0142811483,0.0043132533,0.0028463728,0.0030340859,0.0039075484,0.0033345932,0.0029792971,0.0036162462,0.0094469799,0.0116777804],[0.0028166479,0.0061225389,0.0130579638,0.0268742241,0.0024971333,0.0058820983,0.0035169407,0.0009116194,0.0030582848,0.0043933321,0.0058258485,0.011508397],[0.0027891629,0.0064824816,0.0155741277,0.0120203341,0.0052857126,0.0030714352,0.0019003393,0.0044306706,0.0024322108,0.0017446514,0.0025564887,0.0034308805],[0.0023299725,0.0044263342,0.0166244635,0.006689716,0.0011049936,0.0019533895,0.0019221894,0.0027045754,0.0014871922,0.0063620485,0.0021845443,0.0056740412],[0.0033894698,0.0072712032,0.0123619001,0.0057606756,0.0075267916,0.0071681374,0.008140181,0.0120854504,0.0062372904,0.004020378,0.004320161,0.0027229805],[0.004585374,0.0093760435,0.0127471704,0.0085767486,0.0081921413,0.0102174336,0.0040005723,0.0033115456,0.0026880313,0.0051073173,0.0165605833,0.0064688927],[0.0073057378,0.0086908411,0.0131671533,0.0073671275,0.0025466639,0.0037496708,0.0034618794,0.0020113897,0.0056699832,0.0053668594,0.001947334,0.0049160803],[0.0068408905,0.0140992567,0.014460379,0.0064220322,0.0042047522,0.0052529724,0.0077430685,0.0087814468,0.0057094598,0.0026646209,0.0048667961,0.0124189311]],[[0.0029031252,0.006983151,0.0063559714,0.00181896,0.0029498761,0.0069521895,0.0131440968,0.0021489122,0.0136759249,0.0029982655,0.0028884832,0.0032358761],[0.0038462817,0.0081289925,0.0052058705,0.0015126411,0.0067609425,0.0019904864,0.0066479338,0.0030849584,0.0037922215,0.0033785407,0.0065661709,0.0124810219],[0.0045976871,0.0045399089,0.0043405574,0.0076157217,0.0152189659,0.0113586008,0.0148254502,0.0039315816,0.0039636915,0.0068197148,0.003970559,0.006187397],[0.0033160536,0.0016294503,0.0055144705,0.0051852045,0.009592989,0.0025948937,0.0052297027,0.0048113546,0.0040231337,0.0075338803,0.0131583514,0.0047336208],[0.0029742738,0.0029119101,0.0061498122,0.0087646857,0.0089220958,0.0086854857,0.0091302339,0.0078990492,0.0051904095,0.0027566244,0.0023057259,0.004571387],[0.0057386741,0.0082035343,0.0063673275,0.0035690241,0.0012971489,0.0043908678,0.0099724198,0.0157771861,0.0098174548,0.0027359149,0.0028989021,0.0126863311],[0.0086017751,0.0091157867,0.0069540095,0.0026277244,0.0054555854,0.0030074347,0.0126346548,0.0046073972,0.003443055,0.0020984119,0.0031585854,0.0046673965],[0.0068502899,0.0163514659,0.0067298317,0.0263419239,0.0085807242,0.0069854235,0.0019150031,0.0047491605,0.0073775446,0.0050193457,0.0059030327,0.0101386237],[0.009489462,0.018000878,0.009012416,0.0169055001,0.00314461,0.0016947807,0.002190164,0.00749298,0.0164880725,0.0118216278,0.005490729,0.0048610368],[0.0146663069,0.016530679,0.010945632,0.0066155638,0.0069340332,0.0049880985,0.0030019545,0.0035741956,0.0045253484,0.0039133081,0.006949859,0.0079412861]],[[0.0026484235,0.0115114894,0.0043831934,0.0015949019,0.007252143,0.0048212803,0.0043923033,0.0043540573,0.0125114677,0.0064178396,0.0060836975,0.0041874229],[0.0016536786,0.0045252136,0.0052244845,0.0011527017,0.004088306,0.0028517125,0.0036918593,0.0033850811,0.0044558656,0.0118918974,0.0182162417,0.0103506225],[0.0036957982,0.0029011985,0.0082583545,0.0027153291,0.014145856,0.0096252406,0.0112658627,0.0065968448,0.0047673471,0.0130104708,0.0088676604,0.0052812969],[0.0030697603,0.0044121151,0.0056611223,0.0087616147,0.0065028936,0.0030002551,0.0056095287,0.0050227808,0.0077336201,0.0068567609,0.0131357443,0.0077243085],[0.0021419541,0.0021653901,0.0068659692,0.0052038693,0.0106047179,0.0062793445,0.0035557506,0.0047754802,0.0104932976,0.0064352029,0.0106597707,0.0107716056],[0.0032180144,0.0036603084,0.004534728,0.0054876643,0.0023905617,0.0062966387,0.0029571809,0.0045944622,0.0091870955,0.0050676785,0.0069192333,0.0145998269],[0.0053602675,0.00537479,0.0053332518,0.0079513464,0.0060463405,0.014076895,0.0031555226,0.0018970613,0.0126729754,0.0100999767,0.0057375962,0.0086882407],[0.0044019309,0.0029769914,0.004602695,0.0134933425,0.0093022668,0.0103497744,0.0027495296,0.00491294,0.0077583687,0.013418232,0.0133589155,0.0084364004],[0.0028111661,0.005970246,0.0046468945,0.0071377806,0.0058176306,0.0047587015,0.0076684338,0.0063811911,0.0074934957,0.0052312098,0.0043940457,0.0136028462],[0.0054109952,0.0086206042,0.0073707074,0.0107886438,0.01065333,0.0043454969,0.008003941,0.0033647923,0.005402531,0.0091633109,0.0169078734,0.0120583504]],[[0.0086497734,0.0139340155,0.0078820797,0.0038035833,0.0029440427,0.0031955638,0.0097743174,0.0032912452,0.0051006773,0.0044657195,0.0071989752,0.0056672248],[0.0132943659,0.008252085,0.0038865604,0.0025933681,0.0066229454,0.0066194843,0.0100001838,0.0067849923,0.0034589454,0.0088597551,0.0128052939,0.0157763287],[0.0127256406,0.0042815329,0.0025652186,0.0015291356,0.0044212396,0.0055916291,0.0110525456,0.0021805449,0.0086901294,0.0059640327,0.005611035,0.0061580483],[0.006581407,0.0067575924,0.0043540086,0.0080969958,0.0211764589,0.0109242515,0.0146301898,0.0036066739,0.0050715617,0.0065276407,0.004379727,0.0075613887],[0.003616911,0.0067501454,0.0053281768,0.0081101363,0.0052491066,0.0023674344,0.0049418699,0.0018018018,0.0055652617,0.0073210859,0.010046603,0.0145142636],[0.0029659748,0.0116798285,0.0042451508,0.0066072379,0.0071418138,0.011670616,0.0038981688,0.0062168125,0.0060536185,0.0047156792,0.003224781,0.0189058203],[0.0070733034,0.0085212067,0.0027182654,0.0022535061,0.0034108953,0.0024181532,0.0044077213,0.0138847821,0.0131198588,0.007503238,0.0057741595,0.0063993619],[0.0048176532,0.007612257,0.0028160661,0.0061744169,0.013799341,0.0042655382,0.0038607683,0.006603953,0.0055568953,0.0079399325,0.0024196971,0.0064263172],[0.0032647103,0.0109016062,0.00378576,0.0050661327,0.0019904724,0.0035694053,0.0035668445,0.0050103339,0.0114041032,0.0090894636,0.004365941,0.0033255259],[0.0037897994,0.0125747574,0.0051619013,0.0064996861,0.0143586283,0.0126802685,0.0075835448,0.0131723371,0.0073682999,0.0082540679,0.0155233608,0.0090962636]],[[0.0098378606,0.0026958179,0.002725077,0.0037604417,0.0044360429,0.0114552836,0.0041764394,0.0029251992,0.0090371314,0.003939154,0.0071715872,0.009269936],[0.0100398698,0.0018578854,0.0027163731,0.0041887684,0.0067212075,0.0026625118,0.0055849592,0.0029109903,0.0085553195,0.0072121021,0.0116373269,0.0086563243],[0.0079959556,0.0053188979,0.0025266085,0.0097376014,0.0084634145,0.003188637,0.0083809948,0.0054004757,0.0065176818,0.0082219824,0.0041221625,0.0101084838],[0.0048774281,0.0054617981,0.0019193408,0.0126792837,0.0099594618,0.0042515796,0.0068326516,0.0068322228,0.0046728594,0.0023187634,0.0030214248,0.0034514373],[0.0028756195,0.0032507713,0.0026996943,0.0061163857,0.0101552337,0.002092997,0.0036101435,0.0024059054,0.0028862742,0.0013497436,0.0039202637,0.0074406297],[0.0020531761,0.002089457,0.0033460406,0.007364828,0.0094888116,0.0041116953,0.0028862219,0.0036526854,0.005473061,0.0028646449,0.0039513492,0.0052983413],[0.0033678007,0.0033530836,0.0055157389,0.0036128268,0.0027098704,0.0079472014,0.0148460106,0.0073405264,0.0142670622,0.0108186464,0.0038961826,0.0113323193],[0.0049319645,0.0067175807,0.0037731654,0.0070603069,0.0052642635,0.0041257195,0.002197684,0.0031985878,0.0037778235,0.0053070031,0.0140925195,0.0077246129],[0.002815928,0.0088715807,0.00575385,0.0075135938,0.0025956863,0.0031983881,0.0038077864,0.0028949739,0.0104432741,0.0083734911,0.0053320595,0.0038399077],[0.0062206618,0.0037200114,0.0057205281,0.0024978768,0.0055728026,0.0022283447,0.001457844,0.0025185475,0.0076251501,0.0078368538,0.0050645041,0.0055867756]]],\"nonzero\":[[[3,8,17,11,14,13,5,2,13,7,17,17],[1,5,15,4,11,3,6,4,4,7,7,11],[3,4,15,11,7,4,7,13,4,6,18,17],[7,7,16,21,5,13,6,1,4,11,11,17],[7,6,13,18,9,11,1,14,3,1,8,8],[2,10,17,7,1,2,4,5,1,11,4,11],[6,9,17,8,11,10,18,18,19,9,13,6],[10,12,19,10,13,8,9,6,3,6,16,11],[14,12,17,7,7,10,5,2,8,6,5,7],[9,12,13,7,8,8,14,15,8,5,6,9]],[[7,14,8,3,4,8,15,5,19,6,6,5],[7,16,7,2,6,8,10,8,6,6,12,13],[7,10,7,11,18,16,17,8,8,10,5,7],[4,3,8,8,9,6,12,14,10,11,13,8],[5,6,6,12,13,9,13,18,9,6,4,7],[13,12,8,10,2,8,11,21,9,6,5,12],[12,12,10,4,12,4,6,6,4,3,5,9],[11,14,13,16,12,16,8,5,14,6,10,16],[15,16,11,11,9,3,4,13,16,13,11,9],[19,18,11,9,12,11,4,10,11,9,11,8]],[[5,13,6,4,12,8,8,10,14,12,7,10],[0,13,16,3,9,4,9,7,10,17,16,15],[6,5,14,7,9,11,16,17,8,16,13,6],[5,7,7,8,11,10,4,8,10,11,14,9],[4,2,13,11,15,10,5,10,16,9,13,10],[8,9,10,10,5,10,3,7,11,10,11,16],[6,12,11,16,11,17,7,2,20,16,8,10],[6,6,10,12,12,10,6,13,9,9,13,11],[4,7,7,9,11,14,13,16,10,5,10,15],[7,9,11,13,13,11,19,5,8,16,16,11]],[[17,19,13,4,4,9,18,4,8,5,15,8],[16,10,7,8,14,9,20,12,3,14,16,17],[18,6,3,4,8,8,19,3,16,11,8,9],[12,12,7,16,18,16,17,9,8,14,7,11],[5,12,8,10,9,6,10,4,11,10,9,15],[9,19,4,12,10,17,8,7,10,9,6,17],[14,11,4,2,3,2,6,19,17,13,12,8],[13,8,2,5,18,6,11,11,10,9,7,11],[8,10,4,7,3,9,3,12,15,11,7,6],[10,15,7,10,10,16,15,18,6,9,14,14]],[[18,5,5,10,11,15,6,6,11,10,6,14],[18,1,4,9,9,9,10,6,12,13,13,15],[16,15,3,17,11,6,13,10,14,10,8,10],[14,11,2,16,13,6,13,14,7,4,4,5],[5,3,3,9,14,7,6,3,4,2,5,8],[4,3,9,12,14,9,6,5,10,3,7,10],[5,8,12,8,5,15,21,10,15,17,9,10],[8,11,8,13,12,7,7,6,9,7,16,13],[6,13,10,8,7,8,6,10,12,12,9,9],[11,5,7,2,12,3,3,5,6,9,7,8]]]},{\"rat\":\"Superchris\",\"target\":\"position\",\"n_channels\":21,\"signed\":[[[-0.0005945624,-0.0008979604,-0.0071693426,-0.0064321561,-0.0007002669,0.0048215308,0.0009980218,0.0002707299,-0.0014450993,0.0013654363,0.003790983,0.0054652456],[-0.0016760714,0.0007568189,-0.0047329059,-0.0004634839,-0.0003195687,0.0000734464,0.0001140448,0.0002120611,0.0005048936,0.000445564,0.0024839113,0.0009489143],[0.0013517551,-0.0013698632,-0.0081737135,0,0.005662639,-0.0007641274,0.0020552741,0.0020292324,0.0009386167,-0.0013716378,0.0004802243,0.0061035962],[0.0002341528,-0.0037728757,-0.0081533321,-0.0257175912,-0.0006829786,-0.0022634008,-0.0020973647,-0.0005304515,0.0000527305,-0.0001927064,-0.0000046679,0.004404188],[-0.0004270711,-0.0007890256,-0.0170457564,-0.0061797668,-0.0001737773,0.0010611448,0.0014946721,0.0005806564,0,0.0000277169,0.0014676675,-0.0019266845],[0,-0.0015324947,-0.0156167286,-0.0016049554,0.0018470333,0.0001116319,0.0021113916,0.0022923555,0.0006644027,-0.0031769425,-0.0004024635,0.0017106913],[-0.0013122613,-0.0037973835,-0.0070571807,-0.0015424464,0.0020270995,0.0053206539,0.0006318485,-0.0074478838,-0.0014586739,-0.0019603173,0.0013788168,0.0014670257],[-0.0003606869,-0.0020560957,-0.004435902,0.0048411835,0.0056471422,0.0057487151,-0.0000793066,0.000453863,0.0002184739,-0.0029839049,-0.0113117266,-0.0003922655],[-0.0011465821,-0.0024654932,-0.0041307845,0.004005669,0.0005516261,0.0017360971,-0.0002880028,-0.0013660219,-0.0037967035,-0.0045609608,0.000437263,0.0041124382],[-0.000628487,-0.0090559135,-0.0112268601,-0.0065971213,0.0047347898,0.0002506258,-0.0013228486,-0.0029433026,0.0000080604,-0.0008710907,0.0001898655,0.0085327416]],[[-0.0005945624,-0.0008979604,0.0036673934,-0.0007150613,-0.001295336,0.0082598519,-0.005674819,0.0002707299,0.0086193089,0.0013654363,0.0018656527,0.0017818959],[-0.0016760714,0.0016669886,0.0016950181,-0.0004634839,-0.0020150175,0.0000734464,0.0077465871,0.0024096869,0.0005048936,0.000445564,-0.0005172183,-0.0062850761],[-0.002494067,0.0007397159,0.0018189049,0,0.0070472017,0.0072204916,-0.0058422,0.0019601873,-0.0026416321,-0.0013716378,-0.0000716017,-0.0018474527],[0.0000832391,0.0002194,0.0023269229,0.0010322264,-0.0077247237,-0.0022634008,-0.0034916515,-0.0005304515,0.0000527305,0.0031858764,0.0064677439,-0.0005384656],[-0.0003008324,0.0001617594,0.0051463011,0.0000046988,0.0028235859,-0.0042445792,-0.0059786882,-0.0023226256,0,0.0000277169,0.0014676675,-0.002199498],[0,-0.0005725861,0.0046430463,0.0024534867,0.0018470333,0.0001116319,-0.0094141982,-0.0105591235,-0.0102753202,-0.0013026739,-0.0004024635,-0.0031451673],[0.0046838498,0.0006416661,0.0031636071,-0.0009002361,0.0011232771,0.0018709009,-0.0110274694,-0.0002920112,-0.0014586739,-0.0004951498,0.0013788168,-0.0001127132],[0.0035604055,-0.0063314722,0.0007916011,-0.0245423608,0.0014674853,0.0023653801,-0.0000793066,-0.0051029188,0.0002184739,-0.002763951,-0.0004873242,-0.0053342435],[0.0027008162,-0.0062669634,-0.002106549,-0.0174228082,0.0005516261,0.0017360971,-0.0002880028,-0.0013660219,0.011951443,0.0116491115,-0.0012028049,0.0069701454],[0.0072587421,-0.0031107781,0.0007864704,-0.0025952207,-0.0032646988,-0.0001758063,-0.0003225393,-0.0007289717,0.0014658039,-0.0008710907,-0.000625691,0.0003478694]],[[-0.0050186014,0.0110852528,0.0014081695,-0.0007150613,0.0033961368,0.006802817,0.0009980218,-0.0020196061,-0.004320821,-0.0021015106,0.0027759238,0.0017818959],[-0.0016760714,0.0007568189,0.0017878037,-0.0004634839,-0.0003195687,0.0000734464,0.0001907752,0.0002120611,-0.0042011632,-0.0045856551,-0.0175222375,-0.0036066408],[0.0013517551,0.0007397159,0.0027169989,0,-0.0124421411,-0.0054095179,-0.0021499868,0.0038173859,0.0004809679,0.0053861774,0.0000472688,0.0045098713],[0.0005126374,0.0002194,0.0033068292,0.0112084363,-0.0006829786,-0.0005686684,-0.0034839597,-0.0014719981,-0.000210922,-0.0001927064,-0.0046522992,-0.0025990649],[0.0013295682,0.0001617594,0.0046254071,0.0008827359,0.0026763284,0.0010611448,0.0014946721,0.0005806564,0,0.0000277169,0.0014676675,0.0130812161],[0,-0.0005725861,0.0038381895,0.0042055428,0.0004477575,-0.0030271793,0.0021113916,0.0022923555,0.0059058313,-0.0013026739,-0.0004024635,-0.0122835192],[-0.0006713,0.0018723852,0.0031636071,0.0064422904,-0.0038825844,-0.0110584387,0.0009878602,0.0022549537,0.0019482958,0.0030632982,-0.0061143297,-0.0037039353],[-0.0026425973,0.0015494783,0.0020610986,0.0106790548,0.0000185344,-0.0005056452,-0.0000793066,0.000453863,0.0011028168,0.0183303996,0.0018074404,0.0027952268],[-0.0005180781,-0.0037641982,0.0016824991,0.0054058014,-0.0033447702,-0.0069443886,-0.0002880028,-0.0013660219,-0.0026476384,-0.0002152336,-0.0021986527,-0.0197205809],[-0.0036774317,-0.0006170252,0.0019268628,0.0129950109,-0.0027548785,-0.0001758063,-0.0003225393,-0.0007289717,-0.0013532536,0.0071150117,-0.0040158102,-0.0071035193]],[[-0.0020379586,-0.0083913715,0.0006856104,0.00541847,-0.0007002669,0.0048215308,0.0046092719,0.0003705117,-0.0020467316,-0.0019947983,-0.0034679217,-0.0009029854],[0.0008576127,-0.0039374453,-0.0004449341,-0.0004634839,0.003047546,-0.0002937855,-0.0082421825,-0.00304587,0.0012295281,0.003248963,0.0106234415,0.0020033969],[-0.0028362521,0.0007397159,0.0018189049,0,0.0059231098,-0.000282719,-0.0009802958,-0.0012086799,0.0028053175,-0.0014719002,-0.0003514304,0.0000257399],[-0.001064182,0.0031146757,0.0012597902,0.0052873709,0.0179895125,0.0077998461,0.0111703404,-0.0005304515,0.0000527305,-0.0025332298,-0.0010386065,-0.0008279667],[-0.0003008324,0.0003037473,0.0036370239,0.0025561496,0.0033275246,0.0010611448,0.0014946721,0.0005806564,0,-0.0001108677,-0.0025188514,-0.006468075],[0,0.0032502531,0.0038381895,0.0005156527,0.0077351689,0.0026922837,0.0021113916,0.0036820571,0.0019972153,0.0070849642,-0.0004024635,0.0133945991],[-0.0013122613,0.0006416661,0.0031636071,-0.0030993719,-0.0012948916,0.0018709009,0.0006318485,-0.0049397492,0.0073499594,0.0055930141,0.0034042641,-0.0013545149],[0.0024815299,0.0031585122,0.0007916011,0.0041809394,-0.0075662715,0.0001274162,0.0003172264,0.0037413298,-0.0042009674,-0.0098185926,0.001472465,0.0014065272],[-0.0005180781,0.0072578103,0.0016824991,0.004005669,0.0005516261,0.0017360971,0.0011520113,0.0054640876,-0.0014222515,-0.0029072831,0.000437263,0.0041124382],[-0.0003403832,0.011295227,0.0026425722,-0.0019013345,0.0062862804,0.000276793,0.0022904664,0.0051302179,-0.0090968535,-0.0045017396,0.0050495733,-0.0007263995]],[[0.0082456848,-0.0008979604,0.0014081695,0.0024438085,-0.0007002669,-0.0247057302,-0.0009304964,0.0011076347,-0.0008066571,0.0013654363,-0.0049646374,-0.0081260519],[0.0041706011,0.0007568189,0.0016950181,0.0018539355,-0.000393391,0.0000734464,0.0001907752,0.0002120611,0.0019618479,0.000445564,0.0049321031,0.0069394058],[0.0026268089,-0.0008492845,0.0018189049,0,-0.0061908095,-0.0007641274,0.0069172081,-0.0065981257,-0.0015832699,-0.0011710017,-0.000104461,-0.0087917547],[0.0002341528,0.0002194,0.0012597902,0.0081895569,-0.0088988314,-0.0027043759,-0.0020973647,0.0030633526,0.0000527305,-0.0002672338,-0.0007721705,-0.0004386907],[-0.0003008324,0.0001617594,0.0036370239,0.0027361825,-0.0086536614,0.0010611448,0.0014946721,0.0005806564,0,0.0000277169,-0.0018841511,-0.0024869586],[0,-0.0005725861,0.0032973032,-0.0055697267,-0.0118769928,0.0001116319,0.0030800234,0.0022923555,0.0017078712,-0.0013026739,0.0016098539,0.0003233959],[-0.0013880271,0.0006416661,-0.0024336407,-0.0009002361,0.0020270995,0.001995983,0.0087759123,0.0104246907,-0.0063809075,-0.0062008452,-0.0000475681,0.0037041374],[-0.0030386513,0.003679577,0.0007916011,0.0048411835,0.0004331096,-0.0077358662,-0.0000793066,0.000453863,0.0026612028,-0.002763951,0.0085191453,0.0015247549],[-0.0005180781,0.0052388453,0.0028723353,0.004005669,0.001689892,0.0017360971,-0.0002880028,-0.0013660219,-0.0040848498,-0.0039656338,0.0025269315,0.0045255593],[-0.0026124405,0.0014884899,0.0058709547,-0.0019013345,-0.0050014931,-0.0001758063,-0.0003225393,-0.0007289717,0.0089762422,-0.0008710907,-0.0005979379,-0.0010506923]]],\"magnitude\":[[[0.0027442696,0.0042574186,0.0074583663,0.00650546,0.0008678209,0.0050613908,0.0023742676,0.0006454045,0.002580664,0.0013654363,0.0058375014,0.0069270322],[0.0025697748,0.000861565,0.0055888867,0.0004634839,0.0012927234,0.0000734464,0.0022427976,0.0005572374,0.0013775291,0.0015158608,0.0078248281,0.0036367332],[0.0035474141,0.0020054634,0.0088259975,0,0.0059695296,0.0024670379,0.0034878098,0.0022674385,0.0018610402,0.001548537,0.001808252,0.0061035962],[0.0002851811,0.0037728757,0.0081533321,0.0257175912,0.0043986685,0.0022634008,0.0026550794,0.0006810447,0.0000527305,0.0011587267,0.0023623219,0.004404188],[0.0004270711,0.0007890256,0.0170457564,0.0065309817,0.0029500457,0.0010611448,0.0014946721,0.0005806564,0,0.0000277169,0.0014676675,0.0022956985],[0,0.0015324947,0.0158330832,0.0055518401,0.0036202391,0.0006718104,0.0023262343,0.0022923555,0.0024843394,0.0031769425,0.0004024635,0.0029961434],[0.0013425676,0.0037973835,0.0092960798,0.0024221007,0.0020270995,0.0053706867,0.0025451049,0.0091211063,0.0014586739,0.0042425955,0.0016184418,0.0019196234],[0.001606563,0.0064163868,0.004435902,0.0050658554,0.0068438955,0.0064167194,0.0000793066,0.0011911741,0.001118374,0.0048007316,0.0113117266,0.0032310356],[0.0011465821,0.0063144813,0.0056374985,0.0045657219,0.0010069324,0.0017360971,0.0002880028,0.0013660219,0.0041519331,0.0048005113,0.0012731304,0.0053611357],[0.0019310609,0.0139633934,0.0126391275,0.0068746758,0.0084631497,0.0016051714,0.0013228486,0.0029433026,0.0045429849,0.0013401729,0.0017434307,0.0089624491]],[[0.0027442696,0.0042574186,0.0036673934,0.0021872458,0.001295336,0.008499712,0.0064281001,0.0006454045,0.0086193089,0.0013654363,0.0044937721,0.003348002],[0.0025697748,0.0016669886,0.0016950181,0.0004634839,0.0023099927,0.0000734464,0.0077465871,0.0024096869,0.0013775291,0.0015158608,0.008135787,0.009118925],[0.0058549073,0.0007397159,0.0018189049,0,0.0070472017,0.0074568478,0.0095108306,0.0023088656,0.003045594,0.001548537,0.0015078972,0.0046300073],[0.0004360948,0.0013775103,0.0024725574,0.006836535,0.0096437972,0.0022634008,0.0034916515,0.0006810447,0.0000527305,0.0031858764,0.0064677439,0.0014026153],[0.0003513279,0.0002185546,0.0051463011,0.0028203023,0.0047484636,0.0042445792,0.0059786882,0.0023226256,0,0.0000277169,0.0014676675,0.0025685119],[0,0.0008558132,0.0046430463,0.0031167748,0.0036202391,0.0006718104,0.0094141982,0.0105591235,0.0103021531,0.0020118568,0.0004024635,0.0056323294],[0.0049402343,0.0011339537,0.0031636071,0.0020367745,0.0025693929,0.003300835,0.0110274694,0.0042909611,0.0014586739,0.003363495,0.0016184418,0.0029006978],[0.0039592185,0.0160135038,0.0012994001,0.0248064584,0.0037411706,0.0037631223,0.0000793066,0.0051029188,0.001118374,0.0046687593,0.0056010008,0.0063331379],[0.0027008162,0.0155241048,0.0046386662,0.0174228082,0.0010069324,0.0017360971,0.0002880028,0.0013660219,0.0133405255,0.0145265547,0.0022571712,0.00711964],[0.0073739836,0.0156894353,0.0065817582,0.0042410998,0.0042237533,0.0013493121,0.000722663,0.0015335285,0.003012579,0.0013401729,0.0012521682,0.0035166274]],[[0.005398693,0.0125336276,0.0014081695,0.0021872458,0.0033961368,0.006802817,0.0023742676,0.0020196061,0.0045614739,0.0034456044,0.005414032,0.003348002],[0.0025697748,0.000861565,0.0017878037,0.0004634839,0.0012927234,0.0000734464,0.0021660672,0.0005572374,0.0042011632,0.0045856551,0.018293264,0.0066915582],[0.0035474141,0.0007397159,0.0038866977,0,0.016936586,0.0054095179,0.0058482666,0.0038173859,0.0017936595,0.0060651218,0.0014365749,0.0046735532],[0.0005126374,0.0013775103,0.0033068292,0.0120203291,0.0043986685,0.0032802402,0.0040416744,0.0014719981,0.000210922,0.0011587267,0.0053728756,0.0026389749],[0.0013295682,0.0002185546,0.0046254071,0.00229348,0.0050613517,0.0010611448,0.0014946721,0.0005806564,0,0.0000277169,0.0014676675,0.0139472738],[0,0.0008558132,0.0038381895,0.0048688308,0.0044598046,0.0030271793,0.0023262343,0.0022923555,0.0066833654,0.0020118568,0.0004024635,0.0122835192],[0.0017271444,0.0018723852,0.0031636071,0.0064422904,0.0055729098,0.0110584387,0.0027587119,0.0041306497,0.0035297695,0.0054985637,0.0061143297,0.007069531],[0.0029757093,0.0030179631,0.0020610986,0.0106790548,0.0057318732,0.0028262241,0.0000793066,0.0011911741,0.0016489798,0.0183303996,0.0037795092,0.0041491913],[0.0005180781,0.0063085333,0.0021495283,0.0054058014,0.0033447702,0.0069443886,0.0002880028,0.0013660219,0.0034691503,0.0082310065,0.0028546798,0.0197205809],[0.0038206249,0.0073775178,0.0043633508,0.0129950109,0.0042945856,0.0013493121,0.000722663,0.0015335285,0.0047554898,0.0072185694,0.0068465602,0.0072332365]],[[0.0061412523,0.0087534652,0.001841705,0.0067553219,0.0008678209,0.0050613908,0.0046092719,0.0007052736,0.0029416434,0.003381577,0.0055470605,0.0093442906],[0.0089828147,0.0039374453,0.0029789894,0.0004634839,0.0033129922,0.0002937855,0.0082421825,0.00304587,0.0018123098,0.003248963,0.0137877819,0.0109495699],[0.0111171307,0.0007397159,0.0018189049,0,0.0062300004,0.0027668311,0.0078283809,0.0042101859,0.0033719705,0.0016487993,0.001833657,0.0028187132],[0.001064182,0.0031146757,0.0014054247,0.0060114221,0.0179895125,0.0084777391,0.0111703404,0.0006810447,0.0000527305,0.0025630407,0.002982685,0.0017320264],[0.0003513279,0.0003037473,0.0036370239,0.0032975282,0.0052652696,0.0010611448,0.0014946721,0.0005806564,0,0.0001108677,0.0038595789,0.0110528539],[0,0.0032502531,0.0038381895,0.0069701619,0.0077351689,0.0026922837,0.0023262343,0.0036820571,0.0041837154,0.0070849642,0.0004024635,0.0139495172],[0.0013425676,0.0011339537,0.0031636071,0.003356256,0.0040202941,0.003300835,0.0025451049,0.0075752016,0.0081074056,0.0070163933,0.0034042641,0.0113390407],[0.0040499669,0.0039833834,0.0012994001,0.0054620018,0.0086662621,0.0021931627,0.0003172264,0.0037413298,0.0042009674,0.0099065741,0.0036412116,0.0021517599],[0.0005180781,0.008314891,0.0021495283,0.0045657219,0.0010069324,0.0017360971,0.0011520113,0.0054640876,0.0102860052,0.006147017,0.0012731304,0.0053611357],[0.0021039231,0.0138916807,0.0039339252,0.003824768,0.0092281053,0.0051413891,0.0022904664,0.0051302179,0.0103826704,0.0049159704,0.0066009086,0.0028720661]],[[0.0102544477,0.0042574186,0.0014081695,0.0043024796,0.0008678209,0.0247057302,0.0035495047,0.0011475474,0.0032191062,0.0013654363,0.01107428,0.010661791],[0.0067806917,0.000861565,0.0016950181,0.0018539355,0.0021333554,0.0000734464,0.0021660672,0.0005572374,0.0022517017,0.0015158608,0.0096871051,0.0082805647],[0.0043124463,0.0016931161,0.0018189049,0,0.0133285332,0.0024670379,0.0074349952,0.0079209087,0.0031234332,0.0038867795,0.0015276128,0.0101682989],[0.0002851811,0.0013775103,0.0014054247,0.0082773984,0.0088988314,0.0027043759,0.0026550794,0.0030633526,0.0000527305,0.0012034432,0.0028228235,0.0013427504],[0.0003513279,0.0002185546,0.0036370239,0.003405548,0.0086536614,0.0010611448,0.0014946721,0.0005806564,0,0.0000277169,0.0034787587,0.0028559726],[0,0.0008558132,0.0041627212,0.0079307029,0.0124367031,0.0006718104,0.0030800234,0.0022923555,0.0035150292,0.0020118568,0.0016098539,0.0021637662],[0.0013880271,0.0011339537,0.0065219558,0.0020367745,0.0020270995,0.0033758842,0.0089183169,0.0110205295,0.0063809075,0.0067869122,0.0030448266,0.0038493064],[0.0039514154,0.0051480618,0.0012994001,0.0050658554,0.0031205452,0.0077358662,0.0000793066,0.0011911741,0.0030149399,0.0046687593,0.0085852804,0.0022699877],[0.0005180781,0.0084533586,0.0029240362,0.0045657219,0.001689892,0.0017360971,0.0002880028,0.0013660219,0.0045509815,0.0069925321,0.0025269315,0.0056247623],[0.0037718212,0.0084978122,0.0058709547,0.003824768,0.00699646,0.0013493121,0.000722663,0.0015335285,0.0089762422,0.0013401729,0.00126882,0.0030666418]]],\"nonzero\":[[[0,0,3,3,0,0,0,0,0,0,3,2],[0,0,4,0,0,0,1,0,0,0,1,0],[0,1,6,0,0,0,0,0,1,0,2,4],[0,2,5,13,0,0,0,0,0,0,0,4],[1,2,6,3,0,0,0,0,0,0,0,0],[0,1,5,2,0,0,0,0,0,1,0,1],[0,2,5,1,0,1,0,7,0,1,0,0],[0,2,2,0,3,5,0,0,0,1,5,3],[1,4,5,0,0,0,0,0,2,0,0,0],[0,9,6,1,5,1,1,2,1,0,2,6]],[[0,0,1,0,1,2,4,0,6,0,1,0],[0,2,0,0,2,0,5,2,0,0,1,5],[1,0,0,0,2,7,2,1,1,0,1,1],[1,0,1,3,3,0,1,0,0,4,6,1],[0,0,1,1,1,4,3,3,0,0,0,1],[0,0,1,0,0,0,3,5,3,0,0,1],[4,0,0,0,1,0,4,2,0,0,0,1],[1,7,0,9,1,3,0,1,0,0,1,3],[3,7,3,8,0,0,0,0,9,8,1,3],[4,9,3,1,1,0,0,0,1,0,0,1]],[[1,6,0,0,5,2,0,1,2,2,1,0],[0,0,1,0,0,0,0,0,1,4,6,3],[0,0,2,0,4,4,4,3,1,6,1,2],[1,0,2,4,0,1,1,1,1,0,3,2],[1,0,1,0,2,0,0,0,0,0,0,3],[0,0,0,1,1,3,0,0,1,0,0,5],[1,1,0,4,1,4,1,1,4,2,1,2],[1,0,3,5,5,1,0,0,1,5,2,3],[0,1,0,1,2,4,0,0,0,2,5,6],[1,1,1,6,2,0,0,0,2,3,3,3]],[[3,5,1,2,0,0,3,1,1,2,4,2],[3,1,2,0,1,1,4,3,1,4,5,5],[5,0,0,0,1,1,6,1,5,1,1,1],[1,3,0,1,12,4,6,0,0,2,1,1],[0,1,0,1,2,0,0,0,0,1,2,5],[0,4,0,3,3,2,0,1,3,4,0,5],[0,0,0,1,1,0,0,2,5,1,1,4],[3,1,0,1,4,0,1,3,3,3,0,0],[0,2,0,0,0,0,1,1,4,2,0,0],[2,4,0,0,6,3,2,6,3,3,4,0]],[[6,0,0,3,0,8,3,1,1,0,5,2],[4,0,0,2,2,0,0,0,4,0,3,5],[1,1,0,0,4,0,2,2,2,5,2,5],[0,0,0,2,4,1,0,3,0,1,1,0],[0,0,0,1,9,0,0,0,0,0,1,1],[0,0,1,3,6,0,1,0,3,0,1,0],[1,0,5,0,0,1,10,6,3,2,1,3],[3,1,0,0,0,5,0,0,1,0,4,1],[0,2,3,0,1,0,0,0,1,5,2,2],[2,2,3,0,4,0,0,0,3,0,1,2]]]},{\"rat\":\"Buchanan\",\"target\":\"odor\",\"n_channels\":20,\"signed\":[[[0.0213899528,-0.0016606,0.0085749645,0.0021872257,-0.0022242714,0.0031697366,-0.0009374572,0.0002680404,0.0005996461,0.0101654769,0.0051459258,0.0136018195],[0.0062753809,-0.0010848972,-0.0158534942,-0.0017285544,-0.0010247903,-0.0021854727,0.0010500657,0.0004667319,0.0068048264,0.0138766414,0.0041690836,-0.0004395424],[0.0006286859,-0.0057806909,-0.0186727296,0.0022451015,-0.0024889655,-0.010457458,0.0016672086,0.0037457624,0.0047477418,0.0058391657,0.003356102,0.0069093872],[0.0000101324,-0.0082797521,-0.008474491,0.0004086652,-0.0019674594,-0.0006846796,0.0001535581,0.0022988749,0.002611237,0.0083848399,0.0005013014,0.0037597567],[0,-0.0088837196,-0.0014840253,-0.0003121095,-0.0007563014,-0.0022248509,-0.0010502819,0.0004925578,0.0010403625,0.0001210409,0.0021515351,0.0000480965],[2.112e-7,-0.0072666745,-0.0019476332,0,-0.0011330841,0.0015413235,0.0023998729,0.0075258816,0.0023549421,0.0007279657,0.0006511143,-0.0029088531],[-7.827e-7,-0.0030801476,-0.0003144886,0.0059736574,0.0004978377,0.000783201,0.0008458497,-0.0014607797,0.0002925707,0.0005330641,0.0000776861,-0.004262726],[0.0014791819,0.0005805274,-0.0002309523,-0.000612843,0.0019092335,-0.0000575973,0,0.0013047322,0.0029207616,-0.000964011,-0.0007709432,-0.0048862409],[-0.000422438,-0.0009957192,-0.0045377553,-0.000017509,0.0013761889,-0.0019018517,0.0013971066,-0.0001811597,-0.0006883647,-0.0011202978,0.0017300043,-0.0097428968],[-0.0007095802,-0.000452963,-0.0123483215,-0.002913713,0.0021544498,0.0016360477,0.0020898647,0.0001663773,0.0006108641,0.0004297065,0.0019664188,0.0002978355]],[[-0.0071327625,0.0001384542,-0.0026910643,-0.0015042179,0.0026187521,0.0031697366,-0.0004651278,0.0002680404,-0.0000833801,0.0085452437,-0.0015856685,-0.0176349539],[-0.0010229821,-0.003270346,0.0031465995,-0.0003602744,0.0073203769,0.0001221516,0.0042852808,-0.0027279264,0.003939033,0.0008926463,0.0066988479,0.004542647],[0.0004422604,-0.0015139547,0.0043776058,0.0028410287,0.0009077519,-0.0026307464,0.0034696542,-0.0040427929,-0.0055707161,0.0007923021,0.0067010501,0.0099218064],[-0.0003696755,0.0001008272,0.0030649577,-0.0023836026,-0.0009646333,-0.0022327405,0.0038352031,0.0024050636,0.0003859213,-0.015893573,-0.0015490509,-0.0028920384],[0,-0.0002031258,0.0021239401,-0.0003121095,0.0021548099,0.0011455676,0.0047916969,-0.0095495223,-0.0047762063,0.0001210409,-0.0121977223,-0.0036760177],[0.0000450948,0.0017652207,0.0001922611,0,0.011354427,0.0015413235,-0.0102100382,-0.0196080217,-0.0101599176,-0.0037719037,0.004046499,0.0019701881],[0.0004217588,0.004712119,-0.0009456272,0.0007708314,0.0032764861,-0.0015887348,0.0008458497,-0.0002839161,0.0032099618,0.0005330641,0.0007602251,-0.0018334448],[0.002055027,0.0012003467,0.0003766961,-0.0100668759,0.0030583111,-0.0000575973,0,-0.0052366248,-0.002985224,0.0041732392,-0.0018758845,-0.0006012273],[0.0063325219,0.0008546381,-0.0013569408,-0.0043887191,0.0141267521,0.002113477,0.0013971066,-0.0001811597,-0.0006883647,0.0081152258,0.0017300043,0.0037162541],[0.0022220103,0.0001132407,-0.0063174255,-0.0044685576,-0.0002373474,-0.0070356749,-0.0053680237,-0.0006655092,0.0006108641,0.0004297065,0.0010369204,-0.001191342]],[[-0.0023626675,-0.0028596588,-0.0026910643,0.0026258868,-0.0044699911,-0.0130767853,-0.0023271494,0.0002680404,0.0043029133,0.0061219902,-0.0009229917,0.0015175765],[-0.0011431855,-0.0002937277,0.0072803708,0.0040860931,-0.0010247903,-0.0006599886,-0.0062673773,0.0005829075,0.0068048264,-0.0062531263,0.0083295076,-0.0004395424],[-0.0017871535,-0.000194857,0.0082239845,0.0022451015,-0.0041892937,-0.0001754445,-0.0077108007,0.000287808,-0.000626609,0.0007923021,-0.0049465241,-0.0256359844],[-0.0003696755,0.0025260653,0.0046013743,0.0007947018,-0.0009646333,-0.0006846796,-0.000335521,-0.003735298,0.0072430531,0.0032811021,0.0005013014,-0.0096122422],[0,0.0036005214,0.0004785609,0.0009349391,0.0015645051,0.0009624194,-0.0149553604,-0.0153540443,0.000222222,-0.0015216916,-0.0115735537,0.0000480965],[2.112e-7,0.0017652207,-0.0019476332,0,-0.0021364304,0.0015413235,-0.0006731927,0.0056762077,0.0050011505,0.0017071228,0.0006511143,-0.0005864978],[-0.0023525292,0.0005198539,-0.0010145669,-0.0155720786,-0.0008254374,0.0015848086,0.0008458497,-0.0014607797,-0.0005132994,-0.0017049846,-0.0024154416,-0.0030344062],[-0.0095000181,-0.0008692798,-0.0003560753,0.0088035097,0.0019092335,-0.0060991111,0,-0.0010994005,0.0029207616,0.0043219595,0.0041887142,0.0016027801],[-0.0097642975,-0.001568195,0.0019648987,0.0048126922,-0.0012581779,-0.0019018517,0.0013971066,-0.0001811597,-0.0006883647,-0.0011202978,0.0017300043,0.0064753096],[-0.0081464394,0.0001132407,0.0045013307,-0.0000469557,-0.0002455831,0.0026270188,0.0020898647,0.0001663773,0.0006108641,0.0004297065,0.0069488744,0.0002978355]],[[-0.0095318552,0.0038565093,-0.0005017717,-0.0056791646,0.0036796132,0.0035930636,-0.0009900944,-0.0010721617,-0.004510206,-0.0268759705,-0.0014755439,0.002053547],[-0.0035153363,0.0048703455,0.0027132619,0.0011990812,0.0026159859,0.0001221516,-0.002878339,0.0012115551,0.0093115861,0.0008666845,-0.0195676386,-0.0032240197],[0.0016905875,0.0086371176,0.0030355696,0.002390963,0.0073919307,0.0069455698,0.0016672086,-0.000229929,0.0051410736,-0.006270515,-0.001250131,0.0036355406],[0.0010988941,0.0047511249,0.0004040795,0.0005901178,0.0048613593,0.002507157,-0.0001330096,0.0022988749,-0.0022224205,0.0021138154,0.0000451467,-0.0028920384],[0,0.002743162,-0.0027275128,0.0000013894,0.0005075771,-0.001498944,0.0132992367,0.0236236113,0.0032913999,0.0011585688,0.0122548033,0.0000480965],[2.112e-7,0.0017652207,-0.0019476332,0,0.0007424107,-0.0061652938,-0.0006731927,0.002874447,0.0003647899,0.0006705227,-0.0043730124,0.0007625814],[-7.827e-7,-0.0025764811,-0.0003144886,0.002429598,-0.0005843508,-0.0015624758,0.0008458497,0.0056474704,0.000367731,0.0013861128,0.0053682965,-0.0008492017],[0.0042418753,-0.0021119411,-0.0002212854,0.0035189807,0.0032888259,-0.002480832,0,0.0002914783,-0.0057770608,-0.000964011,-0.0007709432,0.0075236904],[0.0042801181,0.0008546381,0.0019648987,0.0009675849,0.0020957936,-0.0116826575,-0.0068502098,0.002493184,0.0033818968,-0.0011202978,-0.0003185193,-0.0014755197],[0.0103508796,0.0001132407,0.0090927631,0.0074761818,-0.0014341719,-0.0003778903,-0.0036953897,0.0001663773,-0.00537305,0.0000983654,-0.0074756143,0.0002978355]],[[-0.0023626675,0.0005252951,-0.0026910643,0.00237027,0.000395897,0.0031442483,0.0047198291,0.0002680404,-0.0003089733,0.0020432599,-0.0011617217,0.000462011],[-0.000593877,-0.0002213746,0.0027132619,-0.0031963456,-0.0078867819,0.0026011581,0.0038103699,0.0004667319,-0.0268602716,-0.0093828459,0.0003701993,-0.0004395424],[-0.0009743803,-0.001147615,0.0030355696,-0.009722195,-0.0016214234,0.0063180792,0.0009067293,0.0002391513,-0.0036914904,-0.0011532547,-0.0038604973,0.0051692503],[-0.0003696755,0.000901735,0.0004040795,0.0005901178,-0.0009646333,0.0010949428,-0.0035202308,-0.0032675153,-0.0080177909,0.0021138154,0.0005013014,0.011636563],[0,0.002743162,0.0016090371,-0.0003121095,-0.0034705907,0.0016158079,-0.0020852914,0.0007873981,0.000222222,0.0001210409,0.0093649382,0.0035317283],[-0.0000457283,0.0019710123,0.0056506385,0,-0.0088273233,0.0015413235,0.0091565502,0.0035314861,0.0024390351,0.0006662926,-0.0009757153,0.0007625814],[0.0019323358,0.0004246558,0.0025891714,0.0063979917,-0.0023645356,0.000783201,-0.0033833988,-0.0024419949,-0.0033569642,-0.0007472564,-0.0037907659,0.0099797786],[0.0017239337,0.0012003467,0.0004316168,-0.0016427713,-0.0101656042,0.0086951378,0,0.0047398148,0.0029207616,-0.0065671768,-0.0007709432,-0.0036390024],[-0.0004259042,0.0008546381,0.0019648987,-0.001374049,-0.0163405576,0.0133728842,0.0026588898,-0.0019497049,-0.0013168028,-0.0047543323,-0.0048714938,0.0010268525],[-0.0037168705,0.0001132407,0.0050716537,-0.0000469557,-0.0002373474,0.0031504986,0.0048836839,0.0001663773,0.0035404578,-0.0013874849,-0.0024765992,0.0002978355]]],\"magnitude\":[[[0.0213899528,0.0024673801,0.0136144608,0.0037763598,0.0035409214,0.0031697366,0.0019454772,0.0002680404,0.0022280411,0.0108245409,0.0051459258,0.0136018195],[0.0062753809,0.0033610056,0.0158534942,0.0026556252,0.0024872006,0.0024983288,0.0022331987,0.0008111314,0.0068048264,0.0138766414,0.0052290417,0.0010831443],[0.0011280167,0.0064485237,0.0186727296,0.0033254033,0.0031725405,0.010457458,0.0023881868,0.0037457624,0.0078220578,0.0058442827,0.0053227159,0.0093906833],[0.0005975602,0.0086649656,0.0087314035,0.0006989894,0.0019674594,0.001303904,0.0015115891,0.0022988749,0.002611237,0.0083848399,0.0005013014,0.0071385246],[0,0.0100622347,0.0035805656,0.0003121095,0.0018420171,0.0032093963,0.0072471026,0.0044038405,0.001726047,0.0005360521,0.0051727729,0.0000480965],[2.112e-7,0.0072666745,0.0019476332,0,0.0047407139,0.0017253679,0.0064606936,0.0107155732,0.0033275072,0.0023806067,0.0020092682,0.0032803636],[0.0009414813,0.0078235075,0.0005945199,0.0070820292,0.0011130073,0.000783201,0.0008458497,0.0018532658,0.0021342874,0.000956279,0.0013959192,0.0043537176],[0.0024421088,0.007614712,0.0007632535,0.0042310479,0.003434748,0.0044564068,0,0.0030840668,0.0029207616,0.0025022105,0.0007709432,0.0048862409],[0.0027263905,0.0129741091,0.0058664911,0.001765993,0.0058474458,0.0036313558,0.0013971066,0.0008885778,0.0009397399,0.0025739116,0.0017300043,0.0103236409],[0.0031972377,0.017462477,0.0148392511,0.0036216011,0.006498837,0.0022821329,0.0032073924,0.0001663773,0.0017827016,0.0005018839,0.0033358549,0.0002978355]],[[0.0100004376,0.0059547577,0.0037320091,0.0048403567,0.0047732869,0.0031697366,0.0024178067,0.0002680404,0.0021248223,0.0097127759,0.0019021814,0.0176886335],[0.0043993525,0.0077550106,0.0049734431,0.0017894526,0.0083781722,0.0011137542,0.0053018789,0.0027279264,0.0085243025,0.0062849268,0.0071140488,0.004542647],[0.0010161614,0.0032558819,0.0064529718,0.0036829596,0.0038225035,0.0061919835,0.0034696542,0.0042693504,0.0087707214,0.0016452178,0.0067010501,0.0111657753],[0.0003696755,0.0030921438,0.0057715259,0.0024561837,0.0009646333,0.0022327405,0.0040804008,0.0024050636,0.0032830148,0.015893573,0.0017315128,0.0028920384],[0,0.0048538784,0.0058325737,0.0003121095,0.0035886838,0.001406923,0.0107522898,0.0105084378,0.0047762063,0.0005360521,0.0149839992,0.0036760177],[0.0000450948,0.0018475373,0.0032315698,0,0.011354427,0.0017253679,0.0106578925,0.0218152765,0.010836423,0.0050805284,0.004046499,0.0019701881],[0.0011950062,0.0058465715,0.0012256585,0.0059508535,0.0032764861,0.0025270055,0.0008458497,0.002559384,0.003781029,0.000956279,0.0018054426,0.0025161157],[0.0029005308,0.0027109652,0.0003986644,0.0103235277,0.0043499097,0.0044564068,0,0.0057929763,0.0060935113,0.0055845606,0.0018758845,0.0012712942],[0.007445308,0.0036069522,0.0039580024,0.0043887191,0.0143035523,0.0058257683,0.0013971066,0.0008885778,0.0009397399,0.0081152258,0.0017300043,0.005895637],[0.0058295515,0.0043656192,0.0128291573,0.0049667501,0.0018054683,0.0075915534,0.0076821255,0.0006655092,0.0017827016,0.0005018839,0.0038935539,0.001191342]],[[0.0071383806,0.0035434342,0.0037320091,0.0033432387,0.0044699911,0.0130869807,0.004896104,0.0002680404,0.0043029133,0.0079485865,0.0015045753,0.0034506682],[0.0048422368,0.0028863039,0.0074537058,0.0040860931,0.0024872006,0.0015830383,0.0078387391,0.0008808367,0.0068048264,0.0105723903,0.0098046667,0.0010831443],[0.0027524528,0.0034843747,0.0087607989,0.0033254033,0.0088873363,0.0037366816,0.0085786006,0.000287808,0.0038266143,0.0016452178,0.0059906706,0.0275384517],[0.0003696755,0.0035861973,0.0047844229,0.0007947018,0.0009646333,0.001303904,0.0018050365,0.0059193786,0.0072430531,0.0057895119,0.0005013014,0.0096122422],[0,0.0036005214,0.0024030139,0.0010603387,0.003234501,0.0012970341,0.0155901497,0.0189514296,0.0012351627,0.0015216916,0.0162511244,0.0000480965],[2.112e-7,0.0018475373,0.0019476332,0,0.0053427217,0.0017253679,0.0046168543,0.0085216822,0.0050347877,0.0029681009,0.0020092682,0.0020550716],[0.0023525292,0.002202331,0.0010145669,0.0155720786,0.0019069723,0.0015848086,0.0008458497,0.0018532658,0.0015470723,0.0022991082,0.0028917958,0.0036260855],[0.0095000181,0.003952741,0.0008383273,0.00923003,0.003434748,0.0114934538,0,0.0033106417,0.0029207616,0.0063768596,0.0041887142,0.0034753016],[0.0097642975,0.0050606521,0.0019648987,0.0052067298,0.0068976654,0.0036313558,0.0013971066,0.0008885778,0.0009397399,0.0025739116,0.0017300043,0.0067087307],[0.0081464394,0.0043656192,0.0045013307,0.0012530362,0.0018104096,0.0028767155,0.0032073924,0.0001663773,0.0017827016,0.0005018839,0.0069488744,0.0002978355]],[[0.0114398932,0.0057776457,0.0050455846,0.0070183431,0.0058281656,0.0035930636,0.0022483401,0.0010721617,0.004510206,0.0282785387,0.0018361067,0.0039866387],[0.0058947651,0.0058979241,0.0047134405,0.0016686382,0.0049163608,0.0011137542,0.0045902415,0.0012580253,0.0093115861,0.0062732151,0.0206719914,0.0032240197],[0.0017651577,0.0115968409,0.00564775,0.0102867054,0.0100908686,0.008991411,0.0023881868,0.0005984502,0.0132156891,0.006270515,0.0031731761,0.0099431773],[0.0012508173,0.0056602586,0.002266046,0.0005901178,0.0048613593,0.003219006,0.0027789474,0.0022988749,0.0054609905,0.0046222252,0.0007749943,0.0028920384],[0,0.0030861058,0.0043266581,0.0005002088,0.0026003442,0.0027738521,0.0165835753,0.0236236113,0.0036186561,0.0011585688,0.0140780322,0.0000480965],[2.112e-7,0.0018475373,0.0019476332,0,0.003615417,0.0069014714,0.0046168543,0.0069645574,0.0046411539,0.006198007,0.0050237442,0.001077503],[0.0009414813,0.0051463491,0.0005945199,0.0042920869,0.0017623204,0.002190607,0.0008458497,0.0061182159,0.0020756905,0.0017140945,0.0056413121,0.0031066616],[0.0044571991,0.0046983379,0.0007574533,0.0054641238,0.0100566703,0.007403982,0,0.0024761145,0.0075523384,0.0025022105,0.0007709432,0.0075236904],[0.0063161606,0.0036069522,0.0019648987,0.0023570494,0.0048852825,0.0116826575,0.0068502098,0.002493184,0.0033818968,0.0025739116,0.0029591185,0.0053612845],[0.0106501563,0.0043656192,0.0090927631,0.0074761818,0.002523563,0.0027414056,0.0066785451,0.0001663773,0.00537305,0.0009172206,0.0090010747,0.0002978355]],[[0.0071383806,0.0037789172,0.0037320091,0.002862887,0.0034395738,0.0031850297,0.005339849,0.0002680404,0.0018992291,0.0106247487,0.0016478134,0.0039659988],[0.0056249385,0.002842892,0.0047134405,0.003859539,0.0078867819,0.0026011581,0.0039769049,0.0008111314,0.028006589,0.0124575689,0.0062627695,0.0010831443],[0.0034607178,0.0029127199,0.00564775,0.0116365379,0.0023049983,0.0081843642,0.005098907,0.0003170021,0.0121957269,0.0036112429,0.0047393959,0.0114724822],[0.0003696755,0.0026115991,0.002266046,0.0005901178,0.0009646333,0.0023716774,0.0037158624,0.0056811844,0.0136740128,0.0046222252,0.0005013014,0.014041872],[0,0.0030861058,0.0030812996,0.0003121095,0.0034705907,0.0016890671,0.0078842095,0.0046986808,0.0012351627,0.0005360521,0.0111211052,0.0035317283],[0.0000457283,0.0019710123,0.0065065962,0,0.0107290577,0.0017253679,0.0121771937,0.0151136348,0.0033779631,0.0024176106,0.002985366,0.001077503],[0.0021013524,0.0021452122,0.0025891714,0.0071521088,0.003326721,0.000783201,0.0033833988,0.0024419949,0.0033569642,0.0017244713,0.004788017,0.0103734759],[0.0025889599,0.0027109652,0.0004316168,0.0023704468,0.0181733023,0.0097080479,0,0.0051451164,0.0029207616,0.0065671768,0.0007709432,0.0043090692],[0.0027298567,0.0036069522,0.0019648987,0.003122533,0.0181467882,0.0141198759,0.0026588898,0.0019497049,0.0013168028,0.0047543323,0.0056909032,0.0034396565],[0.006204528,0.0043656192,0.0050716537,0.0012530362,0.0018054683,0.0040462804,0.0048836839,0.0001663773,0.0035404578,0.0015921987,0.0066252118,0.0002978355]]],\"nonzero\":[[[6,0,6,2,1,0,0,0,1,2,1,6],[3,1,6,2,0,2,0,0,0,3,1,0],[1,2,6,0,1,3,0,1,2,3,2,4],[1,3,4,1,1,0,0,0,1,2,0,4],[0,2,2,0,0,1,0,0,1,0,0,0],[0,4,0,0,1,0,2,1,0,0,0,1],[0,3,0,1,0,0,0,0,2,0,0,1],[0,3,1,2,0,0,0,2,0,0,0,4],[0,6,2,0,2,0,0,0,0,0,0,6],[0,5,3,2,6,1,0,0,0,0,0,0]],[[2,4,0,1,1,0,1,0,1,3,1,7],[2,3,1,1,5,0,1,3,1,0,1,2],[0,1,1,2,2,2,1,1,1,0,2,1],[0,1,2,2,0,1,1,1,1,4,1,0],[0,3,2,0,2,1,1,3,1,0,3,1],[1,0,1,0,4,0,3,4,4,1,1,2],[1,1,1,1,1,2,0,2,2,0,1,0],[1,0,0,4,1,0,0,4,2,3,1,0],[3,0,1,2,7,1,0,0,0,2,0,3],[1,0,2,1,0,2,2,1,0,0,1,1]],[[0,2,0,1,1,4,3,0,2,0,0,0],[2,1,2,3,0,1,4,1,0,2,1,0],[1,2,1,0,2,0,3,0,0,0,1,7],[0,1,1,1,0,0,1,2,1,1,0,2],[0,1,0,1,1,0,4,7,0,2,3,0],[0,0,0,0,2,0,0,3,1,1,0,2],[1,1,1,5,1,1,0,0,0,1,1,2],[3,1,1,3,0,3,0,1,0,2,1,1],[4,1,0,1,1,0,0,0,0,0,0,2],[4,0,0,0,1,1,0,0,0,0,2,0]],[[2,1,1,1,1,1,2,2,3,9,1,1],[1,5,0,1,3,0,2,1,1,2,5,2],[1,7,0,2,4,2,0,1,4,2,0,1],[1,3,0,0,1,1,2,0,1,0,1,0],[0,0,2,1,1,2,5,5,2,1,2,0],[0,0,0,0,0,4,0,1,2,2,2,0],[0,2,0,0,1,1,0,2,1,2,3,1],[2,1,1,2,4,2,0,0,4,0,0,2],[3,0,0,1,0,3,1,2,2,0,1,1],[3,0,1,2,1,1,2,0,3,2,2,0]],[[0,1,0,1,1,1,1,0,0,2,1,2],[2,0,0,2,3,1,4,0,7,4,3,0],[2,0,0,2,0,2,2,1,4,2,2,2],[0,0,0,0,0,1,1,2,3,0,0,1],[0,0,2,0,1,1,1,1,0,0,3,1],[1,1,5,0,4,0,3,4,1,1,1,0],[1,0,3,3,2,0,1,1,2,1,1,5],[1,0,1,0,6,3,0,1,0,3,0,1],[1,0,0,1,4,5,1,1,1,1,2,0],[2,0,1,0,0,1,1,0,1,1,3,0]]]},{\"rat\":\"Buchanan\",\"target\":\"position\",\"n_channels\":20,\"signed\":[[[0.0197842099,-0.0002234785,0.0001305797,0.0012320397,0.0001012798,0.0009324358,0,0,0,0.0033918433,0.0033037968,0.0030206187],[0.0001368778,0.0018393107,-0.0112015165,-0.0003991536,0,0,0.0014311032,0.0002596896,0.0026571384,0.0055171265,0.0012928488,0],[0,0,-0.0103353585,0,-0.0007134717,0,0.0010672365,0.0000882725,0.0020418678,0.0001297029,-0.0010612787,0.0049314035],[0,-0.0056859955,-0.0026968403,0,0,0,0.0011902796,-0.0002428537,0.0008011838,0.0031276946,0,-0.0014557325],[0,-0.007618548,0,0,0.000047866,0.000472722,0,0.0022566656,0.0003337587,0,0.0030443277,-0.0006900692],[0,-0.0039476298,0,0,-0.0014480969,0,-0.0013800118,0.0019652032,0,-0.0002657068,0,0],[-0.0006105342,-0.0019780645,0,0.0006820546,0,0,0,0,0.0003152128,0,-0.0000974486,-0.0001650553],[0.0007074981,0,0,0.0000331898,0.000171795,0.0000827037,0,-0.0000479224,0,0,-0.0005521459,0],[0.0012801605,-0.002101003,-0.001655153,0,0.0001638281,-0.0000936331,-0.0005702164,0,0,0,0.0001320908,-0.0002045558],[0.0006528788,0.0023440115,-0.0158342738,-0.0000392084,0,0,-0.0004526536,0,0.000273292,0,0.0006238979,0]],[[-0.0049460525,-0.0003060083,0.0000287097,0.0012320397,0.0001012798,0.0009324358,0,0,0,0.0033918433,-0.0008259492,-0.0007551547],[0.0001368778,-0.0073572427,0.0028003791,-0.0003991536,0,0,0.0014311032,-0.0010387583,0.0026571384,-0.0008831392,0.0012928488,0],[0,0,0.0012602746,0,-0.0007134717,0,0.0010672365,-0.0003530899,0.0020418678,-0.0000324257,0.004245115,0.0049314035],[0,0.0014214989,0.0003028682,0,0,0,0.0027742359,0.0009714146,0.0008011838,-0.0106930638,0,-0.0019925677],[0,0.001904637,0,0,0.000047866,0.000472722,0,-0.0047900079,-0.0013350348,0,-0.0101276152,-0.0006900692],[0,0.0009869074,0,0,0.0057923876,0,-0.0050504361,-0.0068785456,0,-0.0002657068,0,0],[-0.0006105342,0.0012393209,0,-0.0007063427,0,0,0,0,0.0003152128,0,-0.0000974486,-0.0001650553],[0.0007074981,0,0,-0.0001327591,0.000171795,0.0000827037,0,-0.0000479224,0,0,0.0022085836,0],[0.0012801605,0.0005252508,0.0004137882,0,0.0097372192,0.0003745325,-0.0005702164,0,0,0,0.0001320908,-0.0002045558],[0.0006528788,-0.0005860029,-0.0053336485,-0.0000179157,0,0,-0.0004526536,0,-0.0010931681,0,0.0006238979,0]],[[-0.0049460525,-0.0002234785,0.0000287097,0.0012320397,-0.0004051192,-0.0037297433,0,0,0,0.0033918433,-0.0008259492,-0.0007551547],[0.0001368778,0.0018393107,0.0028003791,0.0015966145,0,0,-0.0057244129,0.0002596896,0.0026571384,-0.0028506406,0.0012928488,0],[0,0,0.0065545347,0,-0.0007134717,0,-0.0028251896,0.0000882725,0.0020418678,-0.0000324257,-0.0010612787,-0.019725614],[0,0.0014214989,0.0017882355,0,0,0,0.0011902796,-0.0002428537,0.0008011838,0.0025217897,0,-0.0019925677],[0,0.001904637,0,0,0.000047866,-0.0018908881,0,-0.0096968259,0.0003337587,0,-0.003554225,-0.0006900692],[0,0.0009869074,0,0,-0.0014480969,0,-0.0013800118,0.0019652032,0,0.0025113313,0,0],[-0.0006105342,0.0012393209,0,-0.0013398213,0,0,0,0,0.0003152128,0,-0.0000974486,-0.0084994218],[-0.0028299924,0,0,0.0000331898,0.000171795,-0.000330815,0,-0.0000479224,0,0,-0.0005521459,0],[-0.0051206421,0.0005252508,0.0004137882,0,0.0001638281,-0.0000936331,-0.0005702164,0,0,0,0.0001320908,0.000818223],[-0.0026115151,-0.0005860029,0.0070559741,-0.0000179157,0,0,-0.0004526536,0,0.000273292,0,0.0006238979,0]],[[-0.0049460525,-0.0002234785,0.0000287097,-0.004928159,0.0001012798,0.0009324358,0,0,0,-0.0135673732,-0.0008259492,-0.0007551547],[-0.000547511,0.0018393107,0.0028003791,-0.0003991536,0,0,0.0014311032,0.0002596896,0.0026571384,-0.0008831392,-0.0051713951,0],[0,0,0.0012602746,0,0.0028538868,0,0.0010672365,0.0000882725,0.0020418678,-0.0000324257,-0.0010612787,0.0049314035],[0,0.0014214989,0.0003028682,0,0,0,0.0002731769,-0.0002428537,0.0008011838,0.0025217897,0,-0.0019925677],[0,0.001904637,0,0,0.000047866,0.000472722,0,0.0099735029,0.0003337587,0,0.0075931845,-0.0006900692],[0,0.0009869074,0,0,-0.0014480969,0,-0.0013800118,0.0019652032,0,-0.0017142109,0,0],[-0.0006105342,-0.0017398983,0,0.0006820546,0,0,0,0,0.0003152128,0,0.0003897943,0.0015770832],[0.0007074981,0,0,0.0000331898,0.000171795,0.0000827037,0,-0.0000479224,0,0,-0.0005521459,0],[0.0012801605,0.0005252508,0.0004137882,0,0.0001638281,-0.0000936331,-0.0005702164,0,0,0,0.0001320908,-0.0002045558],[0.0006528788,-0.0005860029,0.0070559741,0.0000929556,0,0,-0.0034048875,0,0.000273292,0,0.0003871075,0]],[[-0.0049460525,0.0009764437,-0.0002167089,0.0012320397,0.0001012798,0.0009324358,0,0,0,0.0033918433,-0.0008259492,-0.0007551547],[0.0001368778,0.0018393107,0.0028003791,-0.0003991536,0,0,0.0014311032,0.0002596896,-0.0106285538,-0.0009002076,0.0012928488,0],[0,0,0.0012602746,0,-0.0007134717,0,-0.0003765199,0.0000882725,-0.0081674714,-0.0000324257,-0.0010612787,0.0049314035],[0,0.0014214989,0.0003028682,0,0,0,-0.005427972,-0.0002428537,-0.0032047354,0.0025217897,0,0.0074334356],[0,0.001904637,0,0,-0.0001914641,0.000472722,0,0.0022566656,0.0003337587,0,0.0030443277,0.0027602768],[0,0.0009869074,0,0,-0.0014480969,0,0.0091904717,0.000982936,0,-0.0002657068,0,0],[0.0024421368,0.0012393209,0,0.0006820546,0,0,0,0,-0.001260851,0,-0.0000974486,0.0072524493],[0.0007074981,0,0,0.0000331898,-0.0006871801,0.0000827037,0,0.0001916897,0,0,-0.0005521459,0],[0.0012801605,0.0005252508,0.0004137882,0,-0.0102287036,-0.0000936331,0.0022808656,0,0,0,-0.0005283632,-0.0002045558],[0.0006528788,-0.0005860029,0.0070559741,-0.0000179157,0,0,0.0047628483,0,0.000273292,0,-0.0022588013,0]]],\"magnitude\":[[[0.0197842099,0.0002564904,0.0001305797,0.0012320397,0.0001012798,0.0009324358,0,0,0,0.0033918433,0.0033037968,0.0030206187],[0.0001368778,0.0018393107,0.0112015165,0.0003991536,0,0,0.0014311032,0.0002596896,0.0026571384,0.0055171265,0.0012928488,0],[0,0,0.0103353585,0,0.0007134717,0,0.0010672365,0.0000882725,0.0020418678,0.0001297029,0.0010612787,0.0049314035],[0,0.0056859955,0.0026968403,0,0,0,0.0014570211,0.0002428537,0.0008011838,0.0031276946,0,0.0023146688],[0,0.007618548,0,0,0.000047866,0.000472722,0,0.0041376099,0.0003337587,0,0.0033458785,0.0006900692],[0,0.0039476298,0,0,0.0014480969,0,0.0013800118,0.0019652032,0,0.0008451084,0,0],[0.0006105342,0.0031697521,0,0.0006820546,0,0,0,0,0.0003152128,0,0.0000974486,0.0034988019],[0.0007074981,0,0,0.0000331898,0.0007928639,0.0000827037,0,0.0000479224,0,0,0.0005521459,0],[0.0012801605,0.002101003,0.001655153,0,0.0039931846,0.0000936331,0.0005702164,0,0,0,0.0001320908,0.0002045558],[0.0006528788,0.0023440115,0.0207901228,0.0000392084,0,0,0.0016335472,0,0.000273292,0,0.0006238979,0]],[[0.0049460525,0.0003060083,0.0000694577,0.0012320397,0.0001012798,0.0009324358,0,0,0,0.0033918433,0.0008259492,0.0007551547],[0.0001368778,0.0073572427,0.0028003791,0.0003991536,0,0,0.0014311032,0.0010387583,0.0026571384,0.0016769671,0.0012928488,0],[0,0,0.0033779786,0,0.0007134717,0,0.0010672365,0.0003530899,0.0020418678,0.0000324257,0.004245115,0.0049314035],[0,0.0014214989,0.0008970152,0,0,0,0.0027742359,0.0009714146,0.0008011838,0.0106930638,0,0.0019925677],[0,0.001904637,0,0,0.000047866,0.000472722,0,0.008365614,0.0013350348,0,0.0112490442,0.0006900692],[0,0.0009869074,0,0,0.0057923876,0,0.0050504361,0.0075390938,0,0.0008451084,0,0],[0.0006105342,0.0012393209,0,0.001515093,0,0,0,0,0.0003152128,0,0.0000974486,0.0034988019],[0.0007074981,0,0,0.0001327591,0.0007928639,0.0000827037,0,0.0000479224,0,0,0.0022085836,0],[0.0012801605,0.0005252508,0.0004137882,0,0.0097372192,0.0003745325,0.0005702164,0,0,0,0.0001320908,0.0002045558],[0.0006528788,0.0005860029,0.0144897477,0.0000179157,0,0,0.0016335472,0,0.0010931681,0,0.0006238979,0]],[[0.0049460525,0.0002564904,0.0000694577,0.0012320397,0.0004051192,0.0037297433,0,0,0,0.0033918433,0.0008259492,0.0007551547],[0.0001368778,0.0018393107,0.0028003791,0.0015966145,0,0,0.0057244129,0.0002596896,0.0026571384,0.0028574679,0.0012928488,0],[0,0,0.0065545347,0,0.0007134717,0,0.0034026922,0.0000882725,0.0020418678,0.0000324257,0.0010612787,0.019725614],[0,0.0014214989,0.0017882355,0,0,0,0.0014570211,0.0002428537,0.0008011838,0.0027641517,0,0.0019925677],[0,0.001904637,0,0,0.000047866,0.0018908881,0,0.0125154953,0.0003337587,0,0.0088230021,0.0006900692],[0,0.0009869074,0,0,0.0014480969,0,0.0013800118,0.0019652032,0,0.0025113313,0,0],[0.0006105342,0.0012393209,0,0.0018951802,0,0,0,0,0.0003152128,0,0.0000974486,0.0084994218],[0.0028299924,0,0,0.0000331898,0.0007928639,0.000330815,0,0.0000479224,0,0,0.0005521459,0],[0.0051206421,0.0005252508,0.0004137882,0,0.0039931846,0.0000936331,0.0005702164,0,0,0,0.0001320908,0.000818223],[0.0026115151,0.0005860029,0.0070559741,0.0000179157,0,0,0.0016335472,0,0.000273292,0,0.0006238979,0]],[[0.0049460525,0.0002564904,0.0000694577,0.004928159,0.0001012798,0.0009324358,0,0,0,0.0135673732,0.0008259492,0.0007551547],[0.000547511,0.0018393107,0.0028003791,0.0003991536,0,0,0.0014311032,0.0002596896,0.0026571384,0.0016769671,0.0051713951,0],[0,0,0.0033779786,0,0.0028538868,0,0.0010672365,0.0000882725,0.0020418678,0.0000324257,0.0010612787,0.0049314035],[0,0.0014214989,0.0008970152,0,0,0,0.0023741237,0.0002428537,0.0008011838,0.0027641517,0,0.0019925677],[0,0.001904637,0,0,0.000047866,0.000472722,0,0.0099735029,0.0003337587,0,0.0075931845,0.0006900692],[0,0.0009869074,0,0,0.0014480969,0,0.0013800118,0.0019652032,0,0.0017142109,0,0],[0.0006105342,0.0030268524,0,0.0006820546,0,0,0,0,0.0003152128,0,0.0003897943,0.0045440851],[0.0007074981,0,0,0.0000331898,0.0007928639,0.0000827037,0,0.0000479224,0,0,0.0005521459,0],[0.0012801605,0.0005252508,0.0004137882,0,0.0039931846,0.0000936331,0.0005702164,0,0,0,0.0001320908,0.0002045558],[0.0006528788,0.0005860029,0.0070559741,0.0000929556,0,0,0.0034048875,0,0.000273292,0,0.0007659722,0]],[[0.0049460525,0.0009764437,0.0002167089,0.0012320397,0.0001012798,0.0009324358,0,0,0,0.0033918433,0.0008259492,0.0007551547],[0.0001368778,0.0018393107,0.0028003791,0.0003991536,0,0,0.0014311032,0.0002596896,0.0106285538,0.0016872081,0.0012928488,0],[0,0,0.0033779786,0,0.0007134717,0,0.0019334903,0.0000882725,0.0081674714,0.0000324257,0.0010612787,0.0049314035],[0,0.0014214989,0.0008970152,0,0,0,0.005427972,0.0002428537,0.0032047354,0.0027641517,0,0.0076481697],[0,0.001904637,0,0,0.0001914641,0.000472722,0,0.0041376099,0.0003337587,0,0.0033458785,0.0027602768],[0,0.0009869074,0,0,0.0014480969,0,0.0091904717,0.0036251285,0,0.0008451084,0,0],[0.0024421368,0.0012393209,0,0.0006820546,0,0,0,0,0.001260851,0,0.0000974486,0.0079493047],[0.0007074981,0,0,0.0000331898,0.0031714555,0.0000827037,0,0.0001916897,0,0,0.0005521459,0],[0.0012801605,0.0005252508,0.0004137882,0,0.0102287036,0.0000936331,0.0022808656,0,0,0,0.0005283632,0.0002045558],[0.0006528788,0.0005860029,0.0070559741,0.0000179157,0,0,0.0047628483,0,0.000273292,0,0.0023535175,0]]],\"nonzero\":[[[5,0,1,0,0,0,0,0,0,0,1,2],[0,0,1,0,0,0,0,0,0,3,0,0],[0,0,3,0,0,0,0,0,0,1,0,0],[0,2,1,0,0,0,0,0,0,1,0,1],[0,1,0,0,0,0,0,0,0,0,0,0],[0,1,0,0,0,0,0,0,0,0,0,0],[0,1,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,1,1,0,0,0,0,0,0,0,0,0],[0,1,2,1,0,0,0,0,0,0,0,0]],[[0,1,0,0,0,0,0,0,0,0,0,0],[0,1,0,0,0,0,0,1,0,0,0,0],[0,0,0,0,0,0,0,1,0,0,2,0],[0,0,0,0,0,0,1,1,0,2,0,0],[0,0,0,0,0,0,0,1,1,0,1,0],[0,0,0,0,1,0,1,1,0,0,0,0],[0,0,0,1,0,0,0,0,0,0,0,0],[0,0,0,1,0,0,0,0,0,0,1,0],[0,0,0,0,2,1,0,0,0,0,0,0],[0,0,1,0,0,0,0,0,1,0,0,0]],[[0,0,0,0,1,2,0,0,0,0,0,0],[0,0,0,1,0,0,1,0,0,1,0,0],[0,0,1,0,0,0,1,0,0,0,0,3],[0,0,1,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,1,0,3,0,0,2,0],[0,0,0,0,0,0,0,0,0,1,0,0],[0,0,0,1,0,0,0,0,0,0,0,2],[2,0,0,0,0,1,0,0,0,0,0,0],[1,0,0,0,0,0,0,0,0,0,0,1],[2,0,0,0,0,0,0,0,0,0,0,0]],[[0,0,0,1,0,0,0,0,0,2,0,0],[1,0,0,0,0,0,0,0,0,0,1,0],[0,0,0,0,1,0,0,0,0,0,0,0],[0,0,0,0,0,0,1,0,0,0,0,0],[0,0,0,0,0,0,0,2,0,0,1,0],[0,0,0,0,0,0,0,0,0,1,0,0],[0,1,0,0,0,0,0,0,0,0,1,1],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,1,0,0,1,0,0,0,1,0]],[[0,1,1,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,4,1,0,0],[0,0,0,0,0,0,1,0,2,0,0,0],[0,0,0,0,0,0,1,0,1,0,0,1],[0,0,0,0,1,0,0,0,0,0,0,1],[0,0,0,0,0,0,2,2,0,0,0,0],[1,0,0,0,0,0,0,0,1,0,0,1],[0,0,0,0,2,0,0,1,0,0,0,0],[0,0,0,0,1,0,1,0,0,0,1,0],[0,0,0,0,0,0,1,0,0,0,1,0]]]}],\"latents\":[{\"rat\":\"Mitt\",\"target\":\"odor\",\"n_val\":42,\"explained\":[0.21514347195625305,0.18994338810443878],\"groups\":[{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":1,\"n\":14,\"xy\":[0.33056047558784485,-0.3631570041179657]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":2,\"n\":10,\"xy\":[0.04216857627034187,0.0347006656229496]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":3,\"n\":7,\"xy\":[0.03465383127331734,0.15776102244853973]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":4,\"n\":7,\"xy\":[-0.21091605722904205,-0.08408989012241364]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":5,\"n\":4,\"xy\":[-0.0953269973397255,-0.2622129023075104]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":1,\"n\":14,\"xy\":[0.33056047558784485,-0.3631570041179657]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":2,\"n\":10,\"xy\":[0.04216857627034187,0.0347006656229496]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":3,\"n\":7,\"xy\":[0.03465383127331734,0.15776102244853973]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":4,\"n\":7,\"xy\":[-0.21091605722904205,-0.08408989012241364]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":5,\"n\":4,\"xy\":[-0.0953269973397255,-0.2622129023075104]}]},{\"rat\":\"Mitt\",\"target\":\"position\",\"n_val\":45,\"explained\":[0.23115941882133484,0.1938304752111435],\"groups\":[{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":1,\"n\":14,\"xy\":[0.40387478470802307,0.3365750312805176]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":2,\"n\":11,\"xy\":[0.03752443566918373,0.02022814191877842]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":3,\"n\":9,\"xy\":[-0.17057162523269653,-0.10047897696495056]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":4,\"n\":7,\"xy\":[-0.195915088057518,0.2309826761484146]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":5,\"n\":4,\"xy\":[-0.16417887806892395,0.33589908480644226]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":1,\"n\":14,\"xy\":[0.40387478470802307,0.3365750312805176]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":2,\"n\":11,\"xy\":[0.012395411729812622,0.029389314353466034]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":3,\"n\":8,\"xy\":[-0.06693392992019653,-0.1256960928440094]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":4,\"n\":7,\"xy\":[-0.195915088057518,0.2309826761484146]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":5,\"n\":5,\"xy\":[-0.27599385380744934,0.2688162922859192]},{\"cohort\":\"outseq\",\"label\":\"odor\",\"cls\":2,\"n\":1,\"xy\":[0.056348782032728195,-0.19274351000785828]},{\"cohort\":\"outseq\",\"label\":\"odor\",\"cls\":3,\"n\":2,\"xy\":[-0.4716621935367584,-0.045742787420749664]},{\"cohort\":\"outseq\",\"label\":\"position\",\"cls\":2,\"n\":1,\"xy\":[-0.22007052600383759,-0.09197065234184265]},{\"cohort\":\"outseq\",\"label\":\"position\",\"cls\":3,\"n\":1,\"xy\":[0.056348782032728195,-0.19274351000785828]},{\"cohort\":\"outseq\",\"label\":\"position\",\"cls\":5,\"n\":1,\"xy\":[-0.7232537865638733,0.0004850802943110466]}]},{\"rat\":\"Barat\",\"target\":\"odor\",\"n_val\":23,\"explained\":[0.20731572806835175,0.17832008004188538],\"groups\":[{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":1,\"n\":6,\"xy\":[-0.6199356913566589,0.15993747115135193]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":2,\"n\":7,\"xy\":[-0.2247464507818222,0.028744373470544815]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":3,\"n\":3,\"xy\":[-0.04307541251182556,0.2582007646560669]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":4,\"n\":4,\"xy\":[-0.0677151083946228,0.0241121593862772]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":5,\"n\":3,\"xy\":[-0.09398812800645828,0.23413731157779694]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":1,\"n\":6,\"xy\":[-0.6199356913566589,0.15993747115135193]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":2,\"n\":7,\"xy\":[-0.2247464507818222,0.028744373470544815]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":3,\"n\":3,\"xy\":[-0.04307541251182556,0.2582007646560669]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":4,\"n\":4,\"xy\":[-0.0677151083946228,0.0241121593862772]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":5,\"n\":3,\"xy\":[-0.09398812800645828,0.23413731157779694]}]},{\"rat\":\"Barat\",\"target\":\"position\",\"n_val\":24,\"explained\":[0.21351923048496246,0.18686018884181976],\"groups\":[{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":1,\"n\":7,\"xy\":[-0.5088875889778137,-0.10605573654174805]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":2,\"n\":7,\"xy\":[-0.259915828704834,-0.06684981286525726]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":3,\"n\":3,\"xy\":[0.013912100344896317,0.32898208498954773]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":4,\"n\":4,\"xy\":[-0.09534640610218048,0.19404718279838562]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":5,\"n\":3,\"xy\":[-0.14158965647220612,0.21244709193706512]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":1,\"n\":6,\"xy\":[-0.675508975982666,-0.09884003549814224]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":2,\"n\":7,\"xy\":[-0.259915828704834,-0.06684981286525726]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":3,\"n\":3,\"xy\":[0.013912100344896317,0.32898208498954773]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":4,\"n\":4,\"xy\":[-0.09534640610218048,0.19404718279838562]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":5,\"n\":4,\"xy\":[0.016517922282218933,0.12199783325195312]},{\"cohort\":\"outseq\",\"label\":\"odor\",\"cls\":1,\"n\":1,\"xy\":[0.4908406138420105,-0.14934991300106049]},{\"cohort\":\"outseq\",\"label\":\"position\",\"cls\":5,\"n\":1,\"xy\":[0.4908406138420105,-0.14934991300106049]}]},{\"rat\":\"Stella\",\"target\":\"odor\",\"n_val\":27,\"explained\":[0.2279348522424698,0.19818922877311707],\"groups\":[{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":1,\"n\":7,\"xy\":[-0.3401648700237274,-0.06269024312496185]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":2,\"n\":5,\"xy\":[0.06252145767211914,-0.13261722028255463]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":3,\"n\":6,\"xy\":[0.13122962415218353,0.2035752385854721]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":4,\"n\":6,\"xy\":[0.27014610171318054,0.11515653133392334]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":5,\"n\":3,\"xy\":[0.2146928906440735,0.0017556846141815186]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":1,\"n\":7,\"xy\":[-0.3401648700237274,-0.06269024312496185]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":2,\"n\":5,\"xy\":[0.06252145767211914,-0.13261722028255463]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":3,\"n\":6,\"xy\":[0.13122962415218353,0.2035752385854721]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":4,\"n\":6,\"xy\":[0.27014610171318054,0.11515653133392334]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":5,\"n\":3,\"xy\":[0.2146928906440735,0.0017556846141815186]}]},{\"rat\":\"Stella\",\"target\":\"position\",\"n_val\":30,\"explained\":[0.22544769942760468,0.19892872869968414],\"groups\":[{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":1,\"n\":8,\"xy\":[-0.24979832768440247,-0.08353789895772934]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":2,\"n\":6,\"xy\":[0.05764368176460266,-0.14836524426937103]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":3,\"n\":6,\"xy\":[0.16000694036483765,-0.05441102758049965]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":4,\"n\":6,\"xy\":[0.291663259267807,-0.005800724029541016]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":5,\"n\":4,\"xy\":[0.17716889083385468,-0.05443540960550308]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":1,\"n\":7,\"xy\":[-0.3079190254211426,-0.10785957425832748]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":2,\"n\":6,\"xy\":[0.030168160796165466,0.023731550201773643]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":3,\"n\":6,\"xy\":[0.16000694036483765,-0.05441102758049965]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":4,\"n\":6,\"xy\":[0.291663259267807,-0.005800724029541016]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":5,\"n\":5,\"xy\":[0.20611508190631866,-0.232721745967865]},{\"cohort\":\"outseq\",\"label\":\"odor\",\"cls\":1,\"n\":1,\"xy\":[0.15704664587974548,0.08671372383832932]},{\"cohort\":\"outseq\",\"label\":\"odor\",\"cls\":2,\"n\":1,\"xy\":[0.23305228352546692,-0.6307251453399658]},{\"cohort\":\"outseq\",\"label\":\"odor\",\"cls\":5,\"n\":1,\"xy\":[0.06819913536310196,0.40185561776161194]},{\"cohort\":\"outseq\",\"label\":\"position\",\"cls\":2,\"n\":1,\"xy\":[0.06819913536310196,0.40185561776161194]},{\"cohort\":\"outseq\",\"label\":\"position\",\"cls\":5,\"n\":2,\"xy\":[0.1950494647026062,-0.27200570702552795]}]},{\"rat\":\"Superchris\",\"target\":\"odor\",\"n_val\":28,\"explained\":[0.23345643281936646,0.1773988902568817],\"groups\":[{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":1,\"n\":8,\"xy\":[-0.5642228722572327,0.10784019529819489]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":2,\"n\":6,\"xy\":[0.18925131857395172,0.05067954957485199]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":3,\"n\":6,\"xy\":[0.40710386633872986,0.13997501134872437]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":4,\"n\":4,\"xy\":[0.24919503927230835,0.28043073415756226]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":5,\"n\":4,\"xy\":[-0.16152924299240112,0.12505251169204712]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":1,\"n\":8,\"xy\":[-0.5642228722572327,0.10784019529819489]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":2,\"n\":6,\"xy\":[0.18925131857395172,0.05067954957485199]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":3,\"n\":6,\"xy\":[0.40710386633872986,0.13997501134872437]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":4,\"n\":4,\"xy\":[0.24919503927230835,0.28043073415756226]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":5,\"n\":4,\"xy\":[-0.16152924299240112,0.12505251169204712]}]},{\"rat\":\"Superchris\",\"target\":\"position\",\"n_val\":29,\"explained\":[0.23332813382148743,0.16828767955303192],\"groups\":[{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":1,\"n\":8,\"xy\":[-0.593038022518158,0.10517475008964539]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":2,\"n\":6,\"xy\":[0.13863570988178253,0.08087798207998276]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":3,\"n\":6,\"xy\":[0.3696920573711395,-0.10590964555740356]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":4,\"n\":5,\"xy\":[0.13511136174201965,-0.15559962391853333]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":5,\"n\":4,\"xy\":[-0.17716962099075317,0.03437243774533272]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":1,\"n\":8,\"xy\":[-0.593038022518158,0.10517475008964539]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":2,\"n\":7,\"xy\":[0.08510692417621613,0.10874767601490021]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":3,\"n\":6,\"xy\":[0.3696920573711395,-0.10590964555740356]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":4,\"n\":4,\"xy\":[0.22790566086769104,-0.2634909749031067]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":5,\"n\":4,\"xy\":[-0.17716962099075317,0.03437243774533272]},{\"cohort\":\"outseq\",\"label\":\"odor\",\"cls\":4,\"n\":1,\"xy\":[-0.2360657900571823,0.27596578001976013]},{\"cohort\":\"outseq\",\"label\":\"position\",\"cls\":2,\"n\":1,\"xy\":[-0.2360657900571823,0.27596578001976013]}]},{\"rat\":\"Buchanan\",\"target\":\"odor\",\"n_val\":17,\"explained\":[0.21705973148345947,0.17603173851966858],\"groups\":[{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":1,\"n\":4,\"xy\":[-0.4494326710700989,0.06942704319953918]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":2,\"n\":4,\"xy\":[-0.20084628462791443,-0.07142238318920135]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":3,\"n\":2,\"xy\":[0.130142942070961,-0.32149171829223633]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":4,\"n\":5,\"xy\":[0.041775528341531754,0.03819284588098526]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":5,\"n\":2,\"xy\":[-0.041757240891456604,0.12745150923728943]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":1,\"n\":4,\"xy\":[-0.4494326710700989,0.06942704319953918]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":2,\"n\":4,\"xy\":[-0.20084628462791443,-0.07142238318920135]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":3,\"n\":2,\"xy\":[0.130142942070961,-0.32149171829223633]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":4,\"n\":5,\"xy\":[0.041775528341531754,0.03819284588098526]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":5,\"n\":2,\"xy\":[-0.041757240891456604,0.12745150923728943]}]},{\"rat\":\"Buchanan\",\"target\":\"position\",\"n_val\":21,\"explained\":[0.2387082427740097,0.18300479650497437],\"groups\":[{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":1,\"n\":4,\"xy\":[-0.4731504023075104,-0.2764723002910614]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":2,\"n\":6,\"xy\":[-0.068229079246521,-0.015567251481115818]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":3,\"n\":3,\"xy\":[0.20939071476459503,0.10413260012865067]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":4,\"n\":5,\"xy\":[-0.009494880214333534,-0.07436023652553558]},{\"cohort\":\"all\",\"label\":\"odor\",\"cls\":5,\"n\":3,\"xy\":[-0.11347562074661255,0.022089332342147827]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":1,\"n\":4,\"xy\":[-0.4731504023075104,-0.2764723002910614]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":2,\"n\":5,\"xy\":[-0.3111259341239929,-0.03616871312260628]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":3,\"n\":3,\"xy\":[0.17042286694049835,0.1962825506925583]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":4,\"n\":5,\"xy\":[-0.009494880214333534,-0.07436023652553558]},{\"cohort\":\"all\",\"label\":\"position\",\"cls\":5,\"n\":4,\"xy\":[0.23068295419216156,-0.030685443431138992]},{\"cohort\":\"outseq\",\"label\":\"odor\",\"cls\":2,\"n\":2,\"xy\":[0.37500402331352234,0.0468803234398365]},{\"cohort\":\"outseq\",\"label\":\"odor\",\"cls\":3,\"n\":1,\"xy\":[0.49879950284957886,-0.3442377746105194]},{\"cohort\":\"outseq\",\"label\":\"odor\",\"cls\":5,\"n\":1,\"xy\":[-0.39624711871147156,0.006320592015981674]},{\"cohort\":\"outseq\",\"label\":\"position\",\"cls\":2,\"n\":1,\"xy\":[-0.39624711871147156,0.006320592015981674]},{\"cohort\":\"outseq\",\"label\":\"position\",\"cls\":3,\"n\":1,\"xy\":[0.3818959593772888,-0.06778793781995773]},{\"cohort\":\"outseq\",\"label\":\"position\",\"cls\":5,\"n\":2,\"xy\":[0.43345582485198975,-0.09134459495544434]}]}],\"masking\":[{\"target\":\"odor\",\"session\":\"Mitt\",\"group\":\"freq_0_4\",\"intact_ba\":0.5599999999999999,\"masked_ba\":0.31999999999999995,\"delta\":0.24,\"n_classes\":5,\"n\":42},{\"target\":\"odor\",\"session\":\"Mitt\",\"group\":\"freq_4_8\",\"intact_ba\":0.5599999999999999,\"masked_ba\":0.32571428571428573,\"delta\":0.2342857142857142,\"n_classes\":5,\"n\":42},{\"target\":\"odor\",\"session\":\"Mitt\",\"group\":\"freq_8_12\",\"intact_ba\":0.5599999999999999,\"masked_ba\":0.46571428571428575,\"delta\":0.0942857142857142,\"n_classes\":5,\"n\":42},{\"target\":\"odor\",\"session\":\"Mitt\",\"group\":\"time_0_5\",\"intact_ba\":0.5599999999999999,\"masked_ba\":0.3271428571428571,\"delta\":0.23285714285714282,\"n_classes\":5,\"n\":42},{\"target\":\"odor\",\"session\":\"Mitt\",\"group\":\"time_5_10\",\"intact_ba\":0.5599999999999999,\"masked_ba\":0.4442857142857143,\"delta\":0.11571428571428566,\"n_classes\":5,\"n\":42},{\"target\":\"position\",\"session\":\"Mitt\",\"group\":\"freq_0_4\",\"intact_ba\":0.4545454545454545,\"masked_ba\":0.25032467532467534,\"delta\":0.20422077922077914,\"n_classes\":5,\"n\":45},{\"target\":\"position\",\"session\":\"Mitt\",\"group\":\"freq_4_8\",\"intact_ba\":0.4545454545454545,\"masked_ba\":0.3616883116883117,\"delta\":0.09285714285714275,\"n_classes\":5,\"n\":45},{\"target\":\"position\",\"session\":\"Mitt\",\"group\":\"freq_8_12\",\"intact_ba\":0.4545454545454545,\"masked_ba\":0.3899350649350649,\"delta\":0.06461038961038956,\"n_classes\":5,\"n\":45},{\"target\":\"position\",\"session\":\"Mitt\",\"group\":\"time_0_5\",\"intact_ba\":0.4545454545454545,\"masked_ba\":0.30103896103896105,\"delta\":0.15350649350649342,\"n_classes\":5,\"n\":45},{\"target\":\"position\",\"session\":\"Mitt\",\"group\":\"time_5_10\",\"intact_ba\":0.4545454545454545,\"masked_ba\":0.362012987012987,\"delta\":0.09253246753246747,\"n_classes\":5,\"n\":45},{\"target\":\"odor\",\"session\":\"Barat\",\"group\":\"freq_0_4\",\"intact_ba\":0.45714285714285713,\"masked_ba\":0.35238095238095235,\"delta\":0.10476190476190478,\"n_classes\":5,\"n\":23},{\"target\":\"odor\",\"session\":\"Barat\",\"group\":\"freq_4_8\",\"intact_ba\":0.45714285714285713,\"masked_ba\":0.4619047619047619,\"delta\":-0.004761904761904745,\"n_classes\":5,\"n\":23},{\"target\":\"odor\",\"session\":\"Barat\",\"group\":\"freq_8_12\",\"intact_ba\":0.45714285714285713,\"masked_ba\":0.4904761904761905,\"delta\":-0.03333333333333338,\"n_classes\":5,\"n\":23},{\"target\":\"odor\",\"session\":\"Barat\",\"group\":\"time_0_5\",\"intact_ba\":0.45714285714285713,\"masked_ba\":0.3476190476190476,\"delta\":0.10952380952380952,\"n_classes\":5,\"n\":23},{\"target\":\"odor\",\"session\":\"Barat\",\"group\":\"time_5_10\",\"intact_ba\":0.45714285714285713,\"masked_ba\":0.4738095238095238,\"delta\":-0.016666666666666663,\"n_classes\":5,\"n\":23},{\"target\":\"position\",\"session\":\"Barat\",\"group\":\"freq_0_4\",\"intact_ba\":0.6190476190476191,\"masked_ba\":0.42142857142857143,\"delta\":0.19761904761904764,\"n_classes\":5,\"n\":24},{\"target\":\"position\",\"session\":\"Barat\",\"group\":\"freq_4_8\",\"intact_ba\":0.6190476190476191,\"masked_ba\":0.48571428571428565,\"delta\":0.13333333333333341,\"n_classes\":5,\"n\":24},{\"target\":\"position\",\"session\":\"Barat\",\"group\":\"freq_8_12\",\"intact_ba\":0.6190476190476191,\"masked_ba\":0.4523809523809524,\"delta\":0.16666666666666669,\"n_classes\":5,\"n\":24},{\"target\":\"position\",\"session\":\"Barat\",\"group\":\"time_0_5\",\"intact_ba\":0.6190476190476191,\"masked_ba\":0.530952380952381,\"delta\":0.08809523809523812,\"n_classes\":5,\"n\":24},{\"target\":\"position\",\"session\":\"Barat\",\"group\":\"time_5_10\",\"intact_ba\":0.6190476190476191,\"masked_ba\":0.3738095238095238,\"delta\":0.24523809523809526,\"n_classes\":5,\"n\":24},{\"target\":\"odor\",\"session\":\"Stella\",\"group\":\"freq_0_4\",\"intact_ba\":0.6247619047619047,\"masked_ba\":0.5866666666666667,\"delta\":0.03809523809523807,\"n_classes\":5,\"n\":27},{\"target\":\"odor\",\"session\":\"Stella\",\"group\":\"freq_4_8\",\"intact_ba\":0.6247619047619047,\"masked_ba\":0.45809523809523806,\"delta\":0.16666666666666669,\"n_classes\":5,\"n\":27},{\"target\":\"odor\",\"session\":\"Stella\",\"group\":\"freq_8_12\",\"intact_ba\":0.6247619047619047,\"masked_ba\":0.5847619047619047,\"delta\":0.040000000000000036,\"n_classes\":5,\"n\":27},{\"target\":\"odor\",\"session\":\"Stella\",\"group\":\"time_0_5\",\"intact_ba\":0.6247619047619047,\"masked_ba\":0.5247619047619048,\"delta\":0.09999999999999998,\"n_classes\":5,\"n\":27},{\"target\":\"odor\",\"session\":\"Stella\",\"group\":\"time_5_10\",\"intact_ba\":0.6247619047619047,\"masked_ba\":0.4914285714285714,\"delta\":0.13333333333333336,\"n_classes\":5,\"n\":27},{\"target\":\"position\",\"session\":\"Stella\",\"group\":\"freq_0_4\",\"intact_ba\":0.5847619047619047,\"masked_ba\":0.58,\"delta\":0.004761904761904745,\"n_classes\":5,\"n\":30},{\"target\":\"position\",\"session\":\"Stella\",\"group\":\"freq_4_8\",\"intact_ba\":0.5847619047619047,\"masked_ba\":0.4780952380952381,\"delta\":0.10666666666666663,\"n_classes\":5,\"n\":30},{\"target\":\"position\",\"session\":\"Stella\",\"group\":\"freq_8_12\",\"intact_ba\":0.5847619047619047,\"masked_ba\":0.5514285714285714,\"delta\":0.033333333333333326,\"n_classes\":5,\"n\":30},{\"target\":\"position\",\"session\":\"Stella\",\"group\":\"time_0_5\",\"intact_ba\":0.5847619047619047,\"masked_ba\":0.58,\"delta\":0.004761904761904745,\"n_classes\":5,\"n\":30},{\"target\":\"position\",\"session\":\"Stella\",\"group\":\"time_5_10\",\"intact_ba\":0.5847619047619047,\"masked_ba\":0.5114285714285713,\"delta\":0.07333333333333336,\"n_classes\":5,\"n\":30},{\"target\":\"odor\",\"session\":\"Superchris\",\"group\":\"freq_0_4\",\"intact_ba\":0.38333333333333336,\"masked_ba\":0.15,\"delta\":0.23333333333333336,\"n_classes\":5,\"n\":28},{\"target\":\"odor\",\"session\":\"Superchris\",\"group\":\"freq_4_8\",\"intact_ba\":0.38333333333333336,\"masked_ba\":0.33333333333333337,\"delta\":0.04999999999999999,\"n_classes\":5,\"n\":28},{\"target\":\"odor\",\"session\":\"Superchris\",\"group\":\"freq_8_12\",\"intact_ba\":0.38333333333333336,\"masked_ba\":0.5333333333333333,\"delta\":-0.14999999999999997,\"n_classes\":5,\"n\":28},{\"target\":\"odor\",\"session\":\"Superchris\",\"group\":\"time_0_5\",\"intact_ba\":0.38333333333333336,\"masked_ba\":0.29166666666666663,\"delta\":0.09166666666666673,\"n_classes\":5,\"n\":28},{\"target\":\"odor\",\"session\":\"Superchris\",\"group\":\"time_5_10\",\"intact_ba\":0.38333333333333336,\"masked_ba\":0.3416666666666667,\"delta\":0.041666666666666685,\"n_classes\":5,\"n\":28},{\"target\":\"position\",\"session\":\"Superchris\",\"group\":\"freq_0_4\",\"intact_ba\":0.4309523809523809,\"masked_ba\":0.16904761904761906,\"delta\":0.26190476190476186,\"n_classes\":5,\"n\":29},{\"target\":\"position\",\"session\":\"Superchris\",\"group\":\"freq_4_8\",\"intact_ba\":0.4309523809523809,\"masked_ba\":0.4023809523809524,\"delta\":0.028571428571428525,\"n_classes\":5,\"n\":29},{\"target\":\"position\",\"session\":\"Superchris\",\"group\":\"freq_8_12\",\"intact_ba\":0.4309523809523809,\"masked_ba\":0.5476190476190477,\"delta\":-0.11666666666666675,\"n_classes\":5,\"n\":29},{\"target\":\"position\",\"session\":\"Superchris\",\"group\":\"time_0_5\",\"intact_ba\":0.4309523809523809,\"masked_ba\":0.294047619047619,\"delta\":0.13690476190476192,\"n_classes\":5,\"n\":29},{\"target\":\"position\",\"session\":\"Superchris\",\"group\":\"time_5_10\",\"intact_ba\":0.4309523809523809,\"masked_ba\":0.3,\"delta\":0.13095238095238093,\"n_classes\":5,\"n\":29},{\"target\":\"odor\",\"session\":\"Buchanan\",\"group\":\"freq_0_4\",\"intact_ba\":0.32999999999999996,\"masked_ba\":0.24,\"delta\":0.08999999999999997,\"n_classes\":5,\"n\":17},{\"target\":\"odor\",\"session\":\"Buchanan\",\"group\":\"freq_4_8\",\"intact_ba\":0.32999999999999996,\"masked_ba\":0.35,\"delta\":-0.020000000000000018,\"n_classes\":5,\"n\":17},{\"target\":\"odor\",\"session\":\"Buchanan\",\"group\":\"freq_8_12\",\"intact_ba\":0.32999999999999996,\"masked_ba\":0.36,\"delta\":-0.030000000000000027,\"n_classes\":5,\"n\":17},{\"target\":\"odor\",\"session\":\"Buchanan\",\"group\":\"time_0_5\",\"intact_ba\":0.32999999999999996,\"masked_ba\":0.37,\"delta\":-0.040000000000000036,\"n_classes\":5,\"n\":17},{\"target\":\"odor\",\"session\":\"Buchanan\",\"group\":\"time_5_10\",\"intact_ba\":0.32999999999999996,\"masked_ba\":0.24,\"delta\":0.08999999999999997,\"n_classes\":5,\"n\":17},{\"target\":\"position\",\"session\":\"Buchanan\",\"group\":\"freq_0_4\",\"intact_ba\":0.45,\"masked_ba\":0.29000000000000004,\"delta\":0.15999999999999998,\"n_classes\":5,\"n\":21},{\"target\":\"position\",\"session\":\"Buchanan\",\"group\":\"freq_4_8\",\"intact_ba\":0.45,\"masked_ba\":0.45,\"delta\":0,\"n_classes\":5,\"n\":21},{\"target\":\"position\",\"session\":\"Buchanan\",\"group\":\"freq_8_12\",\"intact_ba\":0.45,\"masked_ba\":0.44000000000000006,\"delta\":0.009999999999999953,\"n_classes\":5,\"n\":21},{\"target\":\"position\",\"session\":\"Buchanan\",\"group\":\"time_0_5\",\"intact_ba\":0.45,\"masked_ba\":0.36666666666666664,\"delta\":0.08333333333333337,\"n_classes\":5,\"n\":21},{\"target\":\"position\",\"session\":\"Buchanan\",\"group\":\"time_5_10\",\"intact_ba\":0.45,\"masked_ba\":0.37,\"delta\":0.08000000000000002,\"n_classes\":5,\"n\":21}]}}")
session_scores = csv_frame('session_scores.csv')
headline = csv_frame('headline.csv')
outseq = csv_frame('outseq.csv')
interpretation = SNAPSHOT['interpretation']

## 01 · Question, cohort, and study design

**Poster copy.** Can learned features improve odor and sequence position decoding from hippocampal local field potentials? We compared two inputs to sparse logistic regression on matched 500 ms windows after odor onset. Models were fit separately for each of five rats.

- **Odor (A–E):** correct in-sequence trials. Odor and its usual position coincide in this cohort.
- **Sequence position (1–5):** correct in-sequence and out-of-sequence trials. Position refers to order, not physical location.
- **Signal preparation:** align recordings and behavior, split chronologically within each rat (70% training, 15% validation, 15% test), and reject invalid and boundary-crossing windows. Apply a narrow 60 Hz notch and compute wavelet features without crossing split boundaries.
- **Direct wavelet amplitude:** log amplitude in ten 50 ms bins. **Wavelet CEBRA-Time:** learn an eight-dimensional representation of amplitude and phase using nearby time samples, then average valid embeddings inside the window.

**Diagram caption.** Both inputs use the same eligible trials within a rat and target. The model heads are fit on training trials. Validation guides selection; previously viewed test results are descriptive.

In [ ]:
fig = go.Figure()
fig.update_xaxes(range=[0,12], visible=False)
fig.update_yaxes(range=[0,4], visible=False)

def box(x,y,w,label,face):
    fig.add_shape(type='rect',x0=x-w/2,x1=x+w/2,y0=y-.5,y1=y+.5,
                  line=dict(color=GRID,width=2),fillcolor=face,layer='below')
    fig.add_annotation(x=x,y=y,text=label,showarrow=False,align='center')

def arrow(x0,y0,x1,y1):
    fig.add_annotation(ax=x0,ay=y0,x=x1,y=y1,axref='x',ayref='y',xref='x',yref='y',
                       text='',showarrow=True,arrowhead=2,arrowcolor=MUTED,arrowwidth=1.5)

box(2,2.1,3.1,'<b>Recordings + behavior</b><br>Match 500 ms windows',WHITE)
box(5.9,3.15,3.5,'<b>Direct wavelet amplitude</b><br>Ten time bins',MINT)
box(5.9,1.05,3.5,'<b>Wavelet amplitude + phase</b><br>CEBRA-Time: eight features',MINT)
box(10,2.1,3.1,'<b>Sparse logistic regression</b><br>Predict odor or position',WHITE)
for y in (3.15,1.05):
    arrow(3.6,2.1,4.08,y); arrow(7.72,y,8.38,2.1)
fig.add_annotation(x=6,y=.2,text='Split by time within each rat · Fit on training · Validate · Report historical test',showarrow=False)
publish(fig,'01-study-design','Study design · Two inputs, one decoder family',height=335,margin=dict(l=12,r=12,t=65,b=15))

## 02 · Decoding performance: test first

**Poster copy.** Across the four rats with all five classes on the previously viewed test, direct amplitude scored 40.0% versus 30.3% on position (9.7 percentage points). Odor was nearly tied (48.9% versus 48.4%). This compares two complete pipelines, not architectures in isolation.

**Figure caption.** Five-class balanced accuracy is mean class recall within a rat, then an equal-rat average. Test excludes Buchanan, which lacks one test class; validation includes all five. The dashed 20% line is uniform-guessing chance for five classes. The historical test is descriptive, not independent confirmation. The separate five-rat test metric averages recall over each rat’s observed classes; do not label that metric as strict five-class balanced accuracy.

In [ ]:
score_keys=['target','model']
test = session_scores.query("split == 'test'").dropna(subset=['ba'])
assert all(set(g.session)==set(RATS)-{'Buchanan'} for _,g in test.groupby(score_keys))
validation = session_scores.query("split == 'val'")
assert all(set(g.session)==set(RATS) and g.ba.notna().all() for _,g in validation.groupby(score_keys))
scores = (pd.concat({'Test · 4 rats':test.groupby(score_keys).ba.mean(),
                     'Validation · 5 rats':validation.groupby(score_keys).ba.mean()},axis=1)*100)
order = pd.MultiIndex.from_product([TARGETS, MODELS], names=score_keys)
scores = scores.reindex(order)
assert scores.notna().all().all()
display(scores.rename(index={**TARGETS,**MODELS}).round(1))
fig = make_subplots(rows=2,cols=2,shared_xaxes=True,column_widths=[.58,.42],
    horizontal_spacing=.10,vertical_spacing=.18,
    subplot_titles=[f'{title} · {split}' for title in TARGETS.values() for split in scores.columns])
for i,t in enumerate(TARGETS,1):
    frame=scores.loc[t]
    for j,split in enumerate(frame,1):
        fig.add_trace(go.Bar(x=frame[split],y=list(MODELS.values()),orientation='h',width=.52,
                      marker_color=MODEL_COLORS,opacity=1 if j==1 else .78,
                      texttemplate='%{x:.1f}%',textposition='outside',cliponaxis=False,
                      hovertemplate='%{y}<br>%{x:.2f}%<extra></extra>'),row=i,col=j)
        fig.add_vline(x=20,line=dict(color=MUTED,dash='dash',width=1),row=i,col=j)
        fig.update_yaxes(autorange='reversed',showgrid=False,showticklabels=j==1,row=i,col=j)
fig.update_xaxes(range=[0,65],tickvals=[0,20,40,60],ticksuffix='%')
fig.update_xaxes(title_text='Balanced accuracy (%)',row=2)
fig.update_layout(showlegend=False)
publish(fig,'02-decoding-performance','Decoding performance · Test first',height=480,
        margin=dict(l=190,r=30,t=88,b=45))
scores.to_csv(OUT/'02-test-and-validation.csv')

# Reference metrics, not a third column for the printed figure.
five_rat_test = headline.query("split == 'test'").set_index(score_keys)['observed_macro_recall']
display((100*five_rat_test.reindex(order)).rename('Test observed-class recall · 5 rats (%)')
        .rename(index={**TARGETS,**MODELS}).round(1).to_frame())

## 03 · Does predicted position follow the actual trial?

**Poster copy.** If odor C normally appears third but is presented fifth, predicting 5 follows actual position; predicting 3 follows usual position. Other predictions follow neither. The validation pattern did not hold on test. With only eight eligible test trials pooled across rats, this does not establish odor-independent position decoding.

**Caption.** Counts include only correct out-of-sequence trials on which the odor’s usual position (learned from training) differs from its actual position. Validation uses the same rule (12 trials). These are trial counts, not balanced accuracy or a causal test.

In [ ]:
pooled = outseq.query("scope == 'pooled_trials'").set_index(['split','model'])
rows = pooled.reindex(pd.MultiIndex.from_product([['test','val'],MODELS],names=['split','model']))
cols = {'follows_actual':'Actual position','follows_usual':'Usual position','neither':'Neither','n':'Trials'}
assert (rows[list(cols)[:3]].sum(axis=1)==rows.n).all()
match_table = rows[list(cols)].rename(columns=cols).rename(index={**MODELS,'test':'Test','val':'Validation'})
display(match_table)
match_table.to_csv(OUT/'03-position-diagnostic.csv')
table_data=match_table.reset_index()
table_data.columns=['Split','Position model input',*cols.values()]
fig=go.Figure(go.Table(
    columnwidth=[.85,2.2,1,1,.65,.55],
    header=dict(values=list(table_data.columns),fill_color=TEAL,font=dict(color='white',size=15),
                align=['left','left','center','center','center','center'],height=38),
    cells=dict(values=[table_data[c].tolist() for c in table_data],height=36,
               fill_color=[[WHITE,WHITE,MINT,MINT]],
               line_color=GRID,align=['left','left','center','center','center','center']),
))
publish(fig,'03-position-diagnostic','Which position did the model predict?',height=265,
        margin=dict(l=22,r=22,t=66,b=12))

## 04 · Validation masking: which feature groups matter to the fitted model?

**Poster copy.** Masking the 4–11 Hz group reduced validation balanced accuracy in every rat for both targets. Average drops were 14.1 points for odor and 16.6 for position.

**Caption.** Replace the selected group with its training mean, then score the same amplitude model without refitting. Change = masked minus intact balanced accuracy, in percentage points. Negative means worse; positive means better. This measures model sensitivity, not causal importance. Masking was not evaluated on test.

In [ ]:
mask = pd.DataFrame(interpretation['masking'])
mask['change'] = 100*(mask.masked_ba-mask.intact_ba)
mask_grid = mask.pivot(index=['target','group'],columns='session',values='change').reindex(columns=RATS)
mask_grid['Mean'] = mask_grid.mean(axis=1)
mask_grid = mask_grid.reindex(pd.MultiIndex.from_product([TARGETS,BANDS],names=['target','group']))
mask_grid.rename(index={**TARGETS,**BANDS}).to_csv(OUT/'04-all-masking-groups.csv')
low_band=mask_grid.xs('freq_0_4',level='group').rename(index=TARGETS).T
display(low_band.round(1))
fig=make_subplots(rows=1,cols=2,subplot_titles=list(low_band),horizontal_spacing=.13)
for j,t in enumerate(low_band,1):
    a=low_band[t]
    fig.add_trace(go.Bar(x=a,y=a.index,orientation='h',width=.56,
                         marker_color=[LOSS]*5+[INK],text=[f'{v:+.1f}' for v in a],texttemplate='%{text}',
                         textposition='outside',cliponaxis=False,
                         hovertemplate='%{y}: %{x:+.2f} points<extra></extra>'),row=1,col=j)
    fig.add_vline(x=0,line_color=MUTED,line_width=1,row=1,col=j)
    fig.update_yaxes(autorange='reversed',showgrid=False,row=1,col=j)
fig.update_xaxes(range=[-33,3],title_text='Change in validation balanced accuracy (points)')
fig.update_layout(showlegend=False)
publish(fig,'04-low-frequency-masking','Masking 4–11 Hz hurt decoding in all five rats',height=400)

fig=make_subplots(rows=1,cols=2,subplot_titles=list(TARGETS.values()),horizontal_spacing=.18)
limit=np.abs(mask_grid.to_numpy()).max()
for j,t in enumerate(TARGETS,1):
    a=mask_grid.loc[t].to_numpy()
    labels=[[f'{v:+.1f}' if abs(v)>=.05 else '0.0' for v in row] for row in a]
    fig.add_trace(go.Heatmap(z=a,x=mask_grid.columns,y=list(BANDS.values()),coloraxis='coloraxis',
                             text=labels,texttemplate='%{text}',textfont=dict(size=13,color=INK),xgap=2,ygap=2,
                             hovertemplate='%{y} · %{x}<br>%{z:+.2f} points<extra></extra>'),row=1,col=j)
fig.update_yaxes(autorange='reversed')
fig.update_xaxes(tickangle=-30)
fig.update_layout(coloraxis=dict(colorscale=SIGN_SCALE,cmin=-limit,cmax=limit,
                                colorbar=dict(title='Change<br>(points)',len=.78)))
publish(fig,'supplement-all-masking-groups','All masks · Negative is worse, positive is better',height=425,
        width=1140,margin=dict(l=100,r=100,t=83,b=90))

## 05 · What did the amplitude model put weight on?

**Poster copy.** The heatmaps show where the fitted amplitude models put large weights. They do not show where brain signals are strongest or where single-feature decoding works best.

**Caption.** Within each rat, coefficients on standardized inputs were centered across classes. Absolute weights were averaged across channels, then classes and rats were weighted equally. Redundant input features can share signal, so interpret these maps together with masking. Change the target and class in the diagnostic cell to see signed weights, magnitudes, and sparsity for every rat.

In [ ]:
weights=interpretation['weights']
frequencies=np.geomspace(4,150,12)
maps={t:np.mean([r['magnitude'] for r in weights if r['target']==t],axis=(0,1)).T for t in TARGETS}
limit=max(a.max() for a in maps.values())
times=np.arange(25,500,50)
frequency_labels=[f'{f:.1f}' for f in frequencies]
fig=make_subplots(rows=1,cols=2,shared_yaxes=True,subplot_titles=list(TARGETS.values()))
for j,t in enumerate(TARGETS,1):
    fig.add_trace(go.Heatmap(z=maps[t],x=times,y=frequency_labels,coloraxis='coloraxis',xgap=1,ygap=1,
                             hovertemplate='Bin center %{x} ms · %{y} Hz<br>Mean |weight| %{z:.4f}<extra></extra>'),row=1,col=j)
fig.update_xaxes(range=[0,500],title_text='Time after odor onset (ms)')
fig.update_yaxes(type='category',categoryorder='array',categoryarray=frequency_labels)
fig.update_yaxes(title_text='Wavelet frequency (Hz)',row=1,col=1)
fig.update_layout(coloraxis=dict(colorscale=WEIGHT_SCALE,cmin=0,cmax=limit,
                                colorbar=dict(title='Mean absolute<br>coefficient',len=.85,tickformat='.3f')))
publish(fig,'05-training-coefficients','Where did the amplitude model put weight?',height=430,
        margin=dict(l=87,r=135,t=84,b=62))

In [ ]:
WEIGHT_TARGET='position'  # 'odor' or 'position'
WEIGHT_CLASS=3            # 1–5; odor 1–5 corresponds to A–E
by_rat={r['rat']:r for r in weights if r['target']==WEIGHT_TARGET}
fig=make_subplots(rows=3,cols=5,column_titles=RATS,shared_yaxes='rows',
                  shared_xaxes='columns',horizontal_spacing=.025,vertical_spacing=.11)
for i,(field,label,cmap) in enumerate([('signed','Signed<br>weight',SIGN_SCALE),
                                      ('magnitude','Absolute<br>weight',WEIGHT_SCALE),
                                      ('nonzero','Fraction<br>zero',WEIGHT_SCALE)],1):
    vals=[np.asarray(by_rat[r][field][WEIGHT_CLASS-1]).T for r in RATS]
    if field=='nonzero': vals=[1-a/by_rat[r]['n_channels'] for r,a in zip(RATS,vals,strict=True)]
    hi=1 if field=='nonzero' else max(np.abs(vals).max(),1e-12)
    axis='coloraxis'+(str(i) if i>1 else '')
    for j,(rat,a) in enumerate(zip(RATS,vals,strict=True),1):
        fig.add_trace(go.Heatmap(z=a,x=times,y=frequency_labels,coloraxis=axis,xgap=1,ygap=1,
                                 hovertemplate=f'{rat} · %{{x}} ms · %{{y}} Hz<br>%{{z:.4f}}<extra></extra>'),row=i,col=j)
        fig.update_xaxes(range=[0,500],tickvals=[0,250,500],row=i,col=j)
        fig.update_yaxes(type='category',categoryorder='array',categoryarray=frequency_labels,
                         tickvals=[frequency_labels[0],frequency_labels[-1]],row=i,col=j)
    y0,y1=fig.get_subplot(i,1).yaxis.domain
    fig.update_layout({axis:dict(colorscale=cmap,cmin=-hi if field=='signed' else 0,cmax=hi,
                                colorbar=dict(title=label,len=y1-y0,y=(y0+y1)/2,tickformat='.2g',nticks=3))})
fig.update_yaxes(title_text='Frequency (Hz)',row=2,col=1)
fig.update_xaxes(title_text='Time after odor onset (ms)',row=3,col=3)
publish(fig,'supplement-coefficients-by-rat',f'{TARGETS[WEIGHT_TARGET]} · class {WEIGHT_CLASS} by rat',
        width=1220,height=705,margin=dict(l=90,r=140,t=100,b=68))

## 06 · What did CEBRA-Time learn? (supplement)

**Caption.** Each labeled dot is the average validation location for one class. The two displayed dimensions are fitted on training embeddings separately for each rat and target. Distances cannot be compared across rats, and centers do not reveal within-class spread or prove decoding or odor independence. No test projection was evaluated. Switch to out-of-sequence trials to see the position model’s behavior on that subset.

In [ ]:
LATENT_COHORT='all'  # 'all' or 'outseq'; odor encoder only has in-sequence trials
LATENT_LABEL='target'  # 'target', 'odor', or 'position'
latent_index={(r['target'],r['rat']):r for r in interpretation['latents']}
latent_targets=['position'] if LATENT_COHORT=='outseq' else list(TARGETS)
latent_titles=[rat for _ in latent_targets for rat in RATS]
fig=make_subplots(rows=len(latent_targets),cols=5,subplot_titles=latent_titles,
                  horizontal_spacing=.045,vertical_spacing=.25 if len(latent_targets)>1 else .10)
for i,target in enumerate(latent_targets,1):
    label=target if LATENT_LABEL=='target' else LATENT_LABEL
    for j,rat in enumerate(RATS,1):
        row=latent_index[target,rat]
        points=[p for p in row['groups'] if p['cohort']==LATENT_COHORT and p['label']==label]
        names=['ABCDE'[p['cls']-1] if label=='odor' else str(p['cls']) for p in points]
        x,y=np.asarray([p['xy'] for p in points],dtype=float).reshape(-1,2).T
        fig.add_trace(go.Scatter(x=x,y=y,mode='markers+text',text=names,textposition='top center',
            marker=dict(size=13,color=[CATS[p['cls']-1] for p in points],line=dict(color=WHITE,width=1)),
            customdata=[p['n'] for p in points],
            hovertemplate='Class %{text}<br>%{customdata} trials<br>First: %{x:.3f}<br>Second: %{y:.3f}<extra></extra>'),
            row=i,col=j)
        fig.add_hline(y=0,line_color=GRID,line_dash='dot',row=i,col=j)
        fig.add_vline(x=0,line_color=GRID,line_dash='dot',row=i,col=j)
        explained=100*sum(row['explained'])
        support=sum(p['n'] for p in points)
        fig.layout.annotations[(i-1)*len(RATS)+j-1].text=f'{rat} · {support} trials<br><sup>{explained:.0f}% variance</sup>'
        fig.update_xaxes(showgrid=False,nticks=4,
                        title_text='First component' if i==len(latent_targets) and j==3 else None,row=i,col=j)
        fig.update_yaxes(showgrid=False,nticks=4,
                        title_text=f'{TARGETS[target]}<br>Second component' if j==1 else None,row=i,col=j)
fig.update_layout(showlegend=False)
publish(fig,'06-cebra-validation-projections','CEBRA-Time · Validation class centers',
        width=1300,height=320*len(latent_targets),margin=dict(l=80,r=35,t=103,b=60))

## 07 · Optional: look at the uploaded recordings

These are illustrations, separate from the reviewed decoding evidence. The first figure shows what a 60 Hz notch does to one raw LFP channel from a contiguous recording segment. The second shows how each channel’s 10-second signal variability changes relative to its own typical value. The heatmap is not an across-channel strength comparison. Neither figure runs the decoder’s full artifact screening. The main poster does not require these figures.

If the cell reports missing files, check RAW_ROOT in Setup and unpack the archives into the stated directory structure. Expect roughly a few hundred megabytes of working memory for this optional part; the large recording is read one channel at a time.

In [ ]:
from contextlib import closing
from zipfile import ZipFile
from numpy.lib import format as npy
from scipy.signal import iirnotch, tf2sos, sosfiltfilt, welch

def rows_from_npz(path):
    # The uploaded provider format stores a C-order channel-by-sample array in data.npy.
    with ZipFile(path) as archive:
        with archive.open('keys.npy') as keys_file:
            keys=npy.read_array(keys_file,allow_pickle=False).tolist()
        with archive.open('data.npy') as stream:
            version=npy.read_magic(stream)
            header={(1,0):npy.read_array_header_1_0,(2,0):npy.read_array_header_2_0}.get(version)
            if header is None:
                raise ValueError(f'Unsupported array header {version} in {path.name}')
            shape,fortran,dtype=header(stream)
            if len(shape)!=2 or fortran or dtype.kind not in 'biuf' or len(keys)!=shape[0]:
                raise ValueError(f'Expected numeric channels by samples in {path.name}')
            size=shape[1]*dtype.itemsize
            for key in keys:
                data=stream.read(size)
                if len(data)!=size:
                    raise ValueError(f'Truncated {path.name}')
                yield key,np.frombuffer(data,dtype=dtype)

def optional_recording_figures(root, session):
    fs=1000; block=10*fs
    lfp=root/session/f'{session}_lfp.npz'; bvr=root/session/f'{session}_bvr.npz'
    if not lfp.is_file() or not bvr.is_file():
        print(f'Optional raw figures skipped: need {lfp} and {bvr}. The poster figures above are complete.')
        return
    with closing(rows_from_npz(bvr)) as source:
        clock=next(row for key,row in source if key=='TimeBin')
    dt=np.diff(clock)
    if not np.isclose(np.median(dt),1/fs,atol=1e-6):
        raise ValueError('Expected TimeBin in seconds at 1,000 samples per second')
    edges=np.r_[0,np.flatnonzero(~np.isclose(dt,1/fs,atol=1e-6))+1,len(clock)]
    complete=[(a,a+(b-a)//block*block) for a,b in zip(edges[:-1],edges[1:],strict=True) if b-a>=block]
    if not complete:
        raise ValueError('No contiguous 10-second recording blocks')
    lengths=np.array([(b-a)//block for a,b in complete])
    labels,maps=[],[]
    psd_pair=None
    for name,row in rows_from_npz(lfp):
        if len(row)!=len(clock):
            raise ValueError('LFP and behavior sample counts do not match')
        label=name.split('_',1)[0]
        sd=np.concatenate([np.std(row[a:b].reshape(-1,block),axis=1) for a,b in complete])
        median=np.median(sd)
        ratio=np.divide(sd,median,out=np.full_like(sd,np.nan),where=(median>0)&(sd>0))
        labels.append(label); maps.append(np.log2(ratio))
        if label=='T1':
            a,b=max(complete,key=lambda pair:pair[1]-pair[0])
            raw=row[a:b]
            if not np.isfinite(raw).all():
                print('Notch example skipped: the selected continuous T1 segment contains non-finite samples.')
                continue
            sos=tf2sos(*iirnotch(60,30,fs=fs)); pad=2199
            filtered=sosfiltfilt(sos,raw,padtype='odd',padlen=pad)
            n=(len(raw)-2*pad)//block*block
            if n<block:
                continue
            def median_psd(signal):
                hz,power=welch(signal[pad:pad+n].reshape(-1,block),fs=fs,nperseg=4000,noverlap=2000,axis=1)
                return hz,np.median(power,axis=0)
            hz,before=median_psd(raw); _,after=median_psd(filtered)
            psd_pair=hz,before,after
    rat=session.split('_',1)[-1].capitalize()
    if psd_pair is not None:
        hz,before,after=psd_pair; select=(hz>=50)&(hz<=70)
        fig=go.Figure()
        for y,label,color in [(before,'Before notch',MUTED),(after,'After notch',TEAL)]:
            fig.add_trace(go.Scatter(x=hz[select],y=10*np.log10(np.maximum(y[select],np.finfo(float).tiny)),
                                     name=label,line=dict(color=color,width=3),mode='lines',
                                     hovertemplate='%{x:.2f} Hz · %{y:.1f} dB'))
        fig.add_vline(x=60,line_color=MUTED,line_dash='dash',annotation_text='60 Hz')
        fig.update_layout(xaxis_title='Frequency (Hz)',yaxis_title='Power (dB re 1 unit²/Hz)',
                          legend=dict(orientation='h',x=0,y=1.08))
        publish(fig,'07-raw-60-hz-example',f'{rat} T1 · 60 Hz notch illustration',width=890,height=415,
                margin=dict(l=90,r=45,t=105,b=60))
    else:
        print('No valid T1 notch example for this session; the variability heatmap is still available.')
    fig=go.Figure(go.Heatmap(z=maps,x=np.arange(1,int(lengths.sum())+1),y=labels,
        colorscale=SIGN_SCALE,zmin=-2,zmax=2,colorbar=dict(title='Variation relative<br>to channel median<br>(log₂)',len=.85),
        hovertemplate='Channel %{y} · window %{x}<br>Relative variability %{z:+.2f}<extra></extra>'))
    for q in np.cumsum(lengths)[:-1]: fig.add_vline(x=q+.5,line_color=MUTED,line_dash='dot')
    fig.update_layout(xaxis_title='Recorded 10-second windows (clock gaps compressed)',yaxis_title='Recording channel')
    fig.update_yaxes(autorange='reversed',tickmode='array',tickvals=labels)
    publish(fig,'07-raw-amplitude-heatmap',f'{rat} · signal variability by channel',height=550,
            margin=dict(l=76,r=160,t=75,b=60))

optional_recording_figures(RAW_ROOT,RAW_SESSION)

## 08 · From notebook to poster

**Suggested layout:** methods and tasks on the left; the test comparison and its limitation in the middle; masking and training coefficients on the right. Keep the ten learned-feature plots, full masking table, and raw recording examples as supplementary material.

| What you need | Find it here | Downloadable asset |
| --- | --- | --- |
| Study design | 01 · Editable text and model diagram | `01-study-design` |
| Decoding scores | 02 · Four-rat test first, five-rat validation alongside | `02-decoding-performance`, `02-test-and-validation.csv` |
| Actual versus usual position | 03 · Example and count table | `03-position-diagnostic` |
| What the models used | 04–06 · Masking, weights, and learned features | `04-low-frequency-masking`, `05-training-coefficients`, `06-cebra-validation-projections` |
| Recording illustrations | 07 · Optional; not model evidence | `07-raw-60-hz-example`, `07-raw-amplitude-heatmap` |

**Limits and next step.** These are five within-rat comparisons, not evidence of generalization to unseen rats. The experiment compares complete pipelines, and odor and position coincide on in-sequence trials. The small mismatched subset needs independent confirmation. Grouped masking is validation-only and noncausal.

**References:** [Shahbaba et al. (2022), recordings and task](https://doi.org/10.1038/s41467-022-28057-6) · [Schneider, Lee, and Mathis (2023), CEBRA](https://doi.org/10.1038/s41586-023-06031-6).

### Optional evidence tables for questions

Class support explains why strict test balanced accuracy cannot include Buchanan. The separate signal audit shows the original recorded diagnostics and their review status; it is not the raw demonstration in Section 07.

In [ ]:
support = csv_frame('class_support.csv')
display(support.query("session == 'Buchanan' and split == 'test' and model == 'historical_anchor'")
        [['target','class','support']].pivot(index='class',columns='target',values='support').rename(columns=TARGETS))
signal_audit = csv_frame('signal_qa.csv')
display(signal_audit[['session','legacy_peak_hz','legacy_attenuation_db','evidence_status']]
        .rename(columns={'session':'Rat','legacy_peak_hz':'Peak frequency (Hz)',
                         'legacy_attenuation_db':'Attenuation (dB)','evidence_status':'Review status'}).round(1))

from zipfile import ZIP_DEFLATED, ZipFile

files = [OUT/'plotly.min.js', *(OUT/f'{name}.html' for name in FIGURES),
         *(OUT/name for name in ('02-test-and-validation.csv','03-position-diagnostic.csv','04-all-masking-groups.csv'))]
if EXPORT_PRINT:
    # Batch exports keep one Chrome session instead of launching it for each image.
    print_jobs = [(figure, width, OUT/f'{name}.{ext}', 3.125 if ext=='png' else 1)
                  for name,(figure,width) in FIGURES.items() for ext in ('svg','pdf','png')]
    figs,widths,paths,scales = zip(*print_jobs,strict=True)
    pio.write_images(list(figs),list(paths),width=list(widths),height=[f.layout.height for f in figs],scale=list(scales))
    files.extend(paths)
else:
    print('Print files skipped. Follow the setup note for Kaleido and Chrome, or use the interactive HTML.')

with ZipFile('poster_assets.zip','w',compression=ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path,arcname=path.as_posix())
print('Download poster_assets.zip. Open the HTML charts after extracting the whole folder.')